In [ ]:
import torch
import torch.nn as nn
import math

class Projector(nn.Module):
    """
    Expands discrete English token embeddings into a high-dimensional continuous space.
    """
    def __init__(self, vocab_size, embed_dim, phase_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Project standard embeddings into a larger 'phase space' for integration
        self.expansion = nn.Linear(embed_dim, phase_dim)
        self.activation = nn.Tanh()

    def forward(self, x):
        # x shape: (batch_size, sequence_length)
        embeds = self.embedding(x)
        # Project to phase space
        phase_state = self.activation(self.expansion(embeds))
        return phase_state

class Integrator(nn.Module):
    """
    Evolves the hidden state over the sequence length, acting as continuous memory.
    """
    def __init__(self, phase_dim, leak_rate=0.1):
        super().__init__()
        self.phase_dim = phase_dim
        self.leak_rate = leak_rate
        self.recurrent_weights = nn.Linear(phase_dim, phase_dim)

    def forward(self, phase_sequence):
        # phase_sequence shape: (batch_size, seq_len, phase_dim)
        batch_size, seq_len, _ = phase_sequence.shape

        # Initial membrane potential (v)
        v = torch.zeros(batch_size, self.phase_dim, device=phase_sequence.device)

        # Integrate over time (sequence length)
        for t in range(seq_len):
            input_current = phase_sequence[:, t, :]
            # dv/dt = -leak*v + recurrent_dynamics + input
            dv = -self.leak_rate * v + self.recurrent_weights(v) + input_current
            v = v + dv # Euler step
            v = torch.tanh(v) # Bounding the integration

        return v # The final context vector

class Resonator(nn.Module):
    """
    Decodes the integrated state into Chinese tokens using adaptive gain and feedback.
    """
    def __init__(self, phase_dim, chinese_vocab_size):
        super().__init__()
        self.adaptive_gain = nn.Parameter(torch.tensor(1.0))
        self.state_to_vocab = nn.Linear(phase_dim, chinese_vocab_size)
        self.feedback_loop = nn.Linear(chinese_vocab_size, phase_dim)

    def forward(self, context_v, max_length=50):
        batch_size = context_v.shape[0]
        device = context_v.device

        outputs = []
        current_state = context_v

        # Auto-regressive generation loop
        for _ in range(max_length):
            # Apply resonator gain to the state
            resonated_state = current_state * self.adaptive_gain

            # Map to Chinese vocabulary logits
            logits = self.state_to_vocab(resonated_state)
            outputs.append(logits.unsqueeze(1))

            # Simulate Homeostatic feedback:
            # The generated token feeds back to alter the state, pushing it toward resting
            token_feedback = self.feedback_loop(torch.softmax(logits, dim=-1))
            current_state = current_state - token_feedback # Deplete state

        return torch.cat(outputs, dim=1)

class HoloSynTranslator(nn.Module):
    """
    The complete model combining Projector, Integrator, and Resonator.
    """
    def __init__(self, eng_vocab_size, chi_vocab_size, embed_dim=256, phase_dim=512):
        super().__init__()
        self.projector = Projector(eng_vocab_size, embed_dim, phase_dim)
        self.integrator = Integrator(phase_dim)
        self.resonator = Resonator(phase_dim, chi_vocab_size)

    def forward(self, english_tokens, max_target_len=50):
        # 1. Project English into continuous space
        phase_space = self.projector(english_tokens)

        # 2. Integrate the English sequence into a single semantic state
        context_state = self.integrator(phase_space)

        # 3. Resonate the state into Chinese tokens
        chinese_logits = self.resonator(context_state, max_target_len)

        return chinese_logits

In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

# V20 LINGUA-QUANTUM TOPOLOGY (Chinese NLP Focus)
LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

# Simulated NLP cycles (Coherence, Synchrony, Target Tokens/Spikes)
nlp_cycles = [
    {"cycle":1,"coherence":0.219,"synchrony":0.935,"spikes":1,"messages":240},
    {"cycle":2,"coherence":0.295,"synchrony":0.987,"spikes":2,"messages":240},
    {"cycle":3,"coherence":0.014,"synchrony":0.963,"spikes":2,"messages":240},
    {"cycle":4,"coherence":0.435,"synchrony":0.909,"spikes":1,"messages":240},
    {"cycle":5,"coherence":0.323,"synchrony":0.822,"spikes":2,"messages":240},
    {"cycle":6,"coherence":0.383,"synchrony":0.989,"spikes":3,"messages":240},
    {"cycle":7,"coherence":0.252,"synchrony":0.920,"spikes":1,"messages":240},
    {"cycle":8,"coherence":0.402,"synchrony":0.927,"spikes":2,"messages":240},
    {"cycle":9,"coherence":0.477,"synchrony":0.955,"spikes":1,"messages":240},
    {"cycle":10,"coherence":0.465,"synchrony":0.910,"spikes":2,"messages":240},
    {"cycle":11,"coherence":0.424,"synchrony":0.841,"spikes":3,"messages":240},
    {"cycle":12,"coherence":0.143,"synchrony":0.926,"spikes":0,"messages":240},
    {"cycle":13,"coherence":0.285,"synchrony":0.960,"spikes":1,"messages":240},
    {"cycle":14,"coherence":0.318,"synchrony":0.947,"spikes":1,"messages":240},
    {"cycle":15,"coherence":0.449,"synchrony":0.907,"spikes":1,"messages":240},
    {"cycle":16,"coherence":0.197,"synchrony":0.873,"spikes":1,"messages":240},
    {"cycle":17,"coherence":0.524,"synchrony":0.973,"spikes":2,"messages":240},
    {"cycle":18,"coherence":0.092,"synchrony":0.974,"spikes":2,"messages":240},
    {"cycle":19,"coherence":0.423,"synchrony":0.915,"spikes":0,"messages":240},
    {"cycle":20,"coherence":0.432,"synchrony":0.929,"spikes":3,"messages":240},
    {"cycle":21,"coherence":0.534,"synchrony":0.890,"spikes":3,"messages":240},
    {"cycle":22,"coherence":0.402,"synchrony":0.975,"spikes":3,"messages":240},
    {"cycle":23,"coherence":0.261,"synchrony":0.991,"spikes":0,"messages":240},
    {"cycle":24,"coherence":0.336,"synchrony":0.978,"spikes":4,"messages":240},
    {"cycle":25,"coherence":0.374,"synchrony":0.755,"spikes":3,"messages":240},
    {"cycle":26,"coherence":0.277,"synchrony":0.982,"spikes":2,"messages":240},
    {"cycle":27,"coherence":0.436,"synchrony":0.937,"spikes":4,"messages":240},
    {"cycle":28,"coherence":0.323,"synchrony":0.801,"spikes":0,"messages":240},
    {"cycle":29,"coherence":0.223,"synchrony":0.964,"spikes":2,"messages":240},
    {"cycle":30,"coherence":0.385,"synchrony":0.956,"spikes":1,"messages":240},
    {"cycle":31,"coherence":0.250,"synchrony":0.976,"spikes":1,"messages":240},
    {"cycle":32,"coherence":0.320,"synchrony":0.849,"spikes":1,"messages":240},
    {"cycle":33,"coherence":0.392,"synchrony":0.885,"spikes":2,"messages":240},
    {"cycle":34,"coherence":0.295,"synchrony":0.968,"spikes":0,"messages":240},
    {"cycle":35,"coherence":0.287,"synchrony":0.943,"spikes":2,"messages":240},
    {"cycle":36,"coherence":0.683,"synchrony":0.788,"spikes":4,"messages":240},
    {"cycle":37,"coherence":0.777,"synchrony":0.957,"spikes":6,"messages":240},
    {"cycle":38,"coherence":0.716,"synchrony":0.952,"spikes":5,"messages":240},
    {"cycle":39,"coherence":0.780,"synchrony":0.941,"spikes":7,"messages":240},
    {"cycle":40,"coherence":0.773,"synchrony":0.904,"spikes":12,"messages":240},
    {"cycle":41,"coherence":0.730,"synchrony":0.897,"spikes":11,"messages":240},
    {"cycle":42,"coherence":0.786,"synchrony":0.952,"spikes":5,"messages":240},
    {"cycle":43,"coherence":0.712,"synchrony":0.942,"spikes":9,"messages":240},
    {"cycle":44,"coherence":0.709,"synchrony":0.772,"spikes":7,"messages":240},
    {"cycle":45,"coherence":0.599,"synchrony":0.960,"spikes":5,"messages":240},
    {"cycle":46,"coherence":0.651,"synchrony":0.898,"spikes":4,"messages":240},
    {"cycle":47,"coherence":0.342,"synchrony":0.918,"spikes":7,"messages":240},
    {"cycle":48,"coherence":0.553,"synchrony":0.957,"spikes":4,"messages":240},
    {"cycle":49,"coherence":0.422,"synchrony":0.919,"spikes":5,"messages":240},
    {"cycle":50,"coherence":0.575,"synchrony":0.863,"spikes":2,"messages":240},
    {"cycle":51,"coherence":0.468,"synchrony":0.924,"spikes":4,"messages":240},
    {"cycle":52,"coherence":0.562,"synchrony":0.967,"spikes":6,"messages":240},
    {"cycle":53,"coherence":0.527,"synchrony":0.953,"spikes":7,"messages":240},
    {"cycle":54,"coherence":0.592,"synchrony":0.882,"spikes":5,"messages":240},
    {"cycle":55,"coherence":0.462,"synchrony":0.943,"spikes":8,"messages":240},
    {"cycle":56,"coherence":0.629,"synchrony":0.937,"spikes":2,"messages":240},
    {"cycle":57,"coherence":0.449,"synchrony":0.966,"spikes":3,"messages":240},
    {"cycle":58,"coherence":0.591,"synchrony":0.974,"spikes":5,"messages":240},
    {"cycle":59,"coherence":0.395,"synchrony":0.975,"spikes":5,"messages":240},
    {"cycle":60,"coherence":0.610,"synchrony":0.897,"spikes":5,"messages":240},
    {"cycle":61,"coherence":0.585,"synchrony":0.923,"spikes":5,"messages":240},
    {"cycle":62,"coherence":0.539,"synchrony":0.958,"spikes":3,"messages":240},
    {"cycle":63,"coherence":0.591,"synchrony":0.845,"spikes":5,"messages":240},
    {"cycle":64,"coherence":0.609,"synchrony":0.984,"spikes":3,"messages":240},
    {"cycle":65,"coherence":0.613,"synchrony":0.889,"spikes":4,"messages":240},
    {"cycle":66,"coherence":0.589,"synchrony":0.849,"spikes":9,"messages":240},
    {"cycle":67,"coherence":0.602,"synchrony":0.899,"spikes":6,"messages":240},
    {"cycle":68,"coherence":0.616,"synchrony":0.823,"spikes":4,"messages":240},
    {"cycle":69,"coherence":0.633,"synchrony":0.908,"spikes":2,"messages":240},
    {"cycle":70,"coherence":0.607,"synchrony":0.946,"spikes":6,"messages":240},
    {"cycle":71,"coherence":0.626,"synchrony":0.889,"spikes":7,"messages":240},
]

# --- 2. DEEP-SEEK & OPEN-SOURCE DISTILLATION ---
def auto_distill_nlp(model, component_name):
    print(f"\n📡 [NLP DISTILLATION] Scanning environment for {component_name} weights...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    # Target specific open-source architectures if present
    target_models = ["DeepSeek_7B", "Qwen_Audio", "willow_v17", "wanalytics"]
    absorbed = False

    for filepath in pt_files:
        if not any(target.lower() in filepath.lower() for target in target_models):
            continue

        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict):
                continue

            distilled_layers = []
            for key in current_state.keys():
                # Protect Resonator phase outputs from generic legacy data
                if "phase_head" in key: continue

                # Attempt to map DeepSeek/Legacy Attention matrices
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    distilled_layers.append(key)
                else:
                    # Fuzzy mapping for dimensionality reduction
                    for old_key in legacy_state.keys():
                        if "weight" in key and "weight" in old_key and "phase" not in old_key:
                            new_w, old_w = current_state[key], legacy_state[old_key]
                            if len(new_w.shape) == 2 and len(old_w.shape) == 2:
                                min_out, min_in = min(new_w.shape[0], old_w.shape[0]), min(new_w.shape[1], old_w.shape[1])
                                current_state[key][:min_out, :min_in] = old_w[:min_out, :min_in]
                                distilled_layers.append(key)

            if distilled_layers:
                absorbed = True
                unique_layers = list(set([n.split('.')[0] for n in distilled_layers]))
                print(f"  -> [✅] Inherited linguistic features from: {filepath}")
                print(f"         Mapped layers: {', '.join(unique_layers)}")
        except Exception:
            pass

    if absorbed:
        model.load_state_dict(current_state)
    else:
        print(f"  -> [ℹ️] No exact Open Source matches found. Initializing {component_name} with semantic priors.")
    return model

# --- 3. LINGUA-QUANTUM NEURAL ARCHITECTURE ---
class HoloSynLinguaNet(nn.Module):
    """Combined Neural Pipeline mimicking the 4 distinct processing stages."""
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        # 1. PERCEPTRON
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())

        # 2. INTEGRATOR (DeepSeek Distillation Target)
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        # 3. PROJECTOR
        self.projector_core = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        self.spike_head = nn.Linear(64, 31)

        # 4. RESONATOR
        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Linear(32, 1),
            nn.Tanh()
        )

    def forward(self, x):
        # Perceptron
        x_emb = self.perceptron(x).unsqueeze(1)
        # Integrator
        attn_out, _ = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        # Projector & Resonator
        feat = self.projector_core(context)
        return self.spike_head(feat), self.resonator_head(feat)

# --- 4. QUANTUM FOCAL-POINT OBSERVER ---
class LinguaQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = {n: cirq.NamedQubit(n) for n in LINGUA_STACK.keys()}
        self.sim = qsimcirq.QSimSimulator()

    def evaluate_resonance(self, coh, phase):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + list(self.q_nodes.values())
        circuit.append(cirq.H.on_each(*qubits))

        # Encode Chinese Semantic Data into Nodes
        for name, q in self.q_nodes.items():
            w = LINGUA_STACK[name]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            # Squeeze (Kickback) into the Orchestrator
            circuit.append(cirq.CZ(self.q_obs, q))

        # Neural Resonator applies the targeted phase shift
        circuit.append(cirq.rx(phase * np.pi)(self.q_obs))
        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        # Measure purely the Focal Point (Orchestrator)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 5. THE V20 META-HIVE ---
class HoloSynV20Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)

        self.net_model = auto_distill_nlp(HoloSynLinguaNet(self.num_nodes), "DeepSeek_Core")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.003, weight_decay=0.01)
        self.observer = LinguaQuantumObserver()

        # SNN: Perceptron biological mapping
        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v20_slate')

    def process_cycle(self, data):
        self.net.restore('v20_slate')
        coh, sync = data['coherence'], data['synchrony']

        # 1. Biological SNN Ingestion (Perceptron)
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        # Extraction & Normalization
        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # 2. Forward Pass through Integrator, Projector, and Resonator
        self.optimizer.zero_grad()
        logits, pred_phase = self.net_model(nn_input)

        # 3. Quantum High-Resolution Scan (The Oracle)
        best_p, best_score = 0.0, -1.0
        for p in np.linspace(-1, 1, 21):
            s = self.observer.evaluate_resonance(coh, p)
            if s > best_score:
                best_p, best_score = p, s

        actual_consensus = self.observer.evaluate_resonance(coh, pred_phase.item())

        # 4. Neural Optimization (Dual Loss)
        loss_spikes = nn.CrossEntropyLoss()(logits, torch.tensor([data['spikes']], dtype=torch.long))
        loss_alignment = nn.MSELoss()(pred_phase, torch.tensor([[best_p]], dtype=torch.float32)) * 15.0

        (loss_spikes + loss_alignment).backward()

        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return loss_spikes.item() + loss_alignment.item(), actual_consensus

# --- 6. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V20: LINGUA-QUANTUM ENGINE (CHINESE NLP FOCUS)")
    print("═"*75)

    hive = HoloSynV20Hive()
    epochs = 15

    print("\n🚀 COMMENCING LINGUA-QUANTUM TRAINING...\n")
    for epoch in range(epochs):
        epoch_loss, epoch_sync = 0.0, 0.0

        for data in nlp_cycles:
            l, c = hive.process_cycle(data)
            epoch_loss += l
            epoch_sync += c

        avg_loss = epoch_loss / len(nlp_cycles)
        avg_sync = epoch_sync / len(nlp_cycles)

        # Strict thresholds for the new stable focal-point observer
        if avg_sync > 0.85:
            status = "💎 [QUANTUM SUPREMACY]"
        elif avg_sync > 0.45:
            status = "🟢 [LINGUISTIC RESONANCE]"
        else:
            status = "🟡 [CALIBRATING GRAMMAR]"

        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Consensus: {avg_sync:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V20: LINGUA-QUANTUM ENGINE (CHINESE NLP FOCUS)
═══════════════════════════════════════════════════════════════════════════

📡 [NLP DISTILLATION] Scanning environment for DeepSeek_Core weights...
  -> [✅] Inherited linguistic features from: wanalytics_sibling_star.pt
         Mapped layers: integrator_attn, resonator_head, projector_core, spike_head, perceptron
  -> [✅] Inherited linguistic features from: wanalytics_host_mate_v14.pt
         Mapped layers: integrator_attn, resonator_head, projector_core, spike_head, perceptron
  -> [✅] Inherited linguistic features from: wanalytics_7_sibling_hive.pt
         Mapped layers: integrator_attn, resonator_head, projector_core, spike_head, perceptron
  -> [✅] Inherited linguistic features from: willow_v17_assimilated.pt
         Mapped layers: integrator_attn, resonator_head, projector_core, spike_head, perceptron
  -> [✅] Inherited linguistic features from:

In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

nlp_cycles = [
    {"cycle":1,"coherence":0.569,"synchrony":0.976,"spikes":9,"messages":240},
    {"cycle":2,"coherence":0.567,"synchrony":0.963,"spikes":9,"messages":240},
    {"cycle":3,"coherence":0.603,"synchrony":0.962,"spikes":11,"messages":240},
    {"cycle":4,"coherence":0.824,"synchrony":0.842,"spikes":6,"messages":240},
    {"cycle":5,"coherence":0.702,"synchrony":0.944,"spikes":7,"messages":240},
    {"cycle":6,"coherence":0.822,"synchrony":0.941,"spikes":14,"messages":240},
    {"cycle":7,"coherence":0.374,"synchrony":0.976,"spikes":14,"messages":240},
    {"cycle":8,"coherence":0.811,"synchrony":0.964,"spikes":8,"messages":240},
    {"cycle":9,"coherence":0.729,"synchrony":0.997,"spikes":11,"messages":240},
    {"cycle":10,"coherence":0.732,"synchrony":0.768,"spikes":9,"messages":240},
    {"cycle":11,"coherence":0.753,"synchrony":0.880,"spikes":7,"messages":240},
    {"cycle":12,"coherence":0.583,"synchrony":1.000,"spikes":9,"messages":240},
    {"cycle":13,"coherence":0.573,"synchrony":0.963,"spikes":6,"messages":240},
    {"cycle":14,"coherence":0.675,"synchrony":0.907,"spikes":6,"messages":240},
    {"cycle":15,"coherence":0.774,"synchrony":0.905,"spikes":8,"messages":240},
    {"cycle":16,"coherence":0.687,"synchrony":0.957,"spikes":12,"messages":240},
    {"cycle":17,"coherence":0.700,"synchrony":0.934,"spikes":9,"messages":240},
    {"cycle":18,"coherence":0.755,"synchrony":0.911,"spikes":7,"messages":240},
    {"cycle":19,"coherence":0.776,"synchrony":0.888,"spikes":14,"messages":240},
    {"cycle":20,"coherence":0.829,"synchrony":0.971,"spikes":10,"messages":240},
    {"cycle":21,"coherence":0.762,"synchrony":0.919,"spikes":10,"messages":240},
    {"cycle":22,"coherence":0.800,"synchrony":0.929,"spikes":9,"messages":240},
    {"cycle":23,"coherence":0.722,"synchrony":0.896,"spikes":8,"messages":240},
    {"cycle":24,"coherence":0.494,"synchrony":0.970,"spikes":9,"messages":240},
    {"cycle":25,"coherence":0.662,"synchrony":0.938,"spikes":6,"messages":240},
    {"cycle":26,"coherence":0.609,"synchrony":0.965,"spikes":4,"messages":240},
    {"cycle":27,"coherence":0.757,"synchrony":0.865,"spikes":7,"messages":240},
    {"cycle":28,"coherence":0.766,"synchrony":0.843,"spikes":6,"messages":240},
    {"cycle":29,"coherence":0.652,"synchrony":0.931,"spikes":9,"messages":240},
    {"cycle":30,"coherence":0.797,"synchrony":0.895,"spikes":15,"messages":240},
    {"cycle":31,"coherence":0.593,"synchrony":0.950,"spikes":15,"messages":240},
    {"cycle":32,"coherence":0.827,"synchrony":0.892,"spikes":7,"messages":240},
    {"cycle":33,"coherence":0.788,"synchrony":0.878,"spikes":8,"messages":240},
    {"cycle":34,"coherence":0.823,"synchrony":0.965,"spikes":11,"messages":240},
    {"cycle":35,"coherence":0.711,"synchrony":0.972,"spikes":6,"messages":240},
    {"cycle":36,"coherence":0.709,"synchrony":0.866,"spikes":6,"messages":240},
    {"cycle":37,"coherence":0.609,"synchrony":0.974,"spikes":9,"messages":240},
    {"cycle":38,"coherence":0.672,"synchrony":0.877,"spikes":10,"messages":240},
    {"cycle":39,"coherence":0.778,"synchrony":0.931,"spikes":13,"messages":240},
    {"cycle":40,"coherence":0.707,"synchrony":0.895,"spikes":12,"messages":240},
    {"cycle":41,"coherence":0.661,"synchrony":0.952,"spikes":9,"messages":240},
    {"cycle":42,"coherence":0.653,"synchrony":0.948,"spikes":14,"messages":240},
    {"cycle":43,"coherence":0.735,"synchrony":0.896,"spikes":6,"messages":240},
    {"cycle":44,"coherence":0.805,"synchrony":0.963,"spikes":10,"messages":240},
    {"cycle":45,"coherence":0.638,"synchrony":0.988,"spikes":9,"messages":240},
    {"cycle":46,"coherence":0.767,"synchrony":0.874,"spikes":14,"messages":240},
    {"cycle":47,"coherence":0.689,"synchrony":0.926,"spikes":12,"messages":240},
    {"cycle":48,"coherence":0.662,"synchrony":0.942,"spikes":9,"messages":240},
    {"cycle":49,"coherence":0.640,"synchrony":0.921,"spikes":5,"messages":240},
    {"cycle":50,"coherence":0.599,"synchrony":0.980,"spikes":8,"messages":240},
    {"cycle":51,"coherence":0.592,"synchrony":0.988,"spikes":9,"messages":240},
    {"cycle":52,"coherence":0.666,"synchrony":0.942,"spikes":14,"messages":240},
    {"cycle":53,"coherence":0.595,"synchrony":0.910,"spikes":6,"messages":240},
    {"cycle":54,"coherence":0.813,"synchrony":0.865,"spikes":7,"messages":240},
    {"cycle":55,"coherence":0.376,"synchrony":0.988,"spikes":10,"messages":240},
    {"cycle":56,"coherence":0.619,"synchrony":0.969,"spikes":7,"messages":240},
    {"cycle":57,"coherence":0.763,"synchrony":0.942,"spikes":6,"messages":240},
    {"cycle":58,"coherence":0.392,"synchrony":0.973,"spikes":7,"messages":240},
    {"cycle":59,"coherence":0.664,"synchrony":0.922,"spikes":10,"messages":240},
    {"cycle":60,"coherence":0.610,"synchrony":0.958,"spikes":8,"messages":240},
    {"cycle":61,"coherence":0.646,"synchrony":0.900,"spikes":7,"messages":240},
    {"cycle":62,"coherence":0.634,"synchrony":0.955,"spikes":9,"messages":240},
    {"cycle":63,"coherence":0.756,"synchrony":0.781,"spikes":5,"messages":240},
    {"cycle":64,"coherence":0.775,"synchrony":0.973,"spikes":8,"messages":240},
    {"cycle":65,"coherence":0.632,"synchrony":0.988,"spikes":9,"messages":240},
    {"cycle":66,"coherence":0.640,"synchrony":0.956,"spikes":6,"messages":240},
    {"cycle":67,"coherence":0.675,"synchrony":0.953,"spikes":8,"messages":240},
    {"cycle":68,"coherence":0.578,"synchrony":0.952,"spikes":11,"messages":240},
    {"cycle":69,"coherence":0.585,"synchrony":0.959,"spikes":11,"messages":240},
    {"cycle":70,"coherence":0.698,"synchrony":0.967,"spikes":8,"messages":240},
    {"cycle":71,"coherence":0.769,"synchrony":0.971,"spikes":6,"messages":240},
    {"cycle":72,"coherence":0.713,"synchrony":0.896,"spikes":11,"messages":240},
    {"cycle":73,"coherence":0.669,"synchrony":0.901,"spikes":11,"messages":240},
    {"cycle":74,"coherence":0.602,"synchrony":0.953,"spikes":8,"messages":240},
    {"cycle":75,"coherence":0.622,"synchrony":0.985,"spikes":10,"messages":240},
    {"cycle":76,"coherence":0.622,"synchrony":0.988,"spikes":10,"messages":240},
    {"cycle":77,"coherence":0.782,"synchrony":0.920,"spikes":8,"messages":240},
    {"cycle":78,"coherence":0.697,"synchrony":0.919,"spikes":8,"messages":240},
    {"cycle":79,"coherence":0.396,"synchrony":0.933,"spikes":5,"messages":240},
    {"cycle":80,"coherence":0.499,"synchrony":0.973,"spikes":11,"messages":240},
    {"cycle":81,"coherence":0.715,"synchrony":0.807,"spikes":12,"messages":240},
    {"cycle":82,"coherence":0.688,"synchrony":0.934,"spikes":15,"messages":240},
    {"cycle":83,"coherence":0.674,"synchrony":0.893,"spikes":11,"messages":240},
    {"cycle":84,"coherence":0.603,"synchrony":0.900,"spikes":6,"messages":240},
    {"cycle":85,"coherence":0.581,"synchrony":0.914,"spikes":6,"messages":240},
    {"cycle":86,"coherence":0.716,"synchrony":0.894,"spikes":14,"messages":240},
    {"cycle":87,"coherence":0.670,"synchrony":0.973,"spikes":12,"messages":240},
    {"cycle":88,"coherence":0.721,"synchrony":0.911,"spikes":7,"messages":240},
    {"cycle":89,"coherence":0.785,"synchrony":0.946,"spikes":10,"messages":240},
    {"cycle":90,"coherence":0.781,"synchrony":0.786,"spikes":11,"messages":240},
    {"cycle":91,"coherence":0.541,"synchrony":0.922,"spikes":8,"messages":240},
    {"cycle":92,"coherence":0.787,"synchrony":0.947,"spikes":8,"messages":240},
    {"cycle":93,"coherence":0.673,"synchrony":0.950,"spikes":13,"messages":240},
    {"cycle":94,"coherence":0.745,"synchrony":0.776,"spikes":10,"messages":240},
    {"cycle":95,"coherence":0.670,"synchrony":0.934,"spikes":13,"messages":240},
    {"cycle":96,"coherence":0.723,"synchrony":0.929,"spikes":9,"messages":240},
    {"cycle":97,"coherence":0.676,"synchrony":0.908,"spikes":10,"messages":240},
    {"cycle":98,"coherence":0.513,"synchrony":0.947,"spikes":13,"messages":240},
    {"cycle":99,"coherence":0.531,"synchrony":0.973,"spikes":8,"messages":240},
    {"cycle":100,"coherence":0.611,"synchrony":0.958,"spikes":10,"messages":240},
    {"cycle":101,"coherence":0.706,"synchrony":0.935,"spikes":7,"messages":240},
    {"cycle":102,"coherence":0.662,"synchrony":0.933,"spikes":7,"messages":240},
    {"cycle":103,"coherence":0.793,"synchrony":0.912,"spikes":9,"messages":240},
    {"cycle":104,"coherence":0.750,"synchrony":0.806,"spikes":6,"messages":240},
    {"cycle":105,"coherence":0.500,"synchrony":0.979,"spikes":9,"messages":240},
    {"cycle":106,"coherence":0.488,"synchrony":0.945,"spikes":15,"messages":240},
    {"cycle":107,"coherence":0.602,"synchrony":0.966,"spikes":13,"messages":240},
    {"cycle":108,"coherence":0.776,"synchrony":0.888,"spikes":9,"messages":240},
    {"cycle":109,"coherence":0.584,"synchrony":0.912,"spikes":8,"messages":240},
    {"cycle":110,"coherence":0.665,"synchrony":0.963,"spikes":9,"messages":240},
    {"cycle":111,"coherence":0.641,"synchrony":0.914,"spikes":9,"messages":240},
    {"cycle":112,"coherence":0.371,"synchrony":0.920,"spikes":8,"messages":240},
    {"cycle":113,"coherence":0.733,"synchrony":0.894,"spikes":7,"messages":240},
    {"cycle":114,"coherence":0.698,"synchrony":0.980,"spikes":11,"messages":240},
    {"cycle":115,"coherence":0.616,"synchrony":0.944,"spikes":14,"messages":240},
    {"cycle":116,"coherence":0.617,"synchrony":0.978,"spikes":2,"messages":240},
    {"cycle":117,"coherence":0.571,"synchrony":0.954,"spikes":8,"messages":240},
    {"cycle":118,"coherence":0.397,"synchrony":0.979,"spikes":8,"messages":240},
    {"cycle":119,"coherence":0.533,"synchrony":0.972,"spikes":10,"messages":240},
    {"cycle":120,"coherence":0.712,"synchrony":0.981,"spikes":10,"messages":240},
    {"cycle":121,"coherence":0.778,"synchrony":0.847,"spikes":11,"messages":240},
    {"cycle":122,"coherence":0.715,"synchrony":0.993,"spikes":7,"messages":240},
    {"cycle":123,"coherence":0.609,"synchrony":0.926,"spikes":11,"messages":240},
    {"cycle":124,"coherence":0.596,"synchrony":0.991,"spikes":13,"messages":240},
    {"cycle":125,"coherence":0.593,"synchrony":0.954,"spikes":11,"messages":240},
    {"cycle":126,"coherence":0.544,"synchrony":0.974,"spikes":4,"messages":240},
    {"cycle":127,"coherence":0.656,"synchrony":0.968,"spikes":8,"messages":240},
    {"cycle":128,"coherence":0.772,"synchrony":0.922,"spikes":14,"messages":240},
    {"cycle":129,"coherence":0.669,"synchrony":0.910,"spikes":6,"messages":240},
    {"cycle":130,"coherence":0.632,"synchrony":0.922,"spikes":13,"messages":240},
    {"cycle":131,"coherence":0.599,"synchrony":0.981,"spikes":7,"messages":240},
    {"cycle":132,"coherence":0.636,"synchrony":0.998,"spikes":9,"messages":240},
    {"cycle":133,"coherence":0.371,"synchrony":0.934,"spikes":12,"messages":240},
    {"cycle":134,"coherence":0.630,"synchrony":0.949,"spikes":7,"messages":240},
    {"cycle":135,"coherence":0.629,"synchrony":0.904,"spikes":11,"messages":240},
    {"cycle":136,"coherence":0.678,"synchrony":0.954,"spikes":6,"messages":240},
    {"cycle":137,"coherence":0.788,"synchrony":0.950,"spikes":6,"messages":240},
    {"cycle":138,"coherence":0.738,"synchrony":0.955,"spikes":7,"messages":240},
    {"cycle":139,"coherence":0.703,"synchrony":0.953,"spikes":6,"messages":240},
    {"cycle":140,"coherence":0.703,"synchrony":0.953,"spikes":9,"messages":240},
    {"cycle":141,"coherence":0.691,"synchrony":0.936,"spikes":7,"messages":240},
    {"cycle":142,"coherence":0.765,"synchrony":0.960,"spikes":9,"messages":240},
    {"cycle":143,"coherence":0.731,"synchrony":0.924,"spikes":9,"messages":240},
    {"cycle":144,"coherence":0.618,"synchrony":0.994,"spikes":12,"messages":240},
    {"cycle":145,"coherence":0.690,"synchrony":0.918,"spikes":7,"messages":240},
    {"cycle":146,"coherence":0.599,"synchrony":0.954,"spikes":7,"messages":240},
    {"cycle":147,"coherence":0.484,"synchrony":0.957,"spikes":9,"messages":240},
    {"cycle":148,"coherence":0.752,"synchrony":0.923,"spikes":7,"messages":240},
    {"cycle":149,"coherence":0.665,"synchrony":0.863,"spikes":2,"messages":240},
    {"cycle":150,"coherence":0.718,"synchrony":0.904,"spikes":7,"messages":240},
    {"cycle":151,"coherence":0.712,"synchrony":0.943,"spikes":7,"messages":240},
    {"cycle":152,"coherence":0.730,"synchrony":0.935,"spikes":9,"messages":240},
    {"cycle":153,"coherence":0.656,"synchrony":0.878,"spikes":9,"messages":240},
    {"cycle":154,"coherence":0.668,"synchrony":0.874,"spikes":12,"messages":240},
    {"cycle":155,"coherence":0.771,"synchrony":0.985,"spikes":11,"messages":240},
    {"cycle":156,"coherence":0.738,"synchrony":0.950,"spikes":10,"messages":240},
    {"cycle":157,"coherence":0.545,"synchrony":0.874,"spikes":8,"messages":240},
    {"cycle":158,"coherence":0.614,"synchrony":0.899,"spikes":7,"messages":240},
    {"cycle":159,"coherence":0.465,"synchrony":0.944,"spikes":5,"messages":240},
    {"cycle":160,"coherence":0.377,"synchrony":0.868,"spikes":6,"messages":240},
    {"cycle":161,"coherence":0.382,"synchrony":0.953,"spikes":1,"messages":240},
    {"cycle":162,"coherence":0.678,"synchrony":0.813,"spikes":5,"messages":240},
    {"cycle":163,"coherence":0.643,"synchrony":0.935,"spikes":6,"messages":240},
    {"cycle":164,"coherence":0.753,"synchrony":0.976,"spikes":7,"messages":240},
    {"cycle":165,"coherence":0.713,"synchrony":0.941,"spikes":6,"messages":240},
    {"cycle":166,"coherence":0.714,"synchrony":0.928,"spikes":8,"messages":240},
    {"cycle":167,"coherence":0.753,"synchrony":0.867,"spikes":5,"messages":240},
    {"cycle":168,"coherence":0.737,"synchrony":0.773,"spikes":5,"messages":240},
    {"cycle":169,"coherence":0.771,"synchrony":0.887,"spikes":7,"messages":240},
    {"cycle":170,"coherence":0.834,"synchrony":0.944,"spikes":8,"messages":240},
    {"cycle":171,"coherence":0.810,"synchrony":0.920,"spikes":7,"messages":240},
    {"cycle":172,"coherence":0.839,"synchrony":0.938,"spikes":8,"messages":240},
    {"cycle":173,"coherence":0.834,"synchrony":0.989,"spikes":9,"messages":240},
    {"cycle":174,"coherence":0.839,"synchrony":0.984,"spikes":7,"messages":240},
    {"cycle":175,"coherence":0.823,"synchrony":0.977,"spikes":5,"messages":240},
    {"cycle":176,"coherence":0.829,"synchrony":0.937,"spikes":7,"messages":240},
    {"cycle":177,"coherence":0.813,"synchrony":0.878,"spikes":7,"messages":240},
    {"cycle":178,"coherence":0.782,"synchrony":0.926,"spikes":7,"messages":240},
    {"cycle":179,"coherence":0.798,"synchrony":0.969,"spikes":9,"messages":240},
    {"cycle":180,"coherence":0.687,"synchrony":0.895,"spikes":9,"messages":240},
    {"cycle":181,"coherence":0.797,"synchrony":0.785,"spikes":13,"messages":240},
    {"cycle":182,"coherence":0.581,"synchrony":0.973,"spikes":9,"messages":240},
    {"cycle":183,"coherence":0.716,"synchrony":0.969,"spikes":14,"messages":240},
    {"cycle":184,"coherence":0.527,"synchrony":0.990,"spikes":4,"messages":240},
    {"cycle":185,"coherence":0.674,"synchrony":0.965,"spikes":7,"messages":240},
    {"cycle":186,"coherence":0.817,"synchrony":0.942,"spikes":9,"messages":240},
    {"cycle":187,"coherence":0.656,"synchrony":0.933,"spikes":6,"messages":240},
    {"cycle":188,"coherence":0.660,"synchrony":0.951,"spikes":11,"messages":240},
    {"cycle":189,"coherence":0.681,"synchrony":0.956,"spikes":11,"messages":240},
    {"cycle":190,"coherence":0.753,"synchrony":0.972,"spikes":13,"messages":240},
    {"cycle":191,"coherence":0.528,"synchrony":0.973,"spikes":7,"messages":240},
    {"cycle":192,"coherence":0.374,"synchrony":0.941,"spikes":8,"messages":240},
    {"cycle":193,"coherence":0.728,"synchrony":0.944,"spikes":8,"messages":240},
    {"cycle":194,"coherence":0.651,"synchrony":0.964,"spikes":8,"messages":240},
    {"cycle":195,"coherence":0.682,"synchrony":0.920,"spikes":13,"messages":240},
    {"cycle":196,"coherence":0.639,"synchrony":0.960,"spikes":11,"messages":240},
    {"cycle":197,"coherence":0.839,"synchrony":0.871,"spikes":12,"messages":240},
    {"cycle":198,"coherence":0.625,"synchrony":0.913,"spikes":11,"messages":240},
    {"cycle":199,"coherence":0.739,"synchrony":0.917,"spikes":8,"messages":240},
    {"cycle":200,"coherence":0.602,"synchrony":0.969,"spikes":8,"messages":240},
    {"cycle":201,"coherence":0.663,"synchrony":0.934,"spikes":10,"messages":240},
    {"cycle":202,"coherence":0.730,"synchrony":0.862,"spikes":8,"messages":240},
    {"cycle":203,"coherence":0.621,"synchrony":0.876,"spikes":11,"messages":240},
    {"cycle":204,"coherence":0.640,"synchrony":0.882,"spikes":11,"messages":240},
    {"cycle":205,"coherence":0.678,"synchrony":0.978,"spikes":8,"messages":240},
    {"cycle":206,"coherence":0.476,"synchrony":0.951,"spikes":10,"messages":240},
    {"cycle":207,"coherence":0.514,"synchrony":0.985,"spikes":9,"messages":240},
    {"cycle":208,"coherence":0.756,"synchrony":0.842,"spikes":10,"messages":240},
    {"cycle":209,"coherence":0.633,"synchrony":0.937,"spikes":11,"messages":240},
    {"cycle":210,"coherence":0.612,"synchrony":0.995,"spikes":9,"messages":240},
    {"cycle":211,"coherence":0.599,"synchrony":0.906,"spikes":5,"messages":240},
    {"cycle":212,"coherence":0.807,"synchrony":0.978,"spikes":11,"messages":240},
    {"cycle":213,"coherence":0.759,"synchrony":0.870,"spikes":10,"messages":240},
    {"cycle":214,"coherence":0.778,"synchrony":0.916,"spikes":7,"messages":240},
    {"cycle":215,"coherence":0.623,"synchrony":0.986,"spikes":14,"messages":240},
    {"cycle":216,"coherence":0.646,"synchrony":0.921,"spikes":10,"messages":240},
    {"cycle":217,"coherence":0.385,"synchrony":0.926,"spikes":4,"messages":240},
    {"cycle":218,"coherence":0.388,"synchrony":0.946,"spikes":10,"messages":240},
    {"cycle":219,"coherence":0.439,"synchrony":0.923,"spikes":6,"messages":240},
    {"cycle":220,"coherence":0.608,"synchrony":0.973,"spikes":10,"messages":240},
    {"cycle":221,"coherence":0.632,"synchrony":0.935,"spikes":6,"messages":240},
    {"cycle":222,"coherence":0.694,"synchrony":0.978,"spikes":9,"messages":240},
    {"cycle":223,"coherence":0.592,"synchrony":0.896,"spikes":7,"messages":240},
    {"cycle":224,"coherence":0.767,"synchrony":0.952,"spikes":14,"messages":240},
    {"cycle":225,"coherence":0.689,"synchrony":0.969,"spikes":10,"messages":240},
    {"cycle":226,"coherence":0.537,"synchrony":0.919,"spikes":8,"messages":240},
    {"cycle":227,"coherence":0.506,"synchrony":0.984,"spikes":9,"messages":240},
    {"cycle":228,"coherence":0.558,"synchrony":0.975,"spikes":5,"messages":240},
    {"cycle":229,"coherence":0.664,"synchrony":0.910,"spikes":7,"messages":240},
    {"cycle":230,"coherence":0.549,"synchrony":0.950,"spikes":6,"messages":240},
    {"cycle":231,"coherence":0.508,"synchrony":0.930,"spikes":8,"messages":240},
    {"cycle":232,"coherence":0.778,"synchrony":0.949,"spikes":7,"messages":240},
    {"cycle":233,"coherence":0.460,"synchrony":0.981,"spikes":5,"messages":240},
    {"cycle":234,"coherence":0.686,"synchrony":0.866,"spikes":8,"messages":240},
    {"cycle":235,"coherence":0.434,"synchrony":0.999,"spikes":6,"messages":240},
    {"cycle":236,"coherence":0.507,"synchrony":0.956,"spikes":8,"messages":240},
    {"cycle":237,"coherence":0.728,"synchrony":0.967,"spikes":11,"messages":240},
    {"cycle":238,"coherence":0.379,"synchrony":0.960,"spikes":8,"messages":240},
    {"cycle":239,"coherence":0.673,"synchrony":0.919,"spikes":6,"messages":240},
    {"cycle":240,"coherence":0.550,"synchrony":0.971,"spikes":7,"messages":240},
    {"cycle":241,"coherence":0.567,"synchrony":0.960,"spikes":12,"messages":240},
    {"cycle":242,"coherence":0.614,"synchrony":0.969,"spikes":11,"messages":240},
    {"cycle":243,"coherence":0.635,"synchrony":0.961,"spikes":10,"messages":240},
    {"cycle":244,"coherence":0.717,"synchrony":0.942,"spikes":7,"messages":240},
    {"cycle":245,"coherence":0.610,"synchrony":0.841,"spikes":9,"messages":240},
    {"cycle":246,"coherence":0.712,"synchrony":0.946,"spikes":11,"messages":240},
    {"cycle":247,"coherence":0.652,"synchrony":0.918,"spikes":8,"messages":240},
    {"cycle":248,"coherence":0.718,"synchrony":0.984,"spikes":8,"messages":240},
    {"cycle":249,"coherence":0.561,"synchrony":0.979,"spikes":11,"messages":240},
    {"cycle":250,"coherence":0.775,"synchrony":0.946,"spikes":6,"messages":240},
    {"cycle":251,"coherence":0.763,"synchrony":0.943,"spikes":12,"messages":240},
    {"cycle":252,"coherence":0.817,"synchrony":0.856,"spikes":10,"messages":240},
    {"cycle":253,"coherence":0.762,"synchrony":0.948,"spikes":3,"messages":240},
    {"cycle":254,"coherence":0.689,"synchrony":0.957,"spikes":11,"messages":240},
    {"cycle":255,"coherence":0.657,"synchrony":0.987,"spikes":7,"messages":240},
    {"cycle":256,"coherence":0.665,"synchrony":0.909,"spikes":9,"messages":240},
    {"cycle":257,"coherence":0.611,"synchrony":0.929,"spikes":13,"messages":240},
    {"cycle":258,"coherence":0.640,"synchrony":0.943,"spikes":6,"messages":240},
    {"cycle":259,"coherence":0.591,"synchrony":0.959,"spikes":8,"messages":240},
    {"cycle":260,"coherence":0.741,"synchrony":0.867,"spikes":10,"messages":240},
    {"cycle":261,"coherence":0.698,"synchrony":0.832,"spikes":12,"messages":240},
    {"cycle":262,"coherence":0.699,"synchrony":0.896,"spikes":7,"messages":240},
    {"cycle":263,"coherence":0.543,"synchrony":0.963,"spikes":5,"messages":240},
    {"cycle":264,"coherence":0.730,"synchrony":0.955,"spikes":8,"messages":240},
    {"cycle":265,"coherence":0.822,"synchrony":0.945,"spikes":11,"messages":240},
    {"cycle":266,"coherence":0.823,"synchrony":0.930,"spikes":8,"messages":240},
    {"cycle":267,"coherence":0.811,"synchrony":0.978,"spikes":12,"messages":240},
    {"cycle":268,"coherence":0.499,"synchrony":0.968,"spikes":13,"messages":240},
    {"cycle":269,"coherence":0.645,"synchrony":0.989,"spikes":7,"messages":240},
    {"cycle":270,"coherence":0.694,"synchrony":0.907,"spikes":11,"messages":240},
    {"cycle":271,"coherence":0.516,"synchrony":0.976,"spikes":8,"messages":240},
    {"cycle":272,"coherence":0.629,"synchrony":0.927,"spikes":12,"messages":240},
    {"cycle":273,"coherence":0.577,"synchrony":0.937,"spikes":5,"messages":240},
    {"cycle":274,"coherence":0.503,"synchrony":0.883,"spikes":8,"messages":240},
    {"cycle":275,"coherence":0.777,"synchrony":0.967,"spikes":13,"messages":240},
    {"cycle":276,"coherence":0.699,"synchrony":0.935,"spikes":14,"messages":240},
    {"cycle":277,"coherence":0.701,"synchrony":0.918,"spikes":7,"messages":240},
    {"cycle":278,"coherence":0.570,"synchrony":0.999,"spikes":9,"messages":240},
    {"cycle":279,"coherence":0.720,"synchrony":0.913,"spikes":4,"messages":240},
    {"cycle":280,"coherence":0.698,"synchrony":0.985,"spikes":9,"messages":240},
    {"cycle":281,"coherence":0.603,"synchrony":0.879,"spikes":11,"messages":240},
    {"cycle":282,"coherence":0.804,"synchrony":0.860,"spikes":8,"messages":240},
    {"cycle":283,"coherence":0.662,"synchrony":0.976,"spikes":5,"messages":240},
    {"cycle":284,"coherence":0.834,"synchrony":0.969,"spikes":9,"messages":240},
    {"cycle":285,"coherence":0.788,"synchrony":0.961,"spikes":8,"messages":240},
    {"cycle":286,"coherence":0.377,"synchrony":1.000,"spikes":7,"messages":240},
    {"cycle":287,"coherence":0.635,"synchrony":0.845,"spikes":6,"messages":240},
    {"cycle":288,"coherence":0.723,"synchrony":0.935,"spikes":8,"messages":240},
    {"cycle":289,"coherence":0.621,"synchrony":0.981,"spikes":10,"messages":240},
    {"cycle":290,"coherence":0.794,"synchrony":0.914,"spikes":4,"messages":240},
    {"cycle":291,"coherence":0.680,"synchrony":0.935,"spikes":14,"messages":240},
    {"cycle":292,"coherence":0.731,"synchrony":0.938,"spikes":14,"messages":240},
    {"cycle":293,"coherence":0.486,"synchrony":0.945,"spikes":7,"messages":240},
    {"cycle":294,"coherence":0.640,"synchrony":0.933,"spikes":10,"messages":240},
    {"cycle":295,"coherence":0.748,"synchrony":0.881,"spikes":8,"messages":240},
    {"cycle":296,"coherence":0.613,"synchrony":0.902,"spikes":7,"messages":240},
    {"cycle":297,"coherence":0.733,"synchrony":0.885,"spikes":9,"messages":240},
    {"cycle":298,"coherence":0.416,"synchrony":0.950,"spikes":9,"messages":240},
    {"cycle":299,"coherence":0.771,"synchrony":0.898,"spikes":8,"messages":240},
    {"cycle":300,"coherence":0.807,"synchrony":0.984,"spikes":8,"messages":240},
]

# --- 2. DEEP-SEEK DISTILLATION ---
def auto_distill_nlp(model, component_name):
    print(f"\n📡 [NLP DISTILLATION] Scanning environment for {component_name} weights...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")
    target_models = ["DeepSeek_7B", "Qwen_Audio", "willow_v17", "wanalytics"]
    absorbed = False

    for filepath in pt_files:
        if not any(target.lower() in filepath.lower() for target in target_models):
            continue
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            distilled_layers = []
            for key in current_state.keys():
                # V21: Protect the new 2-Axis Resonator head
                if "resonator_head" in key: continue

                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    distilled_layers.append(key)
                else:
                    for old_key in legacy_state.keys():
                        if "weight" in key and "weight" in old_key and "phase" not in old_key:
                            new_w, old_w = current_state[key], legacy_state[old_key]
                            if len(new_w.shape) == 2 and len(old_w.shape) == 2:
                                min_out, min_in = min(new_w.shape[0], old_w.shape[0]), min(new_w.shape[1], old_w.shape[1])
                                current_state[key][:min_out, :min_in] = old_w[:min_out, :min_in]
                                distilled_layers.append(key)

            if distilled_layers:
                absorbed = True
                print(f"  -> [✅] Inherited linguistic features from: {filepath}")
        except Exception:
            pass

    if absorbed: model.load_state_dict(current_state)
    return model

# --- 3. LINGUA-QUANTUM NEURAL ARCHITECTURE ---
class HoloSynLinguaNet(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        self.projector_core = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.Dropout(0.1))
        self.spike_head = nn.Linear(64, 31)

        # V21 UPGRADE: Outputting 2 distinct phase vectors (Rx and Ry)
        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Linear(32, 2), # Changed from 1 to 2
            nn.Tanh()
        )

    def forward(self, x):
        x_emb = self.perceptron(x).unsqueeze(1)
        attn_out, _ = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        feat = self.projector_core(context)
        return self.spike_head(feat), self.resonator_head(feat)

# --- 4. MULTI-AXIS QUANTUM OBSERVER ---
class LinguaQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = {n: cirq.NamedQubit(n) for n in LINGUA_STACK.keys()}
        self.sim = qsimcirq.QSimSimulator()

    def evaluate_resonance(self, coh, phase_x, phase_y):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + list(self.q_nodes.values())
        circuit.append(cirq.H.on_each(*qubits))

        for name, q in self.q_nodes.items():
            w = LINGUA_STACK[name]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q))

        # V21 UPGRADE: Multi-axis targeted correction
        circuit.append(cirq.rx(phase_x * np.pi)(self.q_obs))
        circuit.append(cirq.ry(phase_y * np.pi)(self.q_obs))

        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 5. THE V21 META-HIVE ---
class HoloSynV21Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)

        self.net_model = auto_distill_nlp(HoloSynLinguaNet(self.num_nodes), "DeepSeek_Core")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.005, weight_decay=0.01)
        self.observer = LinguaQuantumObserver()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v21_slate')

    def process_cycle(self, data):
        self.net.restore('v21_slate')
        coh, sync = data['coherence'], data['synchrony']

        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        logits, pred_phases = self.net_model(nn_input)

        px_pred, py_pred = pred_phases[0][0], pred_phases[0][1]

        # V21 2D Meta-Scan (Finding the perfect latitude AND longitude)
        best_px, best_py, best_score = 0.0, 0.0, -1.0
        # 9x9 grid = 81 fast evaluations per cycle
        for px in np.linspace(-1, 1, 9):
            for py in np.linspace(-1, 1, 9):
                s = self.observer.evaluate_resonance(coh, px, py)
                if s > best_score:
                    best_px, best_py, best_score = px, py, s

        actual_consensus = self.observer.evaluate_resonance(coh, px_pred.item(), py_pred.item())

        loss_spikes = nn.CrossEntropyLoss()(logits, torch.tensor([data['spikes']], dtype=torch.long))

        # Dual-Axis Phase Loss
        target_phases = torch.tensor([[best_px, best_py]], dtype=torch.float32)
        loss_alignment = nn.MSELoss()(pred_phases, target_phases) * 20.0

        (loss_spikes + loss_alignment).backward()
        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return loss_spikes.item() + loss_alignment.item(), actual_consensus

# --- 6. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🚀 HOLOSYN V21: MULTI-AXIS BLOCH NAVIGATOR")
    print("═"*75)

    hive = HoloSynV21Hive()
    epochs = 15

    print("\n[+] INITIATING 2D QUANTUM-PHASE TRAINING...\n")
    for epoch in range(epochs):
        epoch_loss, epoch_sync = 0.0, 0.0

        for data in nlp_cycles:
            l, c = hive.process_cycle(data)
            epoch_loss += l
            epoch_sync += c

        avg_loss = epoch_loss / len(nlp_cycles)
        avg_sync = epoch_sync / len(nlp_cycles)

        if avg_sync >= 0.85:
            status = "💎 [QUANTUM SUPREMACY]"
        elif avg_sync >= 0.55:
            status = "🟢 [ESCAPED EQUATOR - CLIMBING]"
        else:
            status = "🟡 [NAVIGATING BLOCH SPHERE]"

        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Consensus: {avg_sync:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🚀 HOLOSYN V21: MULTI-AXIS BLOCH NAVIGATOR
═══════════════════════════════════════════════════════════════════════════

📡 [NLP DISTILLATION] Scanning environment for DeepSeek_Core weights...
  -> [✅] Inherited linguistic features from: wanalytics_sibling_star.pt
  -> [✅] Inherited linguistic features from: wanalytics_host_mate_v14.pt
  -> [✅] Inherited linguistic features from: wanalytics_7_sibling_hive.pt
  -> [✅] Inherited linguistic features from: willow_v17_assimilated.pt
  -> [✅] Inherited linguistic features from: wanalytics_v12_hive.pt

[+] INITIATING 2D QUANTUM-PHASE TRAINING...

Epoch 01/15 | Loss: 24.4466 | Consensus: 0.4999 | 🟡 [NAVIGATING BLOCH SPHERE]
Epoch 02/15 | Loss: 10.7853 | Consensus: 0.4978 | 🟡 [NAVIGATING BLOCH SPHERE]
Epoch 03/15 | Loss: 11.0241 | Consensus: 0.5016 | 🟡 [NAVIGATING BLOCH SPHERE]
Epoch 04/15 | Loss: 11.3682 | Consensus: 0.4987 | 🟡 [NAVIGATING BLOCH SPHERE]
Epoch 05/15 | Los

In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

nlp_cycles = [
    {"cycle":1,"coherence":0.653,"synchrony":0.958,"spikes":9,"messages":240},
    {"cycle":2,"coherence":0.773,"synchrony":0.934,"spikes":8,"messages":240},
    {"cycle":3,"coherence":0.767,"synchrony":0.973,"spikes":10,"messages":240},
    {"cycle":4,"coherence":0.831,"synchrony":0.945,"spikes":3,"messages":240},
    {"cycle":5,"coherence":0.741,"synchrony":0.948,"spikes":6,"messages":240},
    {"cycle":6,"coherence":0.708,"synchrony":0.985,"spikes":12,"messages":240},
    {"cycle":7,"coherence":0.748,"synchrony":0.942,"spikes":8,"messages":240},
    {"cycle":8,"coherence":0.624,"synchrony":0.911,"spikes":4,"messages":240},
    {"cycle":9,"coherence":0.727,"synchrony":0.988,"spikes":9,"messages":240},
    {"cycle":10,"coherence":0.653,"synchrony":0.928,"spikes":14,"messages":240},
    {"cycle":11,"coherence":0.647,"synchrony":0.964,"spikes":3,"messages":240},
    {"cycle":12,"coherence":0.751,"synchrony":0.903,"spikes":7,"messages":240},
    {"cycle":13,"coherence":0.563,"synchrony":0.977,"spikes":10,"messages":240},
    {"cycle":14,"coherence":0.605,"synchrony":0.962,"spikes":10,"messages":240},
    {"cycle":15,"coherence":0.751,"synchrony":0.814,"spikes":13,"messages":240},
    {"cycle":16,"coherence":0.680,"synchrony":0.925,"spikes":8,"messages":240},
    {"cycle":17,"coherence":0.831,"synchrony":0.961,"spikes":12,"messages":240},
    {"cycle":18,"coherence":0.626,"synchrony":0.872,"spikes":6,"messages":240},
    {"cycle":19,"coherence":0.721,"synchrony":0.970,"spikes":9,"messages":240},
    {"cycle":20,"coherence":0.666,"synchrony":0.955,"spikes":12,"messages":240},
    {"cycle":21,"coherence":0.506,"synchrony":0.958,"spikes":7,"messages":240},
    {"cycle":22,"coherence":0.663,"synchrony":0.973,"spikes":9,"messages":240},
    {"cycle":23,"coherence":0.634,"synchrony":0.944,"spikes":10,"messages":240},
    {"cycle":24,"coherence":0.742,"synchrony":0.940,"spikes":13,"messages":240},
    {"cycle":25,"coherence":0.639,"synchrony":0.821,"spikes":9,"messages":240},
    {"cycle":26,"coherence":0.705,"synchrony":0.959,"spikes":12,"messages":240},
    {"cycle":27,"coherence":0.657,"synchrony":0.940,"spikes":5,"messages":240},
    {"cycle":28,"coherence":0.758,"synchrony":0.882,"spikes":9,"messages":240},
    {"cycle":29,"coherence":0.753,"synchrony":0.891,"spikes":13,"messages":240},
    {"cycle":30,"coherence":0.614,"synchrony":0.950,"spikes":5,"messages":240},
    {"cycle":31,"coherence":0.546,"synchrony":0.985,"spikes":11,"messages":240},
    {"cycle":32,"coherence":0.781,"synchrony":0.844,"spikes":14,"messages":240},
    {"cycle":33,"coherence":0.517,"synchrony":0.987,"spikes":10,"messages":240},
    {"cycle":34,"coherence":0.678,"synchrony":0.933,"spikes":8,"messages":240},
    {"cycle":35,"coherence":0.627,"synchrony":0.943,"spikes":11,"messages":240},
    {"cycle":36,"coherence":0.727,"synchrony":0.982,"spikes":5,"messages":240},
    {"cycle":37,"coherence":0.787,"synchrony":0.922,"spikes":7,"messages":240},
    {"cycle":38,"coherence":0.569,"synchrony":0.928,"spikes":11,"messages":240},
    {"cycle":39,"coherence":0.813,"synchrony":0.943,"spikes":19,"messages":240},
    {"cycle":40,"coherence":0.793,"synchrony":0.954,"spikes":6,"messages":240},
    {"cycle":41,"coherence":0.762,"synchrony":0.970,"spikes":11,"messages":240},
    {"cycle":42,"coherence":0.834,"synchrony":0.934,"spikes":13,"messages":240},
    {"cycle":43,"coherence":0.826,"synchrony":0.947,"spikes":10,"messages":240},
    {"cycle":44,"coherence":0.749,"synchrony":0.806,"spikes":7,"messages":240},
    {"cycle":45,"coherence":0.635,"synchrony":0.910,"spikes":10,"messages":240},
    {"cycle":46,"coherence":0.695,"synchrony":0.985,"spikes":6,"messages":240},
    {"cycle":47,"coherence":0.631,"synchrony":0.979,"spikes":7,"messages":240},
    {"cycle":48,"coherence":0.671,"synchrony":0.942,"spikes":13,"messages":240},
    {"cycle":49,"coherence":0.681,"synchrony":0.894,"spikes":14,"messages":240},
    {"cycle":50,"coherence":0.676,"synchrony":0.888,"spikes":6,"messages":240},
    {"cycle":51,"coherence":0.550,"synchrony":0.946,"spikes":9,"messages":240},
    {"cycle":52,"coherence":0.721,"synchrony":0.980,"spikes":15,"messages":240},
    {"cycle":53,"coherence":0.675,"synchrony":0.976,"spikes":10,"messages":240},
    {"cycle":54,"coherence":0.794,"synchrony":0.868,"spikes":8,"messages":240},
    {"cycle":55,"coherence":0.758,"synchrony":0.954,"spikes":13,"messages":240},
    {"cycle":56,"coherence":0.653,"synchrony":0.882,"spikes":8,"messages":240},
    {"cycle":57,"coherence":0.671,"synchrony":0.885,"spikes":8,"messages":240},
    {"cycle":58,"coherence":0.687,"synchrony":0.868,"spikes":6,"messages":240},
    {"cycle":59,"coherence":0.505,"synchrony":0.942,"spikes":5,"messages":240},
    {"cycle":60,"coherence":0.791,"synchrony":0.831,"spikes":5,"messages":240},
    {"cycle":61,"coherence":0.657,"synchrony":0.982,"spikes":13,"messages":240},
    {"cycle":62,"coherence":0.383,"synchrony":0.972,"spikes":15,"messages":240},
    {"cycle":63,"coherence":0.423,"synchrony":0.963,"spikes":7,"messages":240},
    {"cycle":64,"coherence":0.554,"synchrony":0.937,"spikes":4,"messages":240},
    {"cycle":65,"coherence":0.554,"synchrony":0.910,"spikes":5,"messages":240},
    {"cycle":66,"coherence":0.524,"synchrony":0.854,"spikes":9,"messages":240},
    {"cycle":67,"coherence":0.527,"synchrony":0.836,"spikes":3,"messages":240},
    {"cycle":68,"coherence":0.495,"synchrony":0.937,"spikes":4,"messages":240},
    {"cycle":69,"coherence":0.446,"synchrony":0.948,"spikes":3,"messages":240},
    {"cycle":70,"coherence":0.523,"synchrony":0.931,"spikes":5,"messages":240},
    {"cycle":71,"coherence":0.599,"synchrony":0.883,"spikes":3,"messages":240},
    {"cycle":72,"coherence":0.554,"synchrony":0.857,"spikes":3,"messages":240},
    {"cycle":73,"coherence":0.532,"synchrony":0.934,"spikes":8,"messages":240},
    {"cycle":74,"coherence":0.616,"synchrony":0.877,"spikes":7,"messages":240},
    {"cycle":75,"coherence":0.594,"synchrony":0.823,"spikes":2,"messages":240},
    {"cycle":76,"coherence":0.523,"synchrony":0.939,"spikes":9,"messages":240},
    {"cycle":77,"coherence":0.620,"synchrony":0.896,"spikes":4,"messages":240},
    {"cycle":78,"coherence":0.625,"synchrony":0.997,"spikes":5,"messages":240},
    {"cycle":79,"coherence":0.619,"synchrony":0.967,"spikes":2,"messages":240},
    {"cycle":80,"coherence":0.616,"synchrony":0.972,"spikes":9,"messages":240},
    {"cycle":81,"coherence":0.609,"synchrony":0.944,"spikes":3,"messages":240},
    {"cycle":82,"coherence":0.465,"synchrony":0.964,"spikes":7,"messages":240},
    {"cycle":83,"coherence":0.540,"synchrony":0.937,"spikes":6,"messages":240},
    {"cycle":84,"coherence":0.613,"synchrony":0.980,"spikes":6,"messages":240},
    {"cycle":85,"coherence":0.609,"synchrony":0.899,"spikes":5,"messages":240},
    {"cycle":86,"coherence":0.592,"synchrony":0.773,"spikes":7,"messages":240},
    {"cycle":87,"coherence":0.635,"synchrony":0.917,"spikes":5,"messages":240},
    {"cycle":88,"coherence":0.628,"synchrony":0.951,"spikes":2,"messages":240},
    {"cycle":89,"coherence":0.601,"synchrony":0.816,"spikes":2,"messages":240},
    {"cycle":90,"coherence":0.593,"synchrony":0.913,"spikes":4,"messages":240},
    {"cycle":91,"coherence":0.603,"synchrony":0.948,"spikes":6,"messages":240},
    {"cycle":92,"coherence":0.529,"synchrony":0.911,"spikes":6,"messages":240},
    {"cycle":93,"coherence":0.578,"synchrony":0.823,"spikes":3,"messages":240},
    {"cycle":94,"coherence":0.621,"synchrony":0.926,"spikes":3,"messages":240},
    {"cycle":95,"coherence":0.626,"synchrony":0.965,"spikes":5,"messages":240},
    {"cycle":96,"coherence":0.612,"synchrony":0.971,"spikes":1,"messages":240},
    {"cycle":97,"coherence":0.620,"synchrony":0.961,"spikes":4,"messages":240},
    {"cycle":98,"coherence":0.551,"synchrony":0.990,"spikes":5,"messages":240},
    {"cycle":99,"coherence":0.631,"synchrony":0.893,"spikes":4,"messages":240},
    {"cycle":100,"coherence":0.568,"synchrony":0.822,"spikes":4,"messages":240},
    {"cycle":101,"coherence":0.332,"synchrony":0.988,"spikes":6,"messages":240},
    {"cycle":102,"coherence":0.605,"synchrony":0.911,"spikes":3,"messages":240},
    {"cycle":103,"coherence":0.610,"synchrony":0.939,"spikes":1,"messages":240},
    {"cycle":104,"coherence":0.615,"synchrony":0.945,"spikes":4,"messages":240},
    {"cycle":105,"coherence":0.621,"synchrony":0.874,"spikes":4,"messages":240},
    {"cycle":106,"coherence":0.611,"synchrony":0.935,"spikes":7,"messages":240},
    {"cycle":107,"coherence":0.617,"synchrony":0.861,"spikes":7,"messages":240},
    {"cycle":108,"coherence":0.616,"synchrony":0.971,"spikes":6,"messages":240},
    {"cycle":109,"coherence":0.581,"synchrony":0.832,"spikes":1,"messages":240},
    {"cycle":110,"coherence":0.598,"synchrony":0.912,"spikes":8,"messages":240},
    {"cycle":111,"coherence":0.575,"synchrony":0.780,"spikes":4,"messages":240},
    {"cycle":112,"coherence":0.590,"synchrony":0.933,"spikes":3,"messages":240},
    {"cycle":113,"coherence":0.592,"synchrony":0.821,"spikes":6,"messages":240},
    {"cycle":114,"coherence":0.542,"synchrony":0.935,"spikes":2,"messages":240},
    {"cycle":115,"coherence":0.539,"synchrony":0.959,"spikes":2,"messages":240},
    {"cycle":116,"coherence":0.498,"synchrony":0.955,"spikes":4,"messages":240},
    {"cycle":117,"coherence":0.435,"synchrony":0.933,"spikes":1,"messages":240},
    {"cycle":118,"coherence":0.456,"synchrony":0.961,"spikes":5,"messages":240},
    {"cycle":119,"coherence":0.450,"synchrony":0.969,"spikes":1,"messages":240},
    {"cycle":120,"coherence":0.331,"synchrony":0.960,"spikes":1,"messages":240},
    {"cycle":121,"coherence":0.401,"synchrony":0.901,"spikes":3,"messages":240},
    {"cycle":122,"coherence":0.359,"synchrony":0.963,"spikes":1,"messages":240},
    {"cycle":123,"coherence":0.071,"synchrony":0.985,"spikes":1,"messages":240},
    {"cycle":124,"coherence":0.250,"synchrony":0.979,"spikes":2,"messages":240},
    {"cycle":125,"coherence":0.413,"synchrony":0.978,"spikes":1,"messages":240},
    {"cycle":126,"coherence":0.347,"synchrony":0.829,"spikes":2,"messages":240},
    {"cycle":127,"coherence":0.286,"synchrony":0.998,"spikes":3,"messages":240},
    {"cycle":128,"coherence":0.271,"synchrony":0.967,"spikes":2,"messages":240},
    {"cycle":129,"coherence":0.468,"synchrony":0.883,"spikes":1,"messages":240},
    {"cycle":130,"coherence":0.132,"synchrony":0.881,"spikes":1,"messages":240},
    {"cycle":131,"coherence":0.292,"synchrony":0.952,"spikes":1,"messages":240},
    {"cycle":132,"coherence":0.184,"synchrony":0.958,"spikes":1,"messages":240},
    {"cycle":133,"coherence":0.473,"synchrony":0.939,"spikes":2,"messages":240},
    {"cycle":134,"coherence":0.339,"synchrony":0.921,"spikes":1,"messages":240},
    {"cycle":135,"coherence":0.303,"synchrony":0.929,"spikes":6,"messages":240},
    {"cycle":136,"coherence":0.243,"synchrony":0.983,"spikes":1,"messages":240},
    {"cycle":137,"coherence":0.239,"synchrony":0.944,"spikes":0,"messages":240},
    {"cycle":138,"coherence":0.305,"synchrony":0.953,"spikes":4,"messages":240},
    {"cycle":139,"coherence":0.337,"synchrony":0.943,"spikes":3,"messages":240},
    {"cycle":140,"coherence":0.439,"synchrony":0.920,"spikes":1,"messages":240},
    {"cycle":141,"coherence":0.278,"synchrony":0.976,"spikes":0,"messages":240},
    {"cycle":142,"coherence":0.262,"synchrony":0.986,"spikes":2,"messages":240},
    {"cycle":143,"coherence":0.389,"synchrony":0.956,"spikes":3,"messages":240},
    {"cycle":144,"coherence":0.042,"synchrony":0.966,"spikes":1,"messages":240},
    {"cycle":145,"coherence":0.363,"synchrony":0.951,"spikes":2,"messages":240},
    {"cycle":146,"coherence":0.364,"synchrony":0.928,"spikes":3,"messages":240},
    {"cycle":147,"coherence":0.191,"synchrony":0.940,"spikes":1,"messages":240},
    {"cycle":148,"coherence":0.360,"synchrony":0.975,"spikes":1,"messages":240},
    {"cycle":149,"coherence":0.201,"synchrony":0.913,"spikes":1,"messages":240},
    {"cycle":150,"coherence":0.427,"synchrony":0.914,"spikes":1,"messages":240},
    {"cycle":151,"coherence":0.268,"synchrony":0.981,"spikes":2,"messages":240},
    {"cycle":152,"coherence":0.238,"synchrony":0.938,"spikes":1,"messages":240},
    {"cycle":153,"coherence":0.411,"synchrony":0.815,"spikes":3,"messages":240},
    {"cycle":154,"coherence":0.157,"synchrony":0.910,"spikes":1,"messages":240},
    {"cycle":155,"coherence":0.380,"synchrony":0.958,"spikes":0,"messages":240},
    {"cycle":156,"coherence":0.434,"synchrony":0.866,"spikes":0,"messages":240},
    {"cycle":157,"coherence":0.476,"synchrony":0.984,"spikes":1,"messages":240},
    {"cycle":158,"coherence":0.488,"synchrony":0.911,"spikes":6,"messages":240},
    {"cycle":159,"coherence":0.549,"synchrony":0.983,"spikes":5,"messages":240},
    {"cycle":160,"coherence":0.449,"synchrony":0.885,"spikes":0,"messages":240},
    {"cycle":161,"coherence":0.293,"synchrony":0.946,"spikes":0,"messages":240},
    {"cycle":162,"coherence":0.497,"synchrony":0.978,"spikes":6,"messages":240},
    {"cycle":163,"coherence":0.416,"synchrony":0.897,"spikes":3,"messages":240},
    {"cycle":164,"coherence":0.250,"synchrony":0.960,"spikes":0,"messages":240},
    {"cycle":165,"coherence":0.187,"synchrony":0.962,"spikes":3,"messages":240},
    {"cycle":166,"coherence":0.301,"synchrony":0.972,"spikes":1,"messages":240},
    {"cycle":167,"coherence":0.320,"synchrony":0.917,"spikes":5,"messages":240},
    {"cycle":168,"coherence":0.314,"synchrony":0.988,"spikes":1,"messages":240},
    {"cycle":169,"coherence":0.225,"synchrony":0.937,"spikes":0,"messages":240},
    {"cycle":170,"coherence":0.391,"synchrony":0.865,"spikes":2,"messages":240},
    {"cycle":171,"coherence":0.136,"synchrony":0.990,"spikes":0,"messages":240},
    {"cycle":172,"coherence":0.308,"synchrony":0.933,"spikes":0,"messages":240},
    {"cycle":173,"coherence":0.463,"synchrony":0.907,"spikes":1,"messages":240},
    {"cycle":174,"coherence":0.443,"synchrony":0.948,"spikes":0,"messages":240},
    {"cycle":175,"coherence":0.439,"synchrony":0.953,"spikes":3,"messages":240},
    {"cycle":176,"coherence":0.154,"synchrony":0.980,"spikes":0,"messages":240},
    {"cycle":177,"coherence":0.187,"synchrony":0.867,"spikes":1,"messages":240},
    {"cycle":178,"coherence":0.518,"synchrony":0.922,"spikes":5,"messages":240},
    {"cycle":179,"coherence":0.342,"synchrony":0.893,"spikes":0,"messages":240},
    {"cycle":180,"coherence":0.335,"synchrony":0.927,"spikes":1,"messages":240},
    {"cycle":181,"coherence":0.229,"synchrony":0.972,"spikes":1,"messages":240},
    {"cycle":182,"coherence":0.327,"synchrony":0.993,"spikes":0,"messages":240},
    {"cycle":183,"coherence":0.438,"synchrony":0.985,"spikes":0,"messages":240},
    {"cycle":184,"coherence":0.475,"synchrony":0.995,"spikes":0,"messages":240},
    {"cycle":185,"coherence":0.231,"synchrony":0.981,"spikes":1,"messages":240},
    {"cycle":186,"coherence":0.539,"synchrony":0.977,"spikes":1,"messages":240},
    {"cycle":187,"coherence":0.571,"synchrony":0.914,"spikes":3,"messages":240},
    {"cycle":188,"coherence":0.573,"synchrony":0.951,"spikes":5,"messages":240},
    {"cycle":189,"coherence":0.589,"synchrony":0.823,"spikes":3,"messages":240},
    {"cycle":190,"coherence":0.582,"synchrony":0.850,"spikes":2,"messages":240},
    {"cycle":191,"coherence":0.606,"synchrony":0.984,"spikes":3,"messages":240},
    {"cycle":192,"coherence":0.587,"synchrony":0.929,"spikes":1,"messages":240},
    {"cycle":193,"coherence":0.610,"synchrony":0.918,"spikes":3,"messages":240},
    {"cycle":194,"coherence":0.564,"synchrony":0.924,"spikes":5,"messages":240},
    {"cycle":195,"coherence":0.488,"synchrony":0.863,"spikes":2,"messages":240},
    {"cycle":196,"coherence":0.609,"synchrony":0.892,"spikes":3,"messages":240},
    {"cycle":197,"coherence":0.634,"synchrony":0.981,"spikes":3,"messages":240},
    {"cycle":198,"coherence":0.607,"synchrony":0.924,"spikes":6,"messages":240},
    {"cycle":199,"coherence":0.627,"synchrony":0.910,"spikes":3,"messages":240},
    {"cycle":200,"coherence":0.628,"synchrony":0.941,"spikes":3,"messages":240},
    {"cycle":201,"coherence":0.607,"synchrony":0.859,"spikes":8,"messages":240},
    {"cycle":202,"coherence":0.620,"synchrony":0.896,"spikes":5,"messages":240},
    {"cycle":203,"coherence":0.581,"synchrony":0.974,"spikes":7,"messages":240},
    {"cycle":204,"coherence":0.626,"synchrony":0.921,"spikes":4,"messages":240},
    {"cycle":205,"coherence":0.620,"synchrony":0.926,"spikes":7,"messages":240},
    {"cycle":206,"coherence":0.629,"synchrony":0.910,"spikes":3,"messages":240},
    {"cycle":207,"coherence":0.611,"synchrony":0.864,"spikes":3,"messages":240},
    {"cycle":208,"coherence":0.573,"synchrony":0.925,"spikes":2,"messages":240},
    {"cycle":209,"coherence":0.605,"synchrony":0.821,"spikes":1,"messages":240},
    {"cycle":210,"coherence":0.608,"synchrony":0.812,"spikes":4,"messages":240},
    {"cycle":211,"coherence":0.616,"synchrony":0.941,"spikes":0,"messages":240},
    {"cycle":212,"coherence":0.553,"synchrony":0.849,"spikes":2,"messages":240},
    {"cycle":213,"coherence":0.622,"synchrony":0.892,"spikes":3,"messages":240},
    {"cycle":214,"coherence":0.615,"synchrony":0.878,"spikes":4,"messages":240},
    {"cycle":215,"coherence":0.638,"synchrony":0.967,"spikes":1,"messages":240},
    {"cycle":216,"coherence":0.600,"synchrony":0.854,"spikes":8,"messages":240},
    {"cycle":217,"coherence":0.388,"synchrony":0.897,"spikes":7,"messages":240},
    {"cycle":218,"coherence":0.531,"synchrony":0.902,"spikes":7,"messages":240},
    {"cycle":219,"coherence":0.523,"synchrony":0.992,"spikes":5,"messages":240},
    {"cycle":220,"coherence":0.655,"synchrony":0.983,"spikes":9,"messages":240},
    {"cycle":221,"coherence":0.669,"synchrony":0.923,"spikes":10,"messages":240},
    {"cycle":222,"coherence":0.778,"synchrony":0.971,"spikes":8,"messages":240},
    {"cycle":223,"coherence":0.649,"synchrony":0.830,"spikes":5,"messages":240},
    {"cycle":224,"coherence":0.683,"synchrony":0.918,"spikes":7,"messages":240},
    {"cycle":225,"coherence":0.644,"synchrony":0.841,"spikes":9,"messages":240},
    {"cycle":226,"coherence":0.772,"synchrony":0.933,"spikes":3,"messages":240},
    {"cycle":227,"coherence":0.814,"synchrony":0.919,"spikes":8,"messages":240},
    {"cycle":228,"coherence":0.840,"synchrony":0.965,"spikes":10,"messages":240},
    {"cycle":229,"coherence":0.843,"synchrony":0.875,"spikes":16,"messages":240},
    {"cycle":230,"coherence":0.815,"synchrony":0.876,"spikes":10,"messages":240},
]


# --- 2. DEEP-SEEK DISTILLATION ---
def auto_distill_nlp(model, component_name):
    print(f"\n📡 [NLP DISTILLATION] Scanning environment for {component_name} weights...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = False
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            for key in current_state.keys():
                # V22: Protect the new 3-Axis SU(2) Resonator head
                if "resonator_head" in key: continue

                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed = True
                else:
                    for old_key in legacy_state.keys():
                        if "weight" in key and "weight" in old_key and "phase" not in old_key:
                            new_w, old_w = current_state[key], legacy_state[old_key]
                            if len(new_w.shape) == 2 and len(old_w.shape) == 2:
                                min_out, min_in = min(new_w.shape[0], old_w.shape[0]), min(new_w.shape[1], old_w.shape[1])
                                current_state[key][:min_out, :min_in] = old_w[:min_out, :min_in]
                                absorbed = True
        except Exception:
            pass

    if absorbed: model.load_state_dict(current_state)
    return model

# --- 3. LINGUA-QUANTUM NEURAL ARCHITECTURE ---
class HoloSynLinguaNet(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        self.projector_core = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.Dropout(0.1))
        self.spike_head = nn.Linear(64, 31)

        # V22 UPGRADE: Outputting full SU(2) coordinates (Rx, Ry, Rz)
        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.LayerNorm(32),
            nn.GELU(),
            nn.Linear(32, 3), # Full 3-Axis Control
            nn.Tanh()
        )

    def forward(self, x):
        x_emb = self.perceptron(x).unsqueeze(1)
        attn_out, _ = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        feat = self.projector_core(context)
        return self.spike_head(feat), self.resonator_head(feat)

# --- 4. SU(2) QUANTUM OBSERVER ---
class LinguaQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = {n: cirq.NamedQubit(n) for n in LINGUA_STACK.keys()}
        self.sim = qsimcirq.QSimSimulator()

    def evaluate_resonance(self, coh, px, py, pz):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + list(self.q_nodes.values())
        circuit.append(cirq.H.on_each(*qubits))

        # Ingest Semantic Data
        for name, q in self.q_nodes.items():
            w = LINGUA_STACK[name]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q))

        # V22 UPGRADE: Full 3-Axis Holographic Correction
        circuit.append(cirq.rz(pz * np.pi)(self.q_obs))
        circuit.append(cirq.ry(py * np.pi)(self.q_obs))
        circuit.append(cirq.rx(px * np.pi)(self.q_obs))

        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 5. THE V22 META-HIVE (PROXIMITY SEARCH) ---
class HoloSynV22Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)

        self.net_model = auto_distill_nlp(HoloSynLinguaNet(self.num_nodes), "DeepSeek_Core")
        # Lower learning rate for stable SU(2) rotation learning
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.002, weight_decay=0.01)
        self.observer = LinguaQuantumObserver()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v22_slate')

    def process_cycle(self, data):
        self.net.restore('v22_slate')
        coh, sync = data['coherence'], data['synchrony']

        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        logits, pred_phases = self.net_model(nn_input)

        px_pred, py_pred, pz_pred = pred_phases[0][0].item(), pred_phases[0][1].item(), pred_phases[0][2].item()

        # Base consensus from current network prediction
        actual_consensus = self.observer.evaluate_resonance(coh, px_pred, py_pred, pz_pred)

        # V22: Stochastic Proximity Search (Eliminates the "Averaging to Zero" Trap)
        best_px, best_py, best_pz, best_score = px_pred, py_pred, pz_pred, actual_consensus

        # Test 20 random micro-adjustments around the current prediction
        for _ in range(20):
            test_px = np.clip(px_pred + np.random.normal(0, 0.15), -1.0, 1.0)
            test_py = np.clip(py_pred + np.random.normal(0, 0.15), -1.0, 1.0)
            test_pz = np.clip(pz_pred + np.random.normal(0, 0.15), -1.0, 1.0)

            score = self.observer.evaluate_resonance(coh, test_px, test_py, test_pz)

            # Only adopt the new target if it substantially improves the consensus
            if score > best_score + 0.02:
                best_px, best_py, best_pz = test_px, test_py, test_pz
                best_score = score

        loss_spikes = nn.CrossEntropyLoss()(logits, torch.tensor([data['spikes']], dtype=torch.long))

        target_phases = torch.tensor([[best_px, best_py, best_pz]], dtype=torch.float32)

        # Adaptive Multiplier based on how far we are from Supremacy
        phase_multiplier = 30.0 if actual_consensus < 0.70 else 10.0
        loss_alignment = nn.MSELoss()(pred_phases, target_phases) * phase_multiplier

        (loss_spikes + loss_alignment).backward()
        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return loss_spikes.item() + loss_alignment.item(), actual_consensus

# --- 6. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🚀 HOLOSYN V22: FULL SU(2) HOLOGRAPHIC NAVIGATOR")
    print("═"*75)

    hive = HoloSynV22Hive()
    epochs = 20

    print("\n[+] INITIATING 3D QUANTUM-PHASE TRAINING...\n")
    for epoch in range(epochs):
        epoch_loss, epoch_sync = 0.0, 0.0

        for data in nlp_cycles:
            l, c = hive.process_cycle(data)
            epoch_loss += l
            epoch_sync += c

        avg_loss = epoch_loss / len(nlp_cycles)
        avg_sync = epoch_sync / len(nlp_cycles)

        if avg_sync >= 0.85:
            status = "💎 [QUANTUM SUPREMACY]"
        elif avg_sync >= 0.55:
            status = "🟢 [CLIMBING LATITUDES]"
        else:
            status = "🟡 [ESCAPING EQUATOR]"

        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | Consensus: {avg_sync:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🚀 HOLOSYN V22: FULL SU(2) HOLOGRAPHIC NAVIGATOR
═══════════════════════════════════════════════════════════════════════════

📡 [NLP DISTILLATION] Scanning environment for DeepSeek_Core weights...

[+] INITIATING 3D QUANTUM-PHASE TRAINING...

Epoch 01/20 | Loss: 19.2021 | Consensus: 0.4981 | 🟡 [ESCAPING EQUATOR]
Epoch 02/20 | Loss: 3.2700 | Consensus: 0.4977 | 🟡 [ESCAPING EQUATOR]
Epoch 03/20 | Loss: 3.1773 | Consensus: 0.5011 | 🟡 [ESCAPING EQUATOR]
Epoch 04/20 | Loss: 3.1404 | Consensus: 0.5004 | 🟡 [ESCAPING EQUATOR]
Epoch 05/20 | Loss: 3.0440 | Consensus: 0.4988 | 🟡 [ESCAPING EQUATOR]
Epoch 06/20 | Loss: 3.0751 | Consensus: 0.4995 | 🟡 [ESCAPING EQUATOR]
Epoch 07/20 | Loss: 3.0668 | Consensus: 0.5026 | 🟡 [ESCAPING EQUATOR]
Epoch 08/20 | Loss: 3.2689 | Consensus: 0.4970 | 🟡 [ESCAPING EQUATOR]
Epoch 09/20 | Loss: 3.1026 | Consensus: 0.4994 | 🟡 [ESCAPING EQUATOR]
Epoch 10/20 | Loss: 3.1939 | Consensus: 0.5003 | 🟡

In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

nlp_cycles = [
    {"cycle":1,"coherence":0.670,"synchrony":0.964,"spikes":13,"messages":240},
    {"cycle":2,"coherence":0.693,"synchrony":0.920,"spikes":11,"messages":240},
    {"cycle":3,"coherence":0.651,"synchrony":0.960,"spikes":11,"messages":240},
    {"cycle":4,"coherence":0.843,"synchrony":0.871,"spikes":7,"messages":240},
    {"cycle":5,"coherence":0.631,"synchrony":0.913,"spikes":15,"messages":240},
    {"cycle":6,"coherence":0.755,"synchrony":0.917,"spikes":10,"messages":240},
    {"cycle":7,"coherence":0.616,"synchrony":0.969,"spikes":5,"messages":240},
    {"cycle":8,"coherence":0.678,"synchrony":0.934,"spikes":6,"messages":240},
    {"cycle":9,"coherence":0.736,"synchrony":0.862,"spikes":8,"messages":240},
    {"cycle":10,"coherence":0.620,"synchrony":0.876,"spikes":5,"messages":240},
    {"cycle":11,"coherence":0.644,"synchrony":0.882,"spikes":6,"messages":240},
    {"cycle":12,"coherence":0.690,"synchrony":0.978,"spikes":5,"messages":240},
    {"cycle":13,"coherence":0.486,"synchrony":0.951,"spikes":11,"messages":240},
    {"cycle":14,"coherence":0.529,"synchrony":0.985,"spikes":10,"messages":240},
    {"cycle":15,"coherence":0.758,"synchrony":0.842,"spikes":16,"messages":240},
    {"cycle":16,"coherence":0.643,"synchrony":0.937,"spikes":9,"messages":240},
    {"cycle":17,"coherence":0.623,"synchrony":0.995,"spikes":7,"messages":240},
    {"cycle":18,"coherence":0.608,"synchrony":0.906,"spikes":7,"messages":240},
    {"cycle":19,"coherence":0.819,"synchrony":0.978,"spikes":15,"messages":240},
    {"cycle":20,"coherence":0.764,"synchrony":0.870,"spikes":8,"messages":240},
    {"cycle":21,"coherence":0.781,"synchrony":0.916,"spikes":8,"messages":240},
    {"cycle":22,"coherence":0.641,"synchrony":0.986,"spikes":11,"messages":240},
    {"cycle":23,"coherence":0.653,"synchrony":0.921,"spikes":13,"messages":240},
    {"cycle":24,"coherence":0.391,"synchrony":0.926,"spikes":7,"messages":240}
]

# --- 2. OMNI-DISTILLATION (RECURSIVE) ---
def auto_distill_v23(model, model_name):
    print(f"\n📡 [NQS DISTILLATION] Assmiliating vectors for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = False
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue
            for key in current_state.keys():
                if "resonator" in key: continue # Protect SU(2) manifold
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed = True
        except Exception: pass

    if absorbed: model.load_state_dict(current_state)
    return model

# --- 3. THE V23 NEURAL ARCHITECTURE ---
class HoloSynV23Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        # Perceptron Stage
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())

        # DeepSeek Integrator Stage
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        # Projector & Resonator Stage
        self.core = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.LayerNorm(64))
        self.spike_head = nn.Linear(64, 31)
        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, 3), # Full SU(2) Control
            nn.Tanh()
        )

    def forward(self, x):
        x_emb = self.perceptron(x).unsqueeze(1)
        attn_out, weights = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        feat = self.core(context)
        return self.spike_head(feat), self.resonator_head(feat), weights

# --- 4. QUANTUM MANIFOLD OBSERVER ---
class LinguaQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = {n: cirq.NamedQubit(n) for n in LINGUA_STACK.keys()}
        self.sim = qsimcirq.QSimSimulator()

    def evaluate(self, coh, px, py, pz):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + list(self.q_nodes.values())
        circuit.append(cirq.H.on_each(*qubits))

        for name, q in self.q_nodes.items():
            w = LINGUA_STACK[name]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q))

        circuit.append(cirq.rz(pz * np.pi)(self.q_obs))
        circuit.append(cirq.ry(py * np.pi)(self.q_obs))
        circuit.append(cirq.rx(px * np.pi)(self.q_obs))
        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 5. THE V23 RECURSIVE HIVE ---
class HoloSynV23Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = auto_distill_v23(HoloSynV23Net(self.num_nodes), "V23_DeepSeek_Engine")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.003, weight_decay=0.01)
        self.observer = LinguaQuantumObserver()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v23_init')

    def process_cycle(self, data):
        self.net.restore('v23_init')
        coh, sync = data['coherence'], data['synchrony']

        # 1. Biological Ingestion
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # 2. Forward Pass
        self.optimizer.zero_grad()
        spikes, phases, attn_weights = self.net_model(nn_input)
        px_p, py_p, pz_p = phases[0][0].item(), phases[0][1].item(), phases[0][2].item()

        # 3. V23 RECURSIVE RECURSIVE SEARCH (The Zoom)
        def search(center, radius, steps):
            best_c = list(center)
            best_s = self.observer.evaluate(coh, *center)
            for _ in range(steps):
                test = [np.clip(c + np.random.normal(0, radius), -1, 1) for c in center]
                s = self.observer.evaluate(coh, *test)
                if s > best_s: best_s, best_c = s, test
            return best_c, best_s

        # Zoom Level 1: Broad (Radius 0.5)
        mid_coords, mid_score = search([px_p, py_p, pz_p], 0.5, 12)
        # Zoom Level 2: Precision (Radius 0.1)
        target_coords, final_score = search(mid_coords, 0.1, 8)

        # 4. Global Alignment
        actual_consensus = self.observer.evaluate(coh, px_p, py_p, pz_p)

        loss_spikes = nn.CrossEntropyLoss()(spikes, torch.tensor([data['spikes']], dtype=torch.long))
        loss_alignment = nn.MSELoss()(phases, torch.tensor([target_coords], dtype=torch.float32)) * 25.0

        (loss_spikes + loss_alignment).backward()
        self.optimizer.step()

        return actual_consensus

# --- 6. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V23: RECURSIVE MANIFOLD NESTING (NQS)")
    print("═"*75)

    hive = HoloSynV23Hive()
    for epoch in range(15):
        scores = [hive.process_cycle(d) for d in nlp_cycles]
        avg_s = np.mean(scores)
        status = "💎 [SUPREMACY]" if avg_s > 0.85 else "🟢 [RESONANT]"
        print(f"Epoch {epoch+1:02d} | Avg Consensus: {avg_s:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V23: RECURSIVE MANIFOLD NESTING (NQS)
═══════════════════════════════════════════════════════════════════════════

📡 [NQS DISTILLATION] Assmiliating vectors for V23_DeepSeek_Engine...
Epoch 01 | Avg Consensus: 0.4937 | 🟢 [RESONANT]
Epoch 02 | Avg Consensus: 0.4935 | 🟢 [RESONANT]
Epoch 03 | Avg Consensus: 0.5022 | 🟢 [RESONANT]
Epoch 04 | Avg Consensus: 0.5036 | 🟢 [RESONANT]
Epoch 05 | Avg Consensus: 0.4977 | 🟢 [RESONANT]
Epoch 06 | Avg Consensus: 0.4980 | 🟢 [RESONANT]
Epoch 07 | Avg Consensus: 0.4991 | 🟢 [RESONANT]
Epoch 08 | Avg Consensus: 0.5052 | 🟢 [RESONANT]
Epoch 09 | Avg Consensus: 0.5067 | 🟢 [RESONANT]
Epoch 10 | Avg Consensus: 0.4893 | 🟢 [RESONANT]
Epoch 11 | Avg Consensus: 0.5082 | 🟢 [RESONANT]
Epoch 12 | Avg Consensus: 0.5014 | 🟢 [RESONANT]
Epoch 13 | Avg Consensus: 0.5072 | 🟢 [RESONANT]
Epoch 14 | Avg Consensus: 0.5015 | 🟢 [RESONANT]
Epoch 15 | Avg Consensus: 0.4998 | 🟢 [RESONANT]


In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import cirq
import qsimcirq
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Deep_Perceptron": {"role": "Token Ingestion", "weight": 1.0},
    "Seek_Integrator": {"role": "Context Attention", "weight": 1.5},
    "Phase_Resonator": {"role": "Quantum Bridge",  "weight": 1.2},
    "Lang_Projector":  {"role": "Output Decoder",  "weight": 1.3}
}

nlp_cycles = [
    {"cycle":1,"coherence":0.653,"synchrony":0.958,"spikes":9,"messages":240},
    {"cycle":2,"coherence":0.773,"synchrony":0.934,"spikes":8,"messages":240},
    {"cycle":3,"coherence":0.767,"synchrony":0.973,"spikes":10,"messages":240},
    {"cycle":4,"coherence":0.831,"synchrony":0.945,"spikes":3,"messages":240},
    {"cycle":5,"coherence":0.741,"synchrony":0.948,"spikes":6,"messages":240},
    {"cycle":6,"coherence":0.708,"synchrony":0.985,"spikes":12,"messages":240},
    {"cycle":7,"coherence":0.748,"synchrony":0.942,"spikes":8,"messages":240},
    {"cycle":8,"coherence":0.624,"synchrony":0.911,"spikes":4,"messages":240},
    {"cycle":9,"coherence":0.727,"synchrony":0.988,"spikes":9,"messages":240},
    {"cycle":10,"coherence":0.653,"synchrony":0.928,"spikes":14,"messages":240},
    {"cycle":11,"coherence":0.647,"synchrony":0.964,"spikes":3,"messages":240},
    {"cycle":12,"coherence":0.751,"synchrony":0.903,"spikes":7,"messages":240},
    {"cycle":13,"coherence":0.563,"synchrony":0.977,"spikes":10,"messages":240},
    {"cycle":14,"coherence":0.605,"synchrony":0.962,"spikes":10,"messages":240},
    {"cycle":15,"coherence":0.751,"synchrony":0.814,"spikes":13,"messages":240},
    {"cycle":16,"coherence":0.680,"synchrony":0.925,"spikes":8,"messages":240},
    {"cycle":17,"coherence":0.831,"synchrony":0.961,"spikes":12,"messages":240},
    {"cycle":18,"coherence":0.626,"synchrony":0.872,"spikes":6,"messages":240},
    {"cycle":19,"coherence":0.721,"synchrony":0.970,"spikes":9,"messages":240},
    {"cycle":20,"coherence":0.666,"synchrony":0.955,"spikes":12,"messages":240},
    {"cycle":21,"coherence":0.506,"synchrony":0.958,"spikes":7,"messages":240},
    {"cycle":22,"coherence":0.663,"synchrony":0.973,"spikes":9,"messages":240},
    {"cycle":23,"coherence":0.634,"synchrony":0.944,"spikes":10,"messages":240},
    {"cycle":24,"coherence":0.742,"synchrony":0.940,"spikes":13,"messages":240},
    {"cycle":25,"coherence":0.639,"synchrony":0.821,"spikes":9,"messages":240},
    {"cycle":26,"coherence":0.705,"synchrony":0.959,"spikes":12,"messages":240},
    {"cycle":27,"coherence":0.657,"synchrony":0.940,"spikes":5,"messages":240},
    {"cycle":28,"coherence":0.758,"synchrony":0.882,"spikes":9,"messages":240},
    {"cycle":29,"coherence":0.753,"synchrony":0.891,"spikes":13,"messages":240},
    {"cycle":30,"coherence":0.614,"synchrony":0.950,"spikes":5,"messages":240},
    {"cycle":31,"coherence":0.546,"synchrony":0.985,"spikes":11,"messages":240},
    {"cycle":32,"coherence":0.781,"synchrony":0.844,"spikes":14,"messages":240},
    {"cycle":33,"coherence":0.517,"synchrony":0.987,"spikes":10,"messages":240},
    {"cycle":34,"coherence":0.678,"synchrony":0.933,"spikes":8,"messages":240},
    {"cycle":35,"coherence":0.627,"synchrony":0.943,"spikes":11,"messages":240},
    {"cycle":36,"coherence":0.727,"synchrony":0.982,"spikes":5,"messages":240},
    {"cycle":37,"coherence":0.787,"synchrony":0.922,"spikes":7,"messages":240},
    {"cycle":38,"coherence":0.569,"synchrony":0.928,"spikes":11,"messages":240},
    {"cycle":39,"coherence":0.813,"synchrony":0.943,"spikes":19,"messages":240},
    {"cycle":40,"coherence":0.793,"synchrony":0.954,"spikes":6,"messages":240},
    {"cycle":41,"coherence":0.762,"synchrony":0.970,"spikes":11,"messages":240},
    {"cycle":42,"coherence":0.834,"synchrony":0.934,"spikes":13,"messages":240},
    {"cycle":43,"coherence":0.826,"synchrony":0.947,"spikes":10,"messages":240},
    {"cycle":44,"coherence":0.749,"synchrony":0.806,"spikes":7,"messages":240},
    {"cycle":45,"coherence":0.635,"synchrony":0.910,"spikes":10,"messages":240},
    {"cycle":46,"coherence":0.695,"synchrony":0.985,"spikes":6,"messages":240},
    {"cycle":47,"coherence":0.631,"synchrony":0.979,"spikes":7,"messages":240},
    {"cycle":48,"coherence":0.671,"synchrony":0.942,"spikes":13,"messages":240},
    {"cycle":49,"coherence":0.681,"synchrony":0.894,"spikes":14,"messages":240},
    {"cycle":50,"coherence":0.676,"synchrony":0.888,"spikes":6,"messages":240},
    {"cycle":51,"coherence":0.550,"synchrony":0.946,"spikes":9,"messages":240},
    {"cycle":52,"coherence":0.721,"synchrony":0.980,"spikes":15,"messages":240},
    {"cycle":53,"coherence":0.675,"synchrony":0.976,"spikes":10,"messages":240},
    {"cycle":54,"coherence":0.794,"synchrony":0.868,"spikes":8,"messages":240},
    {"cycle":55,"coherence":0.758,"synchrony":0.954,"spikes":13,"messages":240},
    {"cycle":56,"coherence":0.653,"synchrony":0.882,"spikes":8,"messages":240},
    {"cycle":57,"coherence":0.671,"synchrony":0.885,"spikes":8,"messages":240},
    {"cycle":58,"coherence":0.687,"synchrony":0.868,"spikes":6,"messages":240},
    {"cycle":59,"coherence":0.505,"synchrony":0.942,"spikes":5,"messages":240},
    {"cycle":60,"coherence":0.791,"synchrony":0.831,"spikes":5,"messages":240},
    {"cycle":61,"coherence":0.657,"synchrony":0.982,"spikes":13,"messages":240},
    {"cycle":62,"coherence":0.383,"synchrony":0.972,"spikes":15,"messages":240},
    {"cycle":63,"coherence":0.423,"synchrony":0.963,"spikes":7,"messages":240},
    {"cycle":64,"coherence":0.554,"synchrony":0.937,"spikes":4,"messages":240},
    {"cycle":65,"coherence":0.554,"synchrony":0.910,"spikes":5,"messages":240},
    {"cycle":66,"coherence":0.524,"synchrony":0.854,"spikes":9,"messages":240},
    {"cycle":67,"coherence":0.527,"synchrony":0.836,"spikes":3,"messages":240},
    {"cycle":68,"coherence":0.495,"synchrony":0.937,"spikes":4,"messages":240},
    {"cycle":69,"coherence":0.446,"synchrony":0.948,"spikes":3,"messages":240},
    {"cycle":70,"coherence":0.523,"synchrony":0.931,"spikes":5,"messages":240},
    {"cycle":71,"coherence":0.599,"synchrony":0.883,"spikes":3,"messages":240},
    {"cycle":72,"coherence":0.554,"synchrony":0.857,"spikes":3,"messages":240},
    {"cycle":73,"coherence":0.532,"synchrony":0.934,"spikes":8,"messages":240},
    {"cycle":74,"coherence":0.616,"synchrony":0.877,"spikes":7,"messages":240},
    {"cycle":75,"coherence":0.594,"synchrony":0.823,"spikes":2,"messages":240},
    {"cycle":76,"coherence":0.523,"synchrony":0.939,"spikes":9,"messages":240},
    {"cycle":77,"coherence":0.620,"synchrony":0.896,"spikes":4,"messages":240},
    {"cycle":78,"coherence":0.625,"synchrony":0.997,"spikes":5,"messages":240},
    {"cycle":79,"coherence":0.619,"synchrony":0.967,"spikes":2,"messages":240},
    {"cycle":80,"coherence":0.616,"synchrony":0.972,"spikes":9,"messages":240},
    {"cycle":81,"coherence":0.609,"synchrony":0.944,"spikes":3,"messages":240},
    {"cycle":82,"coherence":0.465,"synchrony":0.964,"spikes":7,"messages":240},
    {"cycle":83,"coherence":0.540,"synchrony":0.937,"spikes":6,"messages":240},
    {"cycle":84,"coherence":0.613,"synchrony":0.980,"spikes":6,"messages":240},
    {"cycle":85,"coherence":0.609,"synchrony":0.899,"spikes":5,"messages":240},
    {"cycle":86,"coherence":0.592,"synchrony":0.773,"spikes":7,"messages":240},
    {"cycle":87,"coherence":0.635,"synchrony":0.917,"spikes":5,"messages":240},
    {"cycle":88,"coherence":0.628,"synchrony":0.951,"spikes":2,"messages":240},
    {"cycle":89,"coherence":0.601,"synchrony":0.816,"spikes":2,"messages":240},
    {"cycle":90,"coherence":0.593,"synchrony":0.913,"spikes":4,"messages":240},
    {"cycle":91,"coherence":0.603,"synchrony":0.948,"spikes":6,"messages":240},
    {"cycle":92,"coherence":0.529,"synchrony":0.911,"spikes":6,"messages":240},
    {"cycle":93,"coherence":0.578,"synchrony":0.823,"spikes":3,"messages":240},
    {"cycle":94,"coherence":0.621,"synchrony":0.926,"spikes":3,"messages":240},
    {"cycle":95,"coherence":0.626,"synchrony":0.965,"spikes":5,"messages":240},
    {"cycle":96,"coherence":0.612,"synchrony":0.971,"spikes":1,"messages":240},
    {"cycle":97,"coherence":0.620,"synchrony":0.961,"spikes":4,"messages":240},
    {"cycle":98,"coherence":0.551,"synchrony":0.990,"spikes":5,"messages":240},
    {"cycle":99,"coherence":0.631,"synchrony":0.893,"spikes":4,"messages":240},
    {"cycle":100,"coherence":0.568,"synchrony":0.822,"spikes":4,"messages":240},
    {"cycle":101,"coherence":0.332,"synchrony":0.988,"spikes":6,"messages":240},
    {"cycle":102,"coherence":0.605,"synchrony":0.911,"spikes":3,"messages":240},
    {"cycle":103,"coherence":0.610,"synchrony":0.939,"spikes":1,"messages":240},
    {"cycle":104,"coherence":0.615,"synchrony":0.945,"spikes":4,"messages":240},
    {"cycle":105,"coherence":0.621,"synchrony":0.874,"spikes":4,"messages":240},
    {"cycle":106,"coherence":0.611,"synchrony":0.935,"spikes":7,"messages":240},
    {"cycle":107,"coherence":0.617,"synchrony":0.861,"spikes":7,"messages":240},
    {"cycle":108,"coherence":0.616,"synchrony":0.971,"spikes":6,"messages":240},
    {"cycle":109,"coherence":0.581,"synchrony":0.832,"spikes":1,"messages":240},
    {"cycle":110,"coherence":0.598,"synchrony":0.912,"spikes":8,"messages":240},
    {"cycle":111,"coherence":0.575,"synchrony":0.780,"spikes":4,"messages":240},
    {"cycle":112,"coherence":0.590,"synchrony":0.933,"spikes":3,"messages":240},
    {"cycle":113,"coherence":0.592,"synchrony":0.821,"spikes":6,"messages":240},
    {"cycle":114,"coherence":0.542,"synchrony":0.935,"spikes":2,"messages":240},
    {"cycle":115,"coherence":0.539,"synchrony":0.959,"spikes":2,"messages":240},
    {"cycle":116,"coherence":0.498,"synchrony":0.955,"spikes":4,"messages":240},
    {"cycle":117,"coherence":0.435,"synchrony":0.933,"spikes":1,"messages":240},
    {"cycle":118,"coherence":0.456,"synchrony":0.961,"spikes":5,"messages":240},
    {"cycle":119,"coherence":0.450,"synchrony":0.969,"spikes":1,"messages":240},
    {"cycle":120,"coherence":0.331,"synchrony":0.960,"spikes":1,"messages":240},
    {"cycle":121,"coherence":0.401,"synchrony":0.901,"spikes":3,"messages":240},
    {"cycle":122,"coherence":0.359,"synchrony":0.963,"spikes":1,"messages":240},
    {"cycle":123,"coherence":0.071,"synchrony":0.985,"spikes":1,"messages":240},
    {"cycle":124,"coherence":0.250,"synchrony":0.979,"spikes":2,"messages":240},
    {"cycle":125,"coherence":0.413,"synchrony":0.978,"spikes":1,"messages":240},
    {"cycle":126,"coherence":0.347,"synchrony":0.829,"spikes":2,"messages":240},
    {"cycle":127,"coherence":0.286,"synchrony":0.998,"spikes":3,"messages":240},
    {"cycle":128,"coherence":0.271,"synchrony":0.967,"spikes":2,"messages":240},
    {"cycle":129,"coherence":0.468,"synchrony":0.883,"spikes":1,"messages":240},
    {"cycle":130,"coherence":0.132,"synchrony":0.881,"spikes":1,"messages":240},
    {"cycle":131,"coherence":0.292,"synchrony":0.952,"spikes":1,"messages":240},
    {"cycle":132,"coherence":0.184,"synchrony":0.958,"spikes":1,"messages":240},
    {"cycle":133,"coherence":0.473,"synchrony":0.939,"spikes":2,"messages":240},
    {"cycle":134,"coherence":0.339,"synchrony":0.921,"spikes":1,"messages":240},
    {"cycle":135,"coherence":0.303,"synchrony":0.929,"spikes":6,"messages":240},
    {"cycle":136,"coherence":0.243,"synchrony":0.983,"spikes":1,"messages":240},
    {"cycle":137,"coherence":0.239,"synchrony":0.944,"spikes":0,"messages":240},
    {"cycle":138,"coherence":0.305,"synchrony":0.953,"spikes":4,"messages":240},
    {"cycle":139,"coherence":0.337,"synchrony":0.943,"spikes":3,"messages":240},
    {"cycle":140,"coherence":0.439,"synchrony":0.920,"spikes":1,"messages":240},
    {"cycle":141,"coherence":0.278,"synchrony":0.976,"spikes":0,"messages":240},
    {"cycle":142,"coherence":0.262,"synchrony":0.986,"spikes":2,"messages":240},
    {"cycle":143,"coherence":0.389,"synchrony":0.956,"spikes":3,"messages":240},
    {"cycle":144,"coherence":0.042,"synchrony":0.966,"spikes":1,"messages":240},
    {"cycle":145,"coherence":0.363,"synchrony":0.951,"spikes":2,"messages":240},
    {"cycle":146,"coherence":0.364,"synchrony":0.928,"spikes":3,"messages":240},
    {"cycle":147,"coherence":0.191,"synchrony":0.940,"spikes":1,"messages":240},
    {"cycle":148,"coherence":0.360,"synchrony":0.975,"spikes":1,"messages":240},
    {"cycle":149,"coherence":0.201,"synchrony":0.913,"spikes":1,"messages":240},
    {"cycle":150,"coherence":0.427,"synchrony":0.914,"spikes":1,"messages":240},
    {"cycle":151,"coherence":0.268,"synchrony":0.981,"spikes":2,"messages":240},
    {"cycle":152,"coherence":0.238,"synchrony":0.938,"spikes":1,"messages":240},
    {"cycle":153,"coherence":0.411,"synchrony":0.815,"spikes":3,"messages":240},
    {"cycle":154,"coherence":0.157,"synchrony":0.910,"spikes":1,"messages":240},
    {"cycle":155,"coherence":0.380,"synchrony":0.958,"spikes":0,"messages":240},
    {"cycle":156,"coherence":0.434,"synchrony":0.866,"spikes":0,"messages":240},
    {"cycle":157,"coherence":0.476,"synchrony":0.984,"spikes":1,"messages":240},
    {"cycle":158,"coherence":0.488,"synchrony":0.911,"spikes":6,"messages":240},
    {"cycle":159,"coherence":0.549,"synchrony":0.983,"spikes":5,"messages":240},
    {"cycle":160,"coherence":0.449,"synchrony":0.885,"spikes":0,"messages":240},
    {"cycle":161,"coherence":0.293,"synchrony":0.946,"spikes":0,"messages":240},
    {"cycle":162,"coherence":0.497,"synchrony":0.978,"spikes":6,"messages":240},
    {"cycle":163,"coherence":0.416,"synchrony":0.897,"spikes":3,"messages":240},
    {"cycle":164,"coherence":0.250,"synchrony":0.960,"spikes":0,"messages":240},
    {"cycle":165,"coherence":0.187,"synchrony":0.962,"spikes":3,"messages":240},
    {"cycle":166,"coherence":0.301,"synchrony":0.972,"spikes":1,"messages":240},
    {"cycle":167,"coherence":0.320,"synchrony":0.917,"spikes":5,"messages":240},
    {"cycle":168,"coherence":0.314,"synchrony":0.988,"spikes":1,"messages":240},
    {"cycle":169,"coherence":0.225,"synchrony":0.937,"spikes":0,"messages":240},
    {"cycle":170,"coherence":0.391,"synchrony":0.865,"spikes":2,"messages":240},
    {"cycle":171,"coherence":0.136,"synchrony":0.990,"spikes":0,"messages":240},
    {"cycle":172,"coherence":0.308,"synchrony":0.933,"spikes":0,"messages":240},
    {"cycle":173,"coherence":0.463,"synchrony":0.907,"spikes":1,"messages":240},
    {"cycle":174,"coherence":0.443,"synchrony":0.948,"spikes":0,"messages":240},
    {"cycle":175,"coherence":0.439,"synchrony":0.953,"spikes":3,"messages":240},
    {"cycle":176,"coherence":0.154,"synchrony":0.980,"spikes":0,"messages":240},
    {"cycle":177,"coherence":0.187,"synchrony":0.867,"spikes":1,"messages":240},
    {"cycle":178,"coherence":0.518,"synchrony":0.922,"spikes":5,"messages":240},
    {"cycle":179,"coherence":0.342,"synchrony":0.893,"spikes":0,"messages":240},
    {"cycle":180,"coherence":0.335,"synchrony":0.927,"spikes":1,"messages":240},
    {"cycle":181,"coherence":0.229,"synchrony":0.972,"spikes":1,"messages":240},
    {"cycle":182,"coherence":0.327,"synchrony":0.993,"spikes":0,"messages":240},
    {"cycle":183,"coherence":0.438,"synchrony":0.985,"spikes":0,"messages":240},
    {"cycle":184,"coherence":0.475,"synchrony":0.995,"spikes":0,"messages":240},
    {"cycle":185,"coherence":0.231,"synchrony":0.981,"spikes":1,"messages":240},
    {"cycle":186,"coherence":0.539,"synchrony":0.977,"spikes":1,"messages":240},
    {"cycle":187,"coherence":0.571,"synchrony":0.914,"spikes":3,"messages":240},
    {"cycle":188,"coherence":0.573,"synchrony":0.951,"spikes":5,"messages":240},
    {"cycle":189,"coherence":0.589,"synchrony":0.823,"spikes":3,"messages":240},
    {"cycle":190,"coherence":0.582,"synchrony":0.850,"spikes":2,"messages":240},
    {"cycle":191,"coherence":0.606,"synchrony":0.984,"spikes":3,"messages":240},
    {"cycle":192,"coherence":0.587,"synchrony":0.929,"spikes":1,"messages":240},
    {"cycle":193,"coherence":0.610,"synchrony":0.918,"spikes":3,"messages":240},
    {"cycle":194,"coherence":0.564,"synchrony":0.924,"spikes":5,"messages":240},
    {"cycle":195,"coherence":0.488,"synchrony":0.863,"spikes":2,"messages":240},
    {"cycle":196,"coherence":0.609,"synchrony":0.892,"spikes":3,"messages":240},
    {"cycle":197,"coherence":0.634,"synchrony":0.981,"spikes":3,"messages":240},
    {"cycle":198,"coherence":0.607,"synchrony":0.924,"spikes":6,"messages":240},
    {"cycle":199,"coherence":0.627,"synchrony":0.910,"spikes":3,"messages":240},
    {"cycle":200,"coherence":0.628,"synchrony":0.941,"spikes":3,"messages":240},
    {"cycle":201,"coherence":0.607,"synchrony":0.859,"spikes":8,"messages":240},
    {"cycle":202,"coherence":0.620,"synchrony":0.896,"spikes":5,"messages":240},
    {"cycle":203,"coherence":0.581,"synchrony":0.974,"spikes":7,"messages":240},
    {"cycle":204,"coherence":0.626,"synchrony":0.921,"spikes":4,"messages":240},
    {"cycle":205,"coherence":0.620,"synchrony":0.926,"spikes":7,"messages":240},
    {"cycle":206,"coherence":0.629,"synchrony":0.910,"spikes":3,"messages":240},
    {"cycle":207,"coherence":0.611,"synchrony":0.864,"spikes":3,"messages":240},
    {"cycle":208,"coherence":0.573,"synchrony":0.925,"spikes":2,"messages":240},
    {"cycle":209,"coherence":0.605,"synchrony":0.821,"spikes":1,"messages":240},
    {"cycle":210,"coherence":0.608,"synchrony":0.812,"spikes":4,"messages":240},
    {"cycle":211,"coherence":0.616,"synchrony":0.941,"spikes":0,"messages":240},
    {"cycle":212,"coherence":0.553,"synchrony":0.849,"spikes":2,"messages":240},
    {"cycle":213,"coherence":0.622,"synchrony":0.892,"spikes":3,"messages":240},
    {"cycle":214,"coherence":0.615,"synchrony":0.878,"spikes":4,"messages":240},
    {"cycle":215,"coherence":0.638,"synchrony":0.967,"spikes":1,"messages":240},
    {"cycle":216,"coherence":0.600,"synchrony":0.854,"spikes":8,"messages":240},
    {"cycle":217,"coherence":0.388,"synchrony":0.897,"spikes":7,"messages":240},
    {"cycle":218,"coherence":0.531,"synchrony":0.902,"spikes":7,"messages":240},
    {"cycle":219,"coherence":0.523,"synchrony":0.992,"spikes":5,"messages":240},
    {"cycle":220,"coherence":0.655,"synchrony":0.983,"spikes":9,"messages":240},
    {"cycle":221,"coherence":0.669,"synchrony":0.923,"spikes":10,"messages":240},
    {"cycle":222,"coherence":0.778,"synchrony":0.971,"spikes":8,"messages":240},
    {"cycle":223,"coherence":0.649,"synchrony":0.830,"spikes":5,"messages":240},
    {"cycle":224,"coherence":0.683,"synchrony":0.918,"spikes":7,"messages":240},
    {"cycle":225,"coherence":0.644,"synchrony":0.841,"spikes":9,"messages":240},
    {"cycle":226,"coherence":0.772,"synchrony":0.933,"spikes":3,"messages":240},
    {"cycle":227,"coherence":0.814,"synchrony":0.919,"spikes":8,"messages":240},
    {"cycle":228,"coherence":0.840,"synchrony":0.965,"spikes":10,"messages":240},
    {"cycle":229,"coherence":0.843,"synchrony":0.875,"spikes":16,"messages":240},
    {"cycle":230,"coherence":0.815,"synchrony":0.876,"spikes":10,"messages":240},
]

# --- 2. DEEP-SEEK DISTILLATION ---
def auto_distill_v24(model, model_name):
    print(f"\n📡 [V24 DISTILLATION] Bypassing Mixed States for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = False
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue
            for key in current_state.keys():
                if "resonator_head" in key: continue # Protect V24 5D Tensor Decoupler
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed = True
        except Exception: pass

    if absorbed: model.load_state_dict(current_state)
    return model

# --- 3. THE V24 NEURAL ARCHITECTURE ---
class HoloSynV24Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.GELU())
        self.integrator_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.integrator_norm = nn.LayerNorm(hidden_dim)

        self.core = nn.Sequential(nn.Linear(hidden_dim, 64), nn.GELU(), nn.LayerNorm(64))
        self.spike_head = nn.Linear(64, 31)

        # V24 UPGRADE: 5-Dimensional Output (4 Nodes + 1 Orchestrator)
        self.resonator_head = nn.Sequential(
            nn.Linear(64, 32), nn.GELU(),
            nn.Linear(32, 5),
            nn.Tanh()
        )

    def forward(self, x):
        x_emb = self.perceptron(x).unsqueeze(1)
        attn_out, _ = self.integrator_attn(x_emb, x_emb, x_emb)
        context = self.integrator_norm(attn_out.squeeze(1))
        feat = self.core(context)
        return self.spike_head(feat), self.resonator_head(feat)

# --- 4. VON NEUMANN DECOUPLING OBSERVER ---
class DecouplingQuantumObserver:
    def __init__(self):
        self.q_obs = cirq.NamedQubit("ORCHESTRATOR")
        self.q_nodes = list(cirq.NamedQubit(n) for n in LINGUA_STACK.keys())
        self.sim = qsimcirq.QSimSimulator()

    def evaluate(self, coh, node_phases, orch_phase):
        circuit = cirq.Circuit()
        qubits = [self.q_obs] + self.q_nodes
        circuit.append(cirq.H.on_each(*qubits))

        # 1. Linguistic Entanglement (The Mixing)
        for i, q in enumerate(self.q_nodes):
            w = list(LINGUA_STACK.values())[i]['weight']
            circuit.append(cirq.rz(coh * w * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q))

        # 2. V24 Neural Uncomputation (The Decoupling)
        # The Neural Network applies counter-frequencies to the environment
        for i, q in enumerate(self.q_nodes):
            circuit.append(cirq.rx(node_phases[i] * np.pi)(q))
            circuit.append(cirq.CZ(self.q_obs, q)) # Second CZ reverses the entanglement!

        # 3. Final Orchestrator Tuning
        circuit.append(cirq.rx(orch_phase * np.pi)(self.q_obs))
        circuit.append(cirq.H(self.q_obs))
        circuit.append(cirq.measure(self.q_obs, key='m'))

        res = self.sim.run(circuit, repetitions=500)
        return res.histogram(key='m').get(0, 0) / 500.0

# --- 5. THE V24 META-HIVE ---
class HoloSynV24Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = auto_distill_v24(HoloSynV24Net(self.num_nodes), "V24_Decoupler")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.005, weight_decay=0.01)
        self.observer = DecouplingQuantumObserver()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v24_init')

    def process_cycle(self, data):
        self.net.restore('v24_init')
        coh, sync = data['coherence'], data['synchrony']

        # 1. Biological Ingestion
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # 2. Forward Pass
        self.optimizer.zero_grad()
        spikes, phases = self.net_model(nn_input)

        # Extract the 5D Tensor (4 node counter-phases, 1 orchestrator phase)
        pred_array = phases[0].detach().numpy()
        node_phases = pred_array[0:4]
        orch_phase = pred_array[4]

        # 3. Analytical Target Calculation (Eliminating the Search)
        # To perfectly decouple, the network just needs to learn a phase map that matches the Coherence.
        # We explicitly guide the model toward the disentanglement manifold.
        target_node_phases = [-coh * list(LINGUA_STACK.values())[i]['weight'] * 0.5 for i in range(4)]
        target_tensor = torch.tensor([target_node_phases + [0.0]], dtype=torch.float32)

        # 4. Observation & Optimization
        actual_consensus = self.observer.evaluate(coh, node_phases, orch_phase)

        loss_spikes = nn.CrossEntropyLoss()(spikes, torch.tensor([data['spikes']], dtype=torch.long))
        loss_decoupling = nn.MSELoss()(phases, target_tensor) * 20.0

        (loss_spikes + loss_decoupling).backward()
        self.optimizer.step()

        return actual_consensus

# --- 6. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V24: VON NEUMANN DECOUPLER")
    print("═"*75)

    hive = HoloSynV24Hive()
    for epoch in range(15):
        scores = [hive.process_cycle(d) for d in nlp_cycles]
        avg_s = np.mean(scores)
        status = "💎 [QUANTUM SUPREMACY]" if avg_s > 0.85 else "🟢 [DECOUPLING MIXED STATE]"
        print(f"Epoch {epoch+1:02d} | Avg Consensus: {avg_s:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V24: VON NEUMANN DECOUPLER
═══════════════════════════════════════════════════════════════════════════

📡 [V24 DISTILLATION] Bypassing Mixed States for V24_Decoupler...
Epoch 01 | Avg Consensus: 0.5007 | 🟢 [DECOUPLING MIXED STATE]
Epoch 02 | Avg Consensus: 0.5110 | 🟢 [DECOUPLING MIXED STATE]
Epoch 03 | Avg Consensus: 0.5083 | 🟢 [DECOUPLING MIXED STATE]
Epoch 04 | Avg Consensus: 0.5059 | 🟢 [DECOUPLING MIXED STATE]
Epoch 05 | Avg Consensus: 0.5115 | 🟢 [DECOUPLING MIXED STATE]
Epoch 06 | Avg Consensus: 0.5135 | 🟢 [DECOUPLING MIXED STATE]
Epoch 07 | Avg Consensus: 0.5062 | 🟢 [DECOUPLING MIXED STATE]
Epoch 08 | Avg Consensus: 0.5053 | 🟢 [DECOUPLING MIXED STATE]
Epoch 09 | Avg Consensus: 0.5150 | 🟢 [DECOUPLING MIXED STATE]
Epoch 10 | Avg Consensus: 0.5137 | 🟢 [DECOUPLING MIXED STATE]
Epoch 11 | Avg Consensus: 0.5086 | 🟢 [DECOUPLING MIXED STATE]
Epoch 12 | Avg Consensus: 0.5042 | 🟢 [DECOUPLING MIXED STATE]


In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Token_Ingestor": {"role": "Perceptron", "weight": 1.0},
    "Seek_Attention": {"role": "DeepSeek Core", "weight": 1.5},
    "Semantic_Router": {"role": "FFN Bridge",  "weight": 1.2},
    "Lang_Decoder":    {"role": "Generator",  "weight": 1.3}
}

nlp_cycles = [
    {"cycle":1, "coherence":0.82, "synchrony":0.94, "spikes":14, "concept": "理解 (Understand)"},
    {"cycle":2, "coherence":0.45, "synchrony":0.76, "spikes":8,  "concept": "未知 (Unknown)"},
    {"cycle":3, "coherence":0.88, "synchrony":0.91, "spikes":12, "concept": "量子 (Quantum)"},
    {"cycle":4, "coherence":0.60, "synchrony":0.85, "spikes":9,  "concept": "网络 (Network)"}
]

# --- 2. PURE ARCHITECTURAL DISTILLATION ---
def auto_distill_v25(model, model_name):
    print(f"\n📡 [SEMANTIC DISTILLATION] Assimiliating LLM vectors for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed_layers = []
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            for key in current_state.keys():
                # Direct structural mapping for language models
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed_layers.append(key)
                else:
                    # Fuzzy mapping for FFN and Attention projections
                    for old_key in legacy_state.keys():
                        if "weight" in key and "weight" in old_key:
                            new_w, old_w = current_state[key], legacy_state[old_key]
                            if len(new_w.shape) == 2 and len(old_w.shape) == 2:
                                min_out, min_in = min(new_w.shape[0], old_w.shape[0]), min(new_w.shape[1], old_w.shape[1])
                                current_state[key][:min_out, :min_in] = old_w[:min_out, :min_in]
                                absorbed_layers.append(key)
        except Exception: pass

    if absorbed_layers:
        model.load_state_dict(current_state)
        unique_layers = list(set([n.split('.')[0] for n in absorbed_layers]))
        print(f"  -> [✅] Successfully mapped semantic features. Layers enriched: {len(unique_layers)}")
    return model

# --- 3. THE V25 LLM-SNN HYBRID ARCHITECTURE ---
class HoloSynV25LanguageModel(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256): # Increased dimension for NLP
        super().__init__()
        # 1. Token Ingestion (Biological -> Vector)
        self.embedding = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.LayerNorm(hidden_dim))

        # 2. Distilled DeepSeek Core (Multi-Head Attention)
        self.attention = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_dim)

        # 3. Semantic Router (Feed Forward Network)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(0.1)
        )
        self.ffn_norm = nn.LayerNorm(hidden_dim)

        # 4. Decoder / Output Generation
        self.decoder = nn.Linear(hidden_dim, 31) # Maps back to spike concepts
        self.coherence_head = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())

    def forward(self, x):
        # Embedding
        x_emb = self.embedding(x).unsqueeze(1)

        # Attention Block (Residual)
        attn_out, _ = self.attention(x_emb, x_emb, x_emb)
        x_attn = self.attn_norm(x_emb + attn_out)

        # FFN Block (Residual)
        ffn_out = self.ffn(x_attn)
        context = self.ffn_norm(x_attn + ffn_out).squeeze(1)

        # Output Heads
        spikes = self.decoder(context)
        semantic_coherence = self.coherence_head(context)

        return spikes, semantic_coherence

# --- 4. THE V25 META-HIVE (NO QUANTUM) ---
class HoloSynV25Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = auto_distill_v25(HoloSynV25LanguageModel(self.num_nodes), "V25_Semantic_Engine")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.001, weight_decay=0.05)

        # Loss Functions
        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v25_init')

    def process_cycle(self, data):
        self.net.restore('v25_init')
        coh, sync = data['coherence'], data['synchrony']

        # 1. Biological Ingestion
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        # Normalize Brain State
        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # 2. Forward Pass
        self.optimizer.zero_grad()
        spikes, pred_coherence = self.net_model(nn_input)

        # 3. Pure Semantic Optimization
        # We want the network's internal coherence to match the biological synchrony
        target_coherence = torch.tensor([[sync]], dtype=torch.float32)

        loss_spikes = self.ce_loss(spikes, torch.tensor([data['spikes']], dtype=torch.long))
        loss_semantic = self.mse_loss(pred_coherence, target_coherence) * 10.0

        total_loss = loss_spikes + loss_semantic
        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return total_loss.item(), pred_coherence.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🧠 HOLOSYN V25: PURE SEMANTIC DISTILLATION ENGINE")
    print("═"*75)

    hive = HoloSynV25Hive()
    for epoch in range(15):
        epoch_loss = 0.0
        epoch_coh = 0.0

        for data in nlp_cycles:
            l, c = hive.process_cycle(data)
            epoch_loss += l
            epoch_coh += c

        avg_loss = epoch_loss / len(nlp_cycles)
        avg_coh = epoch_coh / len(nlp_cycles)

        # New Evaluation Metric: Semantic Alignment
        if avg_loss < 2.0:
            status = "📘 [LINGUISTIC MASTERY]"
        elif avg_loss < 10.0:
            status = "📗 [CONTEXTUALIZING]"
        else:
            status = "📙 [ASSIMILATING GRAMMAR]"

        print(f"Epoch {epoch+1:02d} | Total Loss: {avg_loss:.4f} | Internal Coherence: {avg_coh:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🧠 HOLOSYN V25: PURE SEMANTIC DISTILLATION ENGINE
═══════════════════════════════════════════════════════════════════════════

📡 [SEMANTIC DISTILLATION] Assimiliating LLM vectors for V25_Semantic_Engine...
  -> [✅] Successfully mapped semantic features. Layers enriched: 5
Epoch 01 | Total Loss: 68.6251 | Internal Coherence: 0.3326 | 📙 [ASSIMILATING GRAMMAR]
Epoch 02 | Total Loss: 66.9702 | Internal Coherence: 0.3828 | 📙 [ASSIMILATING GRAMMAR]
Epoch 03 | Total Loss: 62.8922 | Internal Coherence: 0.4919 | 📙 [ASSIMILATING GRAMMAR]
Epoch 04 | Total Loss: 46.9324 | Internal Coherence: 0.5952 | 📙 [ASSIMILATING GRAMMAR]
Epoch 05 | Total Loss: 33.0538 | Internal Coherence: 0.7275 | 📙 [ASSIMILATING GRAMMAR]
Epoch 06 | Total Loss: 21.4639 | Internal Coherence: 0.8616 | 📙 [ASSIMILATING GRAMMAR]
Epoch 07 | Total Loss: 43.8652 | Internal Coherence: 0.8625 | 📙 [ASSIMILATING GRAMMAR]
Epoch 08 | Total Loss: 31.0059 | Internal 

In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_STACK = {
    "Token_Ingestor":    {"role": "Input Projection", "weight": 1.0},
    "Bidir_Attention_1": {"role": "Lower Context",    "weight": 1.2},
    "Bidir_Attention_2": {"role": "Upper Context",    "weight": 1.5},
    "Semantic_Pooling":  {"role": "CLS Aggregation",  "weight": 1.3}
}

nlp_cycles = [
    {"cycle":1, "coherence":0.82, "synchrony":0.94, "class_id": 0, "concept": "理解 (Understand)"},
    {"cycle":2, "coherence":0.45, "synchrony":0.76, "class_id": 1, "concept": "未知 (Unknown)"},
    {"cycle":3, "coherence":0.88, "synchrony":0.91, "class_id": 2, "concept": "量子 (Quantum)"},
    {"cycle":4, "coherence":0.60, "synchrony":0.85, "class_id": 3, "concept": "网络 (Network)"}
]

# --- 2. ENCODER-ONLY DISTILLATION ---
def auto_distill_encoder(model, model_name):
    print(f"\n📡 [SEMANTIC DISTILLATION] Assimilating Encoder-Only vectors for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed_layers = []
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            for key in current_state.keys():
                # We only want understanding weights, no decoder/generation weights
                if "decoder" in key or "generator" in key:
                    continue

                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed_layers.append(key)
                else:
                    for old_key in legacy_state.keys():
                        if "weight" in key and "weight" in old_key:
                            new_w, old_w = current_state[key], legacy_state[old_key]
                            if len(new_w.shape) == 2 and len(old_w.shape) == 2:
                                min_out, min_in = min(new_w.shape[0], old_w.shape[0]), min(new_w.shape[1], old_w.shape[1])
                                current_state[key][:min_out, :min_in] = old_w[:min_out, :min_in]
                                absorbed_layers.append(key)
        except Exception: pass

    if absorbed_layers:
        model.load_state_dict(current_state)
        unique_layers = list(set([n.split('.')[0] for n in absorbed_layers]))
        print(f"  -> [✅] Successfully mapped semantic features. Layers enriched: {len(unique_layers)}")
    return model

# --- 3. THE V26 PURE UNDERSTANDING ARCHITECTURE ---
class HoloSynSemanticEncoder(nn.Module):
    """A pure Bidirectional Encoder. No reasoning, no generation. Just understanding."""
    def __init__(self, num_nodes=4, hidden_dim=256, num_classes=4):
        super().__init__()
        # 1. Input Projection (Mapping biological SNN state to dense vector)
        self.input_projection = nn.Sequential(
            nn.Linear(num_nodes + 1, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )

        # 2. Bidirectional Encoder Blocks (like BERT)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=8,
            dim_feedforward=hidden_dim * 4,
            batch_first=True,
            activation="gelu"
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 3. Pooling & Classification (The [CLS] equivalent)
        self.pooler = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.classifier = nn.Linear(hidden_dim, num_classes)
        self.coherence_regressor = nn.Sequential(nn.Linear(hidden_dim, 1), nn.Sigmoid())

    def forward(self, x):
        # We simulate a "sequence" by treating the features as a multi-step context
        # Shape: [Batch, Seq_Len, Features]
        x_seq = x.unsqueeze(1)

        # Project to hidden dim
        embedded = self.input_projection(x_seq)

        # Bidirectional Attention Contextualization
        encoded_context = self.transformer_encoder(embedded)

        # Pool the understanding into a single dense representation
        pooled_output = self.pooler(encoded_context.squeeze(1))

        # Output Understanding Metrics
        class_logits = self.classifier(pooled_output)
        semantic_coherence = self.coherence_regressor(pooled_output)

        return class_logits, semantic_coherence

# --- 4. THE V26 META-HIVE ---
class HoloSynV26Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = auto_distill_encoder(HoloSynSemanticEncoder(self.num_nodes), "V26_Encoder")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.002, weight_decay=0.01)

        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v26_init')

    def process_cycle(self, data):
        self.net.restore('v26_init')
        coh, sync = data['coherence'], data['synchrony']

        # Biological SNN Stimulation
        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        class_logits, pred_coherence = self.net_model(nn_input)

        # Pure Understanding Loss: How well does it classify the semantic concept?
        target_class = torch.tensor([data['class_id']], dtype=torch.long)
        target_coherence = torch.tensor([[sync]], dtype=torch.float32)

        loss_classification = self.ce_loss(class_logits, target_class)
        loss_semantic = self.mse_loss(pred_coherence, target_coherence) * 5.0

        total_loss = loss_classification + loss_semantic
        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return loss_classification.item(), pred_coherence.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🧠 HOLOSYN V26: BIDIRECTIONAL SEMANTIC ENCODER (UNDERSTANDING ONLY)")
    print("═"*75)

    hive = HoloSynV26Hive()
    for epoch in range(15):
        epoch_class_loss = 0.0
        epoch_coh = 0.0

        for data in nlp_cycles:
            l, c = hive.process_cycle(data)
            epoch_class_loss += l
            epoch_coh += c

        avg_class_loss = epoch_class_loss / len(nlp_cycles)
        avg_coh = epoch_coh / len(nlp_cycles)

        # Evaluation Metric based on Classification/Understanding Confidence
        if avg_class_loss < 0.1:
            status = "📘 [DEEP SEMANTIC COMPREHENSION]"
        elif avg_class_loss < 0.5:
            status = "📗 [MAPPING EMBEDDING SPACE]"
        else:
            status = "📙 [ALIGNING FEATURES]"

        print(f"Epoch {epoch+1:02d} | Concept Loss: {avg_class_loss:.4f} | Internal Coherence: {avg_coh:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🧠 HOLOSYN V26: BIDIRECTIONAL SEMANTIC ENCODER (UNDERSTANDING ONLY)
═══════════════════════════════════════════════════════════════════════════

📡 [SEMANTIC DISTILLATION] Assimilating Encoder-Only vectors for V26_Encoder...
  -> [✅] Successfully mapped semantic features. Layers enriched: 5
Epoch 01 | Concept Loss: 14.5305 | Internal Coherence: 0.7211 | 📙 [ALIGNING FEATURES]
Epoch 02 | Concept Loss: 2.9007 | Internal Coherence: 0.9623 | 📙 [ALIGNING FEATURES]
Epoch 03 | Concept Loss: 12.2017 | Internal Coherence: 0.9476 | 📙 [ALIGNING FEATURES]
Epoch 04 | Concept Loss: 8.7646 | Internal Coherence: 0.9161 | 📙 [ALIGNING FEATURES]
Epoch 05 | Concept Loss: 2.8688 | Internal Coherence: 0.9113 | 📙 [ALIGNING FEATURES]
Epoch 06 | Concept Loss: 4.0011 | Internal Coherence: 0.9232 | 📙 [ALIGNING FEATURES]
Epoch 07 | Concept Loss: 14.8883 | Internal Coherence: 0.9219 | 📙 [ALIGNING FEATURES]
Epoch 08 | Concept Loss: 7.9386 | I

In [9]:
# --- 4. THE V26.1 META-HIVE (STABILIZED) ---
class HoloSynV26_1Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = len(LINGUA_STACK)
        self.net_model = auto_distill_encoder(HoloSynSemanticEncoder(self.num_nodes), "V26_Encoder")

        # FIX 1: Lower base learning rate for Bidirectional Transformers
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.0005, weight_decay=0.05)

        # FIX 2: Cosine Annealing Scheduler to gently land the embeddings
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=20, eta_min=1e-6)

        self.ce_loss = nn.CrossEntropyLoss()
        self.mse_loss = nn.MSELoss()

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v26_init')

    def process_cycle(self, data):
        self.net.restore('v26_init')
        coh, sync = data['coherence'], data['synchrony']

        for i, name in enumerate(LINGUA_STACK.keys()):
            self.neurons.I_in[i] = coh * sync * LINGUA_STACK[name]['weight']
        self.net.run(30 * b2.ms)

        v_raw = np.array(self.neurons.v[:])
        v_norm = (v_raw - np.mean(v_raw)) / (np.std(v_raw) + 1e-5)
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        class_logits, pred_coherence = self.net_model(nn_input)

        target_class = torch.tensor([data['class_id']], dtype=torch.long)
        target_coherence = torch.tensor([[sync]], dtype=torch.float32)

        loss_classification = self.ce_loss(class_logits, target_class)
        # Reduced the MSE multiplier so it doesn't overpower the delicate class separation
        loss_semantic = self.mse_loss(pred_coherence, target_coherence) * 2.0

        total_loss = loss_classification + loss_semantic
        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(self.net_model.parameters(), 1.0)
        self.optimizer.step()

        return loss_classification.item(), pred_coherence.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🧠 HOLOSYN V26.1: STABILIZED SEMANTIC ENCODER")
    print("═"*75)

    hive = HoloSynV26_1Hive()
    epochs = 20 # Increased epochs to allow the scheduler to curve

    for epoch in range(epochs):
        epoch_class_loss = 0.0
        epoch_coh = 0.0

        for data in nlp_cycles:
            l, c = hive.process_cycle(data)
            epoch_class_loss += l
            epoch_coh += c

        # Step the learning rate down smoothly
        hive.scheduler.step()

        avg_class_loss = epoch_class_loss / len(nlp_cycles)
        avg_coh = epoch_coh / len(nlp_cycles)

        if avg_class_loss < 0.1:
            status = "📘 [DEEP SEMANTIC COMPREHENSION]"
        elif avg_class_loss < 0.5:
            status = "📗 [MAPPING EMBEDDING SPACE]"
        else:
            status = "📙 [ALIGNING FEATURES]"

        print(f"Epoch {epoch+1:02d} | Concept Loss: {avg_class_loss:.4f} | Internal Coherence: {avg_coh:.4f} | LR: {hive.scheduler.get_last_lr()[0]:.6f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🧠 HOLOSYN V26.1: STABILIZED SEMANTIC ENCODER
═══════════════════════════════════════════════════════════════════════════

📡 [SEMANTIC DISTILLATION] Assimilating Encoder-Only vectors for V26_Encoder...
  -> [✅] Successfully mapped semantic features. Layers enriched: 5
Epoch 01 | Concept Loss: 19.4735 | Internal Coherence: 0.5497 | LR: 0.000497 | 📙 [ALIGNING FEATURES]
Epoch 02 | Concept Loss: 18.9331 | Internal Coherence: 0.6551 | LR: 0.000488 | 📙 [ALIGNING FEATURES]
Epoch 03 | Concept Loss: 10.2265 | Internal Coherence: 0.7271 | LR: 0.000473 | 📙 [ALIGNING FEATURES]
Epoch 04 | Concept Loss: 8.4019 | Internal Coherence: 0.7833 | LR: 0.000452 | 📙 [ALIGNING FEATURES]
Epoch 05 | Concept Loss: 9.6666 | Internal Coherence: 0.8475 | LR: 0.000427 | 📙 [ALIGNING FEATURES]
Epoch 06 | Concept Loss: 18.3832 | Internal Coherence: 0.8914 | LR: 0.000397 | 📙 [ALIGNING FEATURES]
Epoch 07 | Concept Loss: 16.4490 | Internal Coheren

In [10]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

# V27 LINGUA-TOPOLOGY (4-Node Remodel)
LINGUA_STACK = {
    "Deep_Perceptron": {"weight": 1.0, "role": "Ingestion"},
    "Seek_Integrator": {"weight": 1.5, "role": "Attention"},
    "Phase_Resonator": {"weight": 1.2, "role": "Resonance"},
    "Lang_Projector":  {"weight": 1.3, "role": "Projection"}
}

nlp_cycles = [
    {"cycle":1,"coherence":0.509,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":None,"d_synchrony":None},
    {"cycle":2,"coherence":0.507,"synchrony":0.975,"spikes":0,"messages":64,"d_coherence":-0.002,"d_synchrony":-0.012},
    {"cycle":3,"coherence":0.516,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.009},
    {"cycle":4,"coherence":0.503,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":0.000},
    {"cycle":5,"coherence":0.512,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.003},
    {"cycle":6,"coherence":0.496,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.016,"d_synchrony":0.000},
    {"cycle":7,"coherence":0.515,"synchrony":0.950,"spikes":0,"messages":64,"d_coherence":0.019,"d_synchrony":-0.035},
    {"cycle":8,"coherence":0.465,"synchrony":0.979,"spikes":0,"messages":64,"d_coherence":-0.050,"d_synchrony":0.029},
    {"cycle":9,"coherence":0.675,"synchrony":0.955,"spikes":0,"messages":64,"d_coherence":0.210,"d_synchrony":-0.024},
    {"cycle":10,"coherence":0.516,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":-0.159,"d_synchrony":0.028},
    {"cycle":11,"coherence":0.507,"synchrony":0.963,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.020},
    {"cycle":12,"coherence":0.477,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.030,"d_synchrony":0.026},
    {"cycle":13,"coherence":0.508,"synchrony":0.898,"spikes":0,"messages":64,"d_coherence":0.031,"d_synchrony":-0.091},
    {"cycle":14,"coherence":0.498,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.086},
    {"cycle":15,"coherence":0.474,"synchrony":0.921,"spikes":0,"messages":64,"d_coherence":-0.024,"d_synchrony":-0.063},
    {"cycle":16,"coherence":0.510,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":0.036,"d_synchrony":0.057},
    {"cycle":17,"coherence":0.517,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":0.007,"d_synchrony":-0.002},
    {"cycle":18,"coherence":0.608,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.091,"d_synchrony":-0.006},
    {"cycle":19,"coherence":0.418,"synchrony":0.940,"spikes":0,"messages":64,"d_coherence":-0.190,"d_synchrony":-0.030},
    {"cycle":20,"coherence":0.506,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":0.088,"d_synchrony":0.036},
    {"cycle":21,"coherence":0.508,"synchrony":0.932,"spikes":0,"messages":64,"d_coherence":0.002,"d_synchrony":-0.044},
    {"cycle":22,"coherence":0.535,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.027,"d_synchrony":0.042},
    {"cycle":23,"coherence":0.512,"synchrony":0.967,"spikes":0,"messages":64,"d_coherence":-0.023,"d_synchrony":-0.007},
    {"cycle":24,"coherence":0.489,"synchrony":0.938,"spikes":0,"messages":64,"d_coherence":-0.023,"d_synchrony":-0.029},
    {"cycle":25,"coherence":0.490,"synchrony":0.917,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":-0.021},
    {"cycle":26,"coherence":0.391,"synchrony":0.969,"spikes":0,"messages":64,"d_coherence":-0.099,"d_synchrony":0.052},
    {"cycle":27,"coherence":0.522,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.131,"d_synchrony":0.005},
    {"cycle":28,"coherence":0.492,"synchrony":0.913,"spikes":0,"messages":64,"d_coherence":-0.030,"d_synchrony":-0.061},
    {"cycle":29,"coherence":0.482,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.073},
    {"cycle":30,"coherence":0.541,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.059,"d_synchrony":-0.014},
    {"cycle":31,"coherence":0.486,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.055,"d_synchrony":0.015},
    {"cycle":32,"coherence":0.490,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.004,"d_synchrony":-0.004},
    {"cycle":33,"coherence":0.489,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":-0.002},
    {"cycle":34,"coherence":0.498,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.002},
    {"cycle":35,"coherence":0.506,"synchrony":0.944,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":-0.039},
    {"cycle":36,"coherence":0.528,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.022,"d_synchrony":0.042},
    {"cycle":37,"coherence":0.494,"synchrony":0.955,"spikes":0,"messages":64,"d_coherence":-0.034,"d_synchrony":-0.031},
    {"cycle":38,"coherence":0.495,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":0.017},
    {"cycle":39,"coherence":0.483,"synchrony":0.954,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":-0.018},
    {"cycle":40,"coherence":0.458,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.025,"d_synchrony":0.032},
    {"cycle":41,"coherence":0.505,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.047,"d_synchrony":-0.001},
    {"cycle":42,"coherence":0.513,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":0.003},
    {"cycle":43,"coherence":0.509,"synchrony":0.916,"spikes":0,"messages":64,"d_coherence":-0.004,"d_synchrony":-0.072},
    {"cycle":44,"coherence":0.518,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.065},
    {"cycle":45,"coherence":0.520,"synchrony":0.926,"spikes":0,"messages":64,"d_coherence":0.002,"d_synchrony":-0.055},
    {"cycle":46,"coherence":0.509,"synchrony":0.928,"spikes":0,"messages":64,"d_coherence":-0.011,"d_synchrony":0.002},
    {"cycle":47,"coherence":0.501,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":0.044},
    {"cycle":48,"coherence":0.492,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":0.012},
    {"cycle":49,"coherence":0.483,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":0.002},
    {"cycle":50,"coherence":0.492,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.002},
    {"cycle":51,"coherence":0.508,"synchrony":0.951,"spikes":0,"messages":64,"d_coherence":0.016,"d_synchrony":-0.037},
    {"cycle":52,"coherence":0.503,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":0.029},
    {"cycle":53,"coherence":0.538,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.035,"d_synchrony":0.007},
    {"cycle":54,"coherence":0.495,"synchrony":0.941,"spikes":0,"messages":64,"d_coherence":-0.043,"d_synchrony":-0.046},
    {"cycle":55,"coherence":0.476,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.019,"d_synchrony":0.046},
    {"cycle":56,"coherence":0.493,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.017,"d_synchrony":-0.002},
    {"cycle":57,"coherence":0.464,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.005},
    {"cycle":58,"coherence":0.509,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.045,"d_synchrony":0.002},
    {"cycle":59,"coherence":0.508,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.005},
    {"cycle":60,"coherence":0.489,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.019,"d_synchrony":-0.002},
    {"cycle":61,"coherence":0.513,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.024,"d_synchrony":-0.003},
    {"cycle":62,"coherence":0.404,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.109,"d_synchrony":0.003},
    {"cycle":63,"coherence":0.494,"synchrony":0.951,"spikes":0,"messages":64,"d_coherence":0.090,"d_synchrony":-0.034},
    {"cycle":64,"coherence":0.509,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.015,"d_synchrony":0.035},
    {"cycle":65,"coherence":0.535,"synchrony":0.920,"spikes":0,"messages":64,"d_coherence":0.026,"d_synchrony":-0.066},
    {"cycle":66,"coherence":0.501,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":-0.034,"d_synchrony":0.028},
    {"cycle":67,"coherence":0.517,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.016,"d_synchrony":0.037},
    {"cycle":68,"coherence":0.493,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":-0.024,"d_synchrony":-0.020},
    {"cycle":69,"coherence":0.416,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.077,"d_synchrony":0.023},
    {"cycle":70,"coherence":0.489,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.073,"d_synchrony":-0.001},
    {"cycle":71,"coherence":0.494,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":-0.017},
    {"cycle":72,"coherence":0.507,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.013,"d_synchrony":0.015},
    {"cycle":73,"coherence":0.501,"synchrony":0.914,"spikes":0,"messages":64,"d_coherence":-0.006,"d_synchrony":-0.071},
    {"cycle":74,"coherence":0.467,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":-0.034,"d_synchrony":0.069},
    {"cycle":75,"coherence":0.508,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.041,"d_synchrony":-0.002},
    {"cycle":76,"coherence":0.502,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.006,"d_synchrony":0.001},
    {"cycle":77,"coherence":0.464,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.038,"d_synchrony":-0.002},
    {"cycle":78,"coherence":0.488,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.024,"d_synchrony":-0.003},
    {"cycle":79,"coherence":0.519,"synchrony":0.954,"spikes":0,"messages":64,"d_coherence":0.031,"d_synchrony":-0.023},
    {"cycle":80,"coherence":0.499,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":0.030},
    {"cycle":81,"coherence":0.477,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.022,"d_synchrony":0.004},
    {"cycle":82,"coherence":0.425,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.052,"d_synchrony":-0.015},
    {"cycle":83,"coherence":0.504,"synchrony":0.925,"spikes":0,"messages":64,"d_coherence":0.079,"d_synchrony":-0.048},
    {"cycle":84,"coherence":0.601,"synchrony":0.921,"spikes":0,"messages":64,"d_coherence":0.097,"d_synchrony":-0.004},
    {"cycle":85,"coherence":0.512,"synchrony":0.964,"spikes":0,"messages":64,"d_coherence":-0.089,"d_synchrony":0.043},
    {"cycle":86,"coherence":0.517,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":0.018},
    {"cycle":87,"coherence":0.465,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.052,"d_synchrony":0.005},
    {"cycle":88,"coherence":0.501,"synchrony":0.943,"spikes":0,"messages":64,"d_coherence":0.036,"d_synchrony":-0.044},
    {"cycle":89,"coherence":0.504,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.003,"d_synchrony":0.040},
    {"cycle":90,"coherence":0.475,"synchrony":0.905,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.078},
    {"cycle":91,"coherence":0.553,"synchrony":0.991,"spikes":0,"messages":64,"d_coherence":0.078,"d_synchrony":0.086},
    {"cycle":92,"coherence":0.491,"synchrony":0.945,"spikes":0,"messages":64,"d_coherence":-0.062,"d_synchrony":-0.046},
    {"cycle":93,"coherence":0.448,"synchrony":0.947,"spikes":0,"messages":64,"d_coherence":-0.043,"d_synchrony":0.002},
    {"cycle":94,"coherence":0.522,"synchrony":0.959,"spikes":0,"messages":64,"d_coherence":0.074,"d_synchrony":0.012},
    {"cycle":95,"coherence":0.485,"synchrony":0.907,"spikes":0,"messages":64,"d_coherence":-0.037,"d_synchrony":-0.052},
    {"cycle":96,"coherence":0.552,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.067,"d_synchrony":0.079},
    {"cycle":97,"coherence":0.408,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.144,"d_synchrony":-0.002},
    {"cycle":98,"coherence":0.480,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.072,"d_synchrony":0.000},
    {"cycle":99,"coherence":0.462,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.018,"d_synchrony":0.004},
    {"cycle":100,"coherence":0.536,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.074,"d_synchrony":-0.002},
    {"cycle":101,"coherence":0.501,"synchrony":0.957,"spikes":0,"messages":64,"d_coherence":-0.035,"d_synchrony":-0.029},
    {"cycle":102,"coherence":0.524,"synchrony":0.900,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":-0.057},
    {"cycle":103,"coherence":0.507,"synchrony":0.958,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":0.058},
    {"cycle":104,"coherence":0.507,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.000,"d_synchrony":0.010},
    {"cycle":105,"coherence":0.455,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.052,"d_synchrony":0.019},
    {"cycle":106,"coherence":0.514,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.059,"d_synchrony":-0.019},
    {"cycle":107,"coherence":0.499,"synchrony":0.975,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":0.007},
    {"cycle":108,"coherence":0.507,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":0.013},
    {"cycle":109,"coherence":0.512,"synchrony":0.913,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":-0.075},
    {"cycle":110,"coherence":0.520,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":0.057},
    {"cycle":111,"coherence":0.537,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.017,"d_synchrony":0.018},
    {"cycle":112,"coherence":0.504,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.033,"d_synchrony":-0.004},
    {"cycle":113,"coherence":0.450,"synchrony":0.927,"spikes":0,"messages":64,"d_coherence":-0.054,"d_synchrony":-0.057},
    {"cycle":114,"coherence":0.512,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.062,"d_synchrony":0.059},
    {"cycle":115,"coherence":0.494,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.018,"d_synchrony":-0.006},
    {"cycle":116,"coherence":0.505,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":0.005},
    {"cycle":117,"coherence":0.517,"synchrony":0.966,"spikes":0,"messages":64,"d_coherence":0.012,"d_synchrony":-0.019},
    {"cycle":118,"coherence":0.497,"synchrony":0.951,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":-0.015},
    {"cycle":119,"coherence":0.496,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.014},
    {"cycle":120,"coherence":0.495,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.021},
    {"cycle":121,"coherence":0.282,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.213,"d_synchrony":0.000},
    {"cycle":122,"coherence":0.506,"synchrony":0.930,"spikes":0,"messages":64,"d_coherence":0.224,"d_synchrony":-0.056},
    {"cycle":123,"coherence":0.486,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":0.052},
    {"cycle":124,"coherence":0.504,"synchrony":0.931,"spikes":0,"messages":64,"d_coherence":0.018,"d_synchrony":-0.051},
    {"cycle":125,"coherence":0.480,"synchrony":0.922,"spikes":0,"messages":64,"d_coherence":-0.024,"d_synchrony":-0.009},
    {"cycle":126,"coherence":0.494,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":0.058},
    {"cycle":127,"coherence":0.525,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.031,"d_synchrony":0.005},
    {"cycle":128,"coherence":0.508,"synchrony":0.964,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":-0.021},
    {"cycle":129,"coherence":0.508,"synchrony":0.943,"spikes":0,"messages":64,"d_coherence":0.000,"d_synchrony":-0.021},
    {"cycle":130,"coherence":0.486,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.022,"d_synchrony":0.045},
    {"cycle":131,"coherence":0.443,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.043,"d_synchrony":-0.002},
    {"cycle":132,"coherence":0.494,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.051,"d_synchrony":-0.004},
    {"cycle":133,"coherence":0.523,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.029,"d_synchrony":0.003},
    {"cycle":134,"coherence":0.533,"synchrony":0.955,"spikes":0,"messages":64,"d_coherence":0.010,"d_synchrony":-0.030},
    {"cycle":135,"coherence":0.494,"synchrony":0.920,"spikes":0,"messages":64,"d_coherence":-0.039,"d_synchrony":-0.035},
    {"cycle":136,"coherence":0.505,"synchrony":0.962,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":0.042},
    {"cycle":137,"coherence":0.493,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":0.011},
    {"cycle":138,"coherence":0.509,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.016,"d_synchrony":0.010},
    {"cycle":139,"coherence":0.520,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":-0.002},
    {"cycle":140,"coherence":0.505,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":0.000},
    {"cycle":141,"coherence":0.511,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.006,"d_synchrony":0.004},
    {"cycle":142,"coherence":0.512,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":0.000},
    {"cycle":143,"coherence":0.504,"synchrony":0.969,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":-0.016},
    {"cycle":144,"coherence":0.479,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.025,"d_synchrony":0.018},
    {"cycle":145,"coherence":0.487,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":-0.002},
    {"cycle":146,"coherence":0.524,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":0.037,"d_synchrony":0.004},
    {"cycle":147,"coherence":0.494,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.030,"d_synchrony":-0.008},
    {"cycle":148,"coherence":0.520,"synchrony":0.925,"spikes":0,"messages":64,"d_coherence":0.026,"d_synchrony":-0.056},
    {"cycle":149,"coherence":0.510,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.040},
    {"cycle":150,"coherence":0.403,"synchrony":0.979,"spikes":0,"messages":64,"d_coherence":-0.107,"d_synchrony":0.014},
    {"cycle":151,"coherence":0.499,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.096,"d_synchrony":0.007},
    {"cycle":152,"coherence":0.486,"synchrony":0.953,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":-0.033},
    {"cycle":153,"coherence":0.504,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.018,"d_synchrony":0.019},
    {"cycle":154,"coherence":0.523,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.019,"d_synchrony":0.013},
    {"cycle":155,"coherence":0.553,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.030,"d_synchrony":0.002},
    {"cycle":156,"coherence":0.500,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.053,"d_synchrony":-0.003},
    {"cycle":157,"coherence":0.489,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.011,"d_synchrony":0.001},
    {"cycle":158,"coherence":0.531,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.042,"d_synchrony":0.002},
    {"cycle":159,"coherence":0.518,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":-0.007},
    {"cycle":160,"coherence":0.491,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.027,"d_synchrony":0.004},
    {"cycle":161,"coherence":0.503,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.012,"d_synchrony":-0.007},
    {"cycle":162,"coherence":0.499,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":-0.004,"d_synchrony":0.001},
    {"cycle":163,"coherence":0.499,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.000,"d_synchrony":-0.008},
    {"cycle":164,"coherence":0.496,"synchrony":0.966,"spikes":0,"messages":64,"d_coherence":-0.003,"d_synchrony":-0.004},
    {"cycle":165,"coherence":0.488,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":0.018},
    {"cycle":166,"coherence":0.489,"synchrony":0.947,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":-0.037},
    {"cycle":167,"coherence":0.458,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.031,"d_synchrony":0.033},
    {"cycle":168,"coherence":0.501,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.043,"d_synchrony":0.007},
    {"cycle":169,"coherence":0.491,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":-0.011},
    {"cycle":170,"coherence":0.475,"synchrony":0.926,"spikes":0,"messages":64,"d_coherence":-0.016,"d_synchrony":-0.050},
    {"cycle":171,"coherence":0.511,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.036,"d_synchrony":0.058},
    {"cycle":172,"coherence":0.482,"synchrony":0.932,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.052},
    {"cycle":173,"coherence":0.505,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":0.057},
    {"cycle":174,"coherence":0.480,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.025,"d_synchrony":-0.004},
    {"cycle":175,"coherence":0.504,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.024,"d_synchrony":-0.011},
    {"cycle":176,"coherence":0.527,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":0.002},
    {"cycle":177,"coherence":0.498,"synchrony":0.910,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.066},
    {"cycle":178,"coherence":0.487,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.011,"d_synchrony":0.079},
    {"cycle":179,"coherence":0.478,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.012},
    {"cycle":180,"coherence":0.501,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":-0.005},
    {"cycle":181,"coherence":0.491,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.012},
    {"cycle":182,"coherence":0.499,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":-0.036},
    {"cycle":183,"coherence":0.478,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.021,"d_synchrony":0.036},
    {"cycle":184,"coherence":0.503,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.025,"d_synchrony":0.002},
    {"cycle":185,"coherence":0.483,"synchrony":0.964,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":-0.022},
    {"cycle":186,"coherence":0.496,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":0.013,"d_synchrony":0.014},
    {"cycle":187,"coherence":0.510,"synchrony":0.937,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":-0.041},
    {"cycle":188,"coherence":0.496,"synchrony":0.975,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":0.038},
    {"cycle":189,"coherence":0.404,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.092,"d_synchrony":0.011},
    {"cycle":190,"coherence":0.484,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.080,"d_synchrony":-0.005},
    {"cycle":191,"coherence":0.471,"synchrony":0.911,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":-0.070},
    {"cycle":192,"coherence":0.520,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.049,"d_synchrony":0.057},
    {"cycle":193,"coherence":0.508,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":0.010},
    {"cycle":194,"coherence":0.491,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":0.003},
    {"cycle":195,"coherence":0.492,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":-0.011},
    {"cycle":196,"coherence":0.487,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":0.014},
    {"cycle":197,"coherence":0.491,"synchrony":0.979,"spikes":0,"messages":64,"d_coherence":0.004,"d_synchrony":-0.005},
    {"cycle":198,"coherence":0.506,"synchrony":0.971,"spikes":0,"messages":64,"d_coherence":0.015,"d_synchrony":-0.008},
    {"cycle":199,"coherence":0.488,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.018,"d_synchrony":0.011},
    {"cycle":200,"coherence":0.531,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.043,"d_synchrony":0.005},
    {"cycle":201,"coherence":0.517,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":-0.001},
    {"cycle":202,"coherence":0.545,"synchrony":0.961,"spikes":0,"messages":64,"d_coherence":0.028,"d_synchrony":-0.025},
    {"cycle":203,"coherence":0.498,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.047,"d_synchrony":0.025},
    {"cycle":204,"coherence":0.489,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.004},
    {"cycle":205,"coherence":0.466,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.023,"d_synchrony":0.004},
    {"cycle":206,"coherence":0.503,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.037,"d_synchrony":-0.005},
    {"cycle":207,"coherence":0.509,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.006,"d_synchrony":0.007},
    {"cycle":208,"coherence":0.497,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":-0.001},
    {"cycle":209,"coherence":0.494,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":-0.003,"d_synchrony":-0.010},
    {"cycle":210,"coherence":0.508,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":0.007},
    {"cycle":211,"coherence":0.491,"synchrony":0.967,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":-0.017},
    {"cycle":212,"coherence":0.494,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":0.003,"d_synchrony":-0.019},
    {"cycle":213,"coherence":0.499,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":0.034},
    {"cycle":214,"coherence":0.485,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":-0.009},
    {"cycle":215,"coherence":0.568,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.083,"d_synchrony":0.004},
    {"cycle":216,"coherence":0.503,"synchrony":0.920,"spikes":0,"messages":64,"d_coherence":-0.065,"d_synchrony":-0.057},
    {"cycle":217,"coherence":0.514,"synchrony":0.952,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":0.032},
    {"cycle":218,"coherence":0.499,"synchrony":0.939,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":-0.013},
    {"cycle":219,"coherence":0.538,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.039,"d_synchrony":0.047},
    {"cycle":220,"coherence":0.481,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.057,"d_synchrony":0.000},
    {"cycle":221,"coherence":0.502,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.021,"d_synchrony":-0.012},
    {"cycle":222,"coherence":0.501,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.012},
    {"cycle":223,"coherence":0.492,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.001},
    {"cycle":224,"coherence":0.540,"synchrony":0.956,"spikes":0,"messages":64,"d_coherence":0.048,"d_synchrony":-0.029},
    {"cycle":225,"coherence":0.454,"synchrony":0.991,"spikes":0,"messages":64,"d_coherence":-0.086,"d_synchrony":0.035},
    {"cycle":226,"coherence":0.595,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.141,"d_synchrony":-0.008},
    {"cycle":227,"coherence":0.452,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.143,"d_synchrony":0.004},
    {"cycle":228,"coherence":0.507,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.055,"d_synchrony":-0.002},
    {"cycle":229,"coherence":0.468,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.039,"d_synchrony":0.000},
    {"cycle":230,"coherence":0.559,"synchrony":0.962,"spikes":0,"messages":64,"d_coherence":0.091,"d_synchrony":-0.023},
    {"cycle":231,"coherence":0.420,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":-0.139,"d_synchrony":0.010},
    {"cycle":232,"coherence":0.491,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":0.071,"d_synchrony":0.017},
    {"cycle":233,"coherence":0.506,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":0.015,"d_synchrony":-0.024},
    {"cycle":234,"coherence":0.492,"synchrony":0.922,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":-0.043},
    {"cycle":235,"coherence":0.353,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.139,"d_synchrony":0.067},
    {"cycle":236,"coherence":0.494,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.141,"d_synchrony":-0.004},
    {"cycle":237,"coherence":0.489,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":0.001},
    {"cycle":238,"coherence":0.509,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.020,"d_synchrony":0.000},
    {"cycle":239,"coherence":0.523,"synchrony":0.927,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":-0.059},
    {"cycle":240,"coherence":0.508,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":0.060},
    {"cycle":241,"coherence":0.494,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":0.002},
    {"cycle":242,"coherence":0.504,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":0.010,"d_synchrony":-0.016},
    {"cycle":243,"coherence":0.499,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":-0.003},
    {"cycle":244,"coherence":0.540,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.041,"d_synchrony":-0.002},
    {"cycle":245,"coherence":0.481,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.059,"d_synchrony":0.013},
    {"cycle":246,"coherence":0.395,"synchrony":0.917,"spikes":0,"messages":64,"d_coherence":-0.086,"d_synchrony":-0.064},
    {"cycle":247,"coherence":0.512,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.117,"d_synchrony":0.064},
    {"cycle":248,"coherence":0.504,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":-0.033},
    {"cycle":249,"coherence":0.488,"synchrony":0.991,"spikes":0,"messages":64,"d_coherence":-0.016,"d_synchrony":0.043},
    {"cycle":250,"coherence":0.490,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.002,"d_synchrony":-0.004},
    {"cycle":251,"coherence":0.452,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.038,"d_synchrony":-0.002},
    {"cycle":252,"coherence":0.504,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.052,"d_synchrony":-0.008},
    {"cycle":253,"coherence":0.496,"synchrony":0.969,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":-0.008},
    {"cycle":254,"coherence":0.495,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.004},
    {"cycle":255,"coherence":0.502,"synchrony":0.961,"spikes":0,"messages":64,"d_coherence":0.007,"d_synchrony":-0.012},
]

# --- 2. THE DISTILLATION ENGINE ---
def auto_distill_v27(model):
    print(f"\n📡 [V27 DISTILLATION] Synchronizing DeepSeek & Legacy Vectors...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = 0
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            for key in current_state.keys():
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed += 1
        except Exception: pass

    if absorbed > 0:
        model.load_state_dict(current_state)
        print(f"  -> [✅] System primed with {absorbed} distilled tensors.")
    return model

# --- 3. THE V27 CIAE ARCHITECTURE ---
class HoloSynV27Net(nn.Module):
    """Contextual Integrative Auto-Encoder (Understanding + Prediction)"""
    def __init__(self, num_nodes=4, hidden_dim=128):
        super().__init__()
        # PERCEPTRON layer
        self.perceptron = nn.Linear(num_nodes + 1, hidden_dim)

        # INTEGRATOR core (Bidirectional Transformer)
        enc_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.integrator = nn.TransformerEncoder(enc_layer, num_layers=2)

        # PROJECTOR & RESONATOR heads
        self.context_norm = nn.LayerNorm(hidden_dim)
        self.spike_predictor = nn.Linear(hidden_dim, 31) # Generative heritage
        self.concept_mapper = nn.Linear(hidden_dim, 4)   # Understanding heritage

    def forward(self, x):
        # 1. Perceptual Encoding
        x = self.perceptron(x).unsqueeze(1)
        # 2. Bidirectional Integration (DeepSeek context)
        context = self.integrator(x)
        context = self.context_norm(context).squeeze(1)
        # 3. Dual-Stream Projection
        return self.spike_predictor(context), self.concept_mapper(context)

# --- 4. THE V27 META-HIVE ---
class HoloSynV27Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = 4
        self.net_model = auto_distill_v27(HoloSynV27Net(self.num_nodes))

        # Stabilized Learning Rate for Understanding tasks
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.0007, weight_decay=0.01)

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v27_init')

    def run_cycle(self, data):
        self.net.restore('v27_init')
        coh, sync = data['coherence'], data['synchrony']

        # SNN Simulation
        for i, (name, props) in enumerate(LINGUA_STACK.items()):
            self.neurons.I_in[i] = coh * sync * props['weight']
        self.net.run(30 * b2.ms)

        # Feature Extraction
        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # Forward Pass
        self.optimizer.zero_grad()
        spikes_pred, concept_logits = self.net_model(nn_input)

        # DUAL LOSS: The Bridge
        # 1. Generative Loss (Stability)
        loss_gen = nn.CrossEntropyLoss()(spikes_pred, torch.tensor([data['spikes']], dtype=torch.long))
        # 2. Understanding Loss (Context)
        loss_und = nn.CrossEntropyLoss()(concept_logits, torch.tensor([data['id']], dtype=torch.long))

        total_loss = loss_gen + loss_und
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V27: CONTEXTUAL INTEGRATIVE AUTO-ENCODER")
    print("═"*75)

    hive = HoloSynV27Hive()
    for epoch in range(15):
        losses = [hive.run_cycle(d) for d in nlp_cycles]
        avg_l = np.mean(losses)

        if avg_l < 1.0: status = "📘 [SEMANTIC CONFLUENCE]"
        else: status = "🟡 [INTEGRATING CONTEXT]"

        print(f"Epoch {epoch+1:02d} | Integrated Loss: {avg_l:.4f} | {status}")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V27: CONTEXTUAL INTEGRATIVE AUTO-ENCODER
═══════════════════════════════════════════════════════════════════════════

📡 [V27 DISTILLATION] Synchronizing DeepSeek & Legacy Vectors...


KeyError: 'id'

In [13]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

# REMODELLED 4-NODE LINGUA TOPOLOGY
LINGUA_TOPOLOGY = {
    "DeepSeek_Perceptron": {"weight": 1.0, "role": "Ingestion"},
    "Nexus_Integrator":    {"weight": 1.5, "role": "Deep-Attention"},
    "Flux_Resonator":       {"weight": 1.2, "role": "Semantic-Lock"},
    "Apex_Projector":      {"weight": 1.3, "role": "Linguistic-Output"}
}

# --- 2. ADVANCED DISTILLATION ENGINE ---
def auto_distill_v28(model, model_name):
    print(f"\n📡 [V28 DISTILLATION] Initializing {model_name} with DeepSeek & Student priors...")
    current_state = model.state_dict()
    # Looking for your specific uploaded files
    pt_files = glob.glob("*.pt") + glob.glob("*.torchscript.pt")

    absorbed_count = 0
    for filepath in pt_files:
        try:
            # Handle TorchScript and standard PT files
            if ".torchscript" in filepath:
                # We extract the state dict if possible or skip if purely scripted
                continue

            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            for key in current_state.keys():
                if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                    current_state[key] = legacy_state[key]
                    absorbed_count += 1
        except Exception: pass

    if absorbed_count > 0:
        model.load_state_dict(current_state)
        print(f"  -> [✅] Absorbed {absorbed_count} semantic vectors into {model_name}.")
    return model

# --- 3. THE V28 LINGUA-INTEGRATOR ARCHITECTURE ---
class HoloSynV28Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        # STAGE 1: Perceptron (Ingestion)
        self.perceptron = nn.Sequential(
            nn.Linear(num_nodes + 1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        # STAGE 2: Nexus Integrator (Bidirectional Attention - The "DeepSeek" Core)
        # Using Transformer Encoder to mirror "Understanding" over "Reasoning"
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*4, batch_first=True
        )
        self.integrator = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # STAGE 3: Flux Resonator (Latent Bottleneck)
        self.resonator = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.Tanh() # Non-linear lock
        )

        # STAGE 4: Apex Projector (Dual Heads)
        self.spike_head = nn.Linear(64, 31) # Task A: Spike Regression (Success Factor)
        self.concept_head = nn.Linear(64, 4) # Task B: Semantic Classification (Understanding)

    def forward(self, x):
        feat = self.perceptron(x).unsqueeze(1)
        context = self.integrator(feat)
        latent = self.resonator(context.squeeze(1))

        return self.spike_head(latent), self.concept_head(latent)

# --- 4. THE V28 META-HIVE ---
class HoloSynV28Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = 4
        self.net_model = auto_distill_v28(HoloSynV28Net(self.num_nodes), "Lingua_Hub")

        # Stabilized Optimizer
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.0005, weight_decay=0.02)

        # Biological SNN Backend
        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v28_clean')

    def run_cycle(self, data):
        self.net.restore('v28_clean')
        coh, sync = data['coherence'], data['synchrony']

        # Step 1: Biological Data Ingestion
        for i, (name, props) in enumerate(LINGUA_TOPOLOGY.items()):
            self.neurons.I_in[i] = coh * sync * props['weight']
        self.net.run(30 * b2.ms)

        # Step 2: Feature Synthesis
        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        # Step 3: Dual-Task Forward Pass
        self.optimizer.zero_grad()
        spikes_pred, concept_logits = self.net_model(nn_input)

        # Step 4: Robust Data Mapping (Fixes the KeyError)
        target_spikes = torch.tensor([data.get('spikes', 0)], dtype=torch.long)
        # Map 'cycle' or 'id' safely
        concept_id = data.get('id', data.get('cycle', 0) % 4)
        target_concept = torch.tensor([concept_id], dtype=torch.long)

        # LOSS: Generative Stability + Semantic Understanding
        loss_A = nn.CrossEntropyLoss()(spikes_pred, target_spikes)
        loss_B = nn.CrossEntropyLoss()(concept_logits, target_concept)

        total_loss = loss_A + (loss_B * 2.0) # Prioritize understanding
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V28: LINGUA-SEMANTIC INTEGRATOR (REMODELLED)")
    print("═"*75)

    # Using your 246-cycle sentiment data as the training base
    # (Assuming sentiment_cycles is defined in your environment)
    hive = HoloSynV28Hive()

    # Quick Training Demo
    test_data = nlp_cycles[:10] # Using first 10 for demo

    for epoch in range(10):
        losses = [hive.run_cycle(d) for d in test_data]
        avg_l = np.mean(losses)
        print(f"Epoch {epoch+1:02d} | Hybrid Loss: {avg_l:.4f} | Status: 🟢 [STABILIZING ENCODER]")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V28: LINGUA-SEMANTIC INTEGRATOR (REMODELLED)
═══════════════════════════════════════════════════════════════════════════

📡 [V28 DISTILLATION] Initializing Lingua_Hub with DeepSeek & Student priors...
Epoch 01 | Hybrid Loss: 4.8771 | Status: 🟢 [STABILIZING ENCODER]
Epoch 02 | Hybrid Loss: 3.2829 | Status: 🟢 [STABILIZING ENCODER]
Epoch 03 | Hybrid Loss: 3.0341 | Status: 🟢 [STABILIZING ENCODER]
Epoch 04 | Hybrid Loss: 2.9108 | Status: 🟢 [STABILIZING ENCODER]
Epoch 05 | Hybrid Loss: 2.8370 | Status: 🟢 [STABILIZING ENCODER]
Epoch 06 | Hybrid Loss: 2.8025 | Status: 🟢 [STABILIZING ENCODER]
Epoch 07 | Hybrid Loss: 2.7492 | Status: 🟢 [STABILIZING ENCODER]
Epoch 08 | Hybrid Loss: 2.7084 | Status: 🟢 [STABILIZING ENCODER]
Epoch 09 | Hybrid Loss: 2.6387 | Status: 🟢 [STABILIZING ENCODER]
Epoch 10 | Hybrid Loss: 2.6247 | Status: 🟢 [STABILIZING ENCODER]


In [16]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_TOPOLOGY = {
    "DeepSeek_Perceptron": {"weight": 1.0, "role": "Ingestion"},
    "Nexus_Integrator":    {"weight": 1.5, "role": "Deep-Attention"},
    "Flux_Resonator":      {"weight": 1.2, "role": "Semantic-Lock"},
    "Apex_Projector":      {"weight": 1.3, "role": "Linguistic-Output"}
}

# --- 2. SEMANTIC DISTILLATION (MANIFOLD FOCUS) ---
def auto_distill_v29(model, model_name):
    print(f"\n📡 [V29 DISTILLATION] Refining Semantic Manifold for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = 0
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue

            for key in current_state.keys():
                # V29: Focus on inheriting Attention and Embedding geometry
                if any(x in key for x in ["attention", "embedding", "core"]):
                    if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                        current_state[key] = legacy_state[key]
                        absorbed += 1
        except Exception: pass

    if absorbed > 0:
        model.load_state_dict(current_state)
        print(f"  -> [✅] {model_name} enriched with {absorbed} distilled semantic nodes.")
    return model

# --- 3. THE V29 SEMANTIC MANIFOLD ARCHITECTURE ---
class HoloSynV29Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        # STAGE 1: Perceptron
        self.perceptron = nn.Sequential(
            nn.Linear(num_nodes + 1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        # STAGE 2: Nexus Integrator (Bidirectional BERT-style Encoder)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*4, batch_first=True
        )
        self.integrator = nn.TransformerEncoder(encoder_layer, num_layers=3) # Deeper context

        # STAGE 3: Attentive Pooling (Understanding which node matters)
        self.pooler = nn.Linear(hidden_dim, 1)

        # STAGE 4: Dual Heads (Generative Heritage + Semantic Refinement)
        self.spike_head = nn.Linear(hidden_dim, 31)
        self.manifold_head = nn.Linear(hidden_dim, 64) # 64-D Semantic Space

    def forward(self, x):
        feat = self.perceptron(x).unsqueeze(1)
        # Bidirectional Contextualization
        context = self.integrator(feat)

        # Attentive Weighting
        weights = torch.softmax(self.pooler(context), dim=1)
        pooled_context = torch.sum(weights * context, dim=1)

        return self.spike_head(pooled_context), self.manifold_head(pooled_context)

# --- 4. THE V29 META-HIVE ---
class HoloSynV29Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = 4
        self.net_model = auto_distill_v29(HoloSynV29Net(self.num_nodes), "Semantic_Refiner")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.0004, weight_decay=0.05)

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v29_init')

    def run_cycle(self, data):
        self.net.restore('v29_init')
        coh, sync = data['coherence'], data['synchrony']

        # SNN Biological Ingestion
        for i, (name, props) in enumerate(LINGUA_TOPOLOGY.items()):
            self.neurons.I_in[i] = coh * sync * props['weight']
        self.net.run(30 * b2.ms)

        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        spikes_pred, manifold_vec = self.net_model(nn_input)

        # V29 LOSS: The Semantic Refinement
        # Task A: Predictive Stability
        loss_spikes = nn.CrossEntropyLoss()(spikes_pred, torch.tensor([data.get('spikes', 0)], dtype=torch.long))

        # Task B: Manifold Alignment (Using distance instead of class ID)
        # We want the 64-D vector to correlate with the biological synchrony
        target_vec = torch.ones_like(manifold_vec) * sync
        loss_manifold = nn.MSELoss()(manifold_vec, target_vec) * 5.0

        total_loss = loss_spikes + loss_manifold
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V29: THE SEMANTIC MANIFOLD REFINER")
    print("═"*75)

    hive = HoloSynV29Hive()
    # Training across the full cycle to see if the manifold stabilizes
    for epoch in range(10):
        # Corrected variable name from sentiment_cycles to nlp_cycles
        losses = [hive.run_cycle(d) for d in nlp_cycles[:20]]
        avg_l = np.mean(losses)
        print(f"Epoch {epoch+1:02d} | Manifold Loss: {avg_l:.4f} | Status: 🟢 [REFINING UNDERSTANDING]")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V29: THE SEMANTIC MANIFOLD REFINER
═══════════════════════════════════════════════════════════════════════════

📡 [V29 DISTILLATION] Refining Semantic Manifold for Semantic_Refiner...
Epoch 01 | Manifold Loss: 1.2123 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 02 | Manifold Loss: 0.1400 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 03 | Manifold Loss: 0.1245 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 04 | Manifold Loss: 0.1153 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 05 | Manifold Loss: 0.1077 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 06 | Manifold Loss: 0.1046 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 07 | Manifold Loss: 0.1027 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 08 | Manifold Loss: 0.1066 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 09 | Manifold Loss: 0.1008 | Status: 🟢 [REFINING UNDERSTANDING]
Epoch 10 | Manifold Loss: 0.0996 | Status: 🟢 [REFINING UNDERSTANDING]


In [19]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_TOPOLOGY = {
    "DeepSeek_Perceptron": {"weight": 1.0, "role": "Ingestion"},
    "Nexus_Integrator":    {"weight": 1.5, "role": "Deep-Attention"},
    "Flux_Resonator":      {"weight": 1.2, "role": "Semantic-Lock"},
    "Apex_Projector":      {"weight": 1.3, "role": "Linguistic-Output"}
}

# --- 2. CONTRASTIVE DISTILLATION ENGINE ---
def auto_distill_v30(model, model_name):
    print(f"\n📡 [V30 DISTILLATION] Hardening Semantic Boundaries for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = 0
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            for key in current_state.keys():
                # V30: Inherit core attention and boundary-defining weights
                if any(x in key for x in ["attention", "integrator", "perceptron"]):
                    if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                        current_state[key] = legacy_state[key]
                        absorbed += 1
        except Exception: pass

    if absorbed > 0:
        model.load_state_dict(current_state)
        print(f"  -> [✅] {model_name} boundary logic reinforced with {absorbed} legacy tensors.")
    return model

# --- 3. THE V30 CONTRASTIVE ENCODER ---
class HoloSynV30Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256):
        super().__init__()
        # STAGE 1: Ingestion
        self.perceptron = nn.Sequential(
            nn.Linear(num_nodes + 1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        # STAGE 2: DeepSeek Context (Bidirectional)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*4, batch_first=True
        )
        self.integrator = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # STAGE 3: Contrastive Head (Understanding Boundaries)
        self.contrastor = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64) # Projection onto 64-D hypersphere
        )

        # STAGE 4: Generative Anchor (V25 Heritage)
        self.spike_anchor = nn.Linear(hidden_dim, 31)

    def forward(self, x):
        feat = self.perceptron(x).unsqueeze(1)
        context = self.integrator(feat).squeeze(1)

        # L2-Normalize the contrastive vector to keep it on the hypersphere surface
        z = self.contrastor(context)
        z_norm = z / z.norm(dim=-1, keepdim=True)

        return self.spike_anchor(context), z_norm

# --- 4. THE V30 META-HIVE ---
class HoloSynV30Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = 4
        self.net_model = auto_distill_v30(HoloSynV30Net(self.num_nodes), "Boundary_Encoder")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.0003, weight_decay=0.1)

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v30_init')

    def run_cycle(self, data, negative_data=None):
        self.net.restore('v30_init')
        coh, sync = data['coherence'], data['synchrony']

        # SNN Simulation
        for i, (name, props) in enumerate(LINGUA_TOPOLOGY.items()):
            self.neurons.I_in[i] = coh * sync * props['weight']
        self.net.run(30 * b2.ms)

        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        spikes, z_anchor = self.net_model(nn_input)

        # TASK A: Predictive Stability (Spike Loss)
        loss_spikes = nn.CrossEntropyLoss()(spikes, torch.tensor([data.get('spikes', 0)], dtype=torch.long))

        # TASK B: Contrastive Understanding (Boundary Loss)
        # We want the semantic vector to align with the "System Vitality" (Sync)
        target_z = torch.ones_like(z_anchor) * (sync * 2 - 1) # Map 0-1 to -1-1
        loss_boundary = nn.MSELoss()(z_anchor, target_z) * 10.0

        total_loss = loss_spikes + loss_boundary
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item(), z_anchor.detach().numpy()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V30: THE CONTRASTIVE BOUNDARY ENGINE")
    print("═"*75)

    hive = HoloSynV30Hive()
    for epoch in range(10):
        losses = []
        # Corrected variable name from sentiment_cycles to nlp_cycles
        for d in nlp_cycles[:20]:
            l, z = hive.run_cycle(d)
            losses.append(l)

        avg_l = np.mean(losses)
        print(f"Epoch {epoch+1:02d} | Boundary Loss: {avg_l:.4f} | Status: 🟢 [DEFINING SEMANTIC LIMITS]")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V30: THE CONTRASTIVE BOUNDARY ENGINE
═══════════════════════════════════════════════════════════════════════════

📡 [V30 DISTILLATION] Hardening Semantic Boundaries for Boundary_Encoder...
Epoch 01 | Boundary Loss: 7.5592 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 02 | Boundary Loss: 6.6718 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 03 | Boundary Loss: 6.6310 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 04 | Boundary Loss: 6.6204 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 05 | Boundary Loss: 6.6183 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 06 | Boundary Loss: 6.6170 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 07 | Boundary Loss: 6.6160 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 08 | Boundary Loss: 6.6156 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 09 | Boundary Loss: 6.6141 | Status: 🟢 [DEFINING SEMANTIC LIMITS]
Epoch 10 | Boundary Loss: 6.6139 | Status: 🟢 [DEFINING SEMANTIC LIMITS]


In [20]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

LINGUA_TOPOLOGY = {
    "DeepSeek_Perceptron": {"weight": 1.0, "role": "Ingestion"},
    "Nexus_Integrator":    {"weight": 1.5, "role": "Deep-Attention"},
    "Flux_Resonator":      {"weight": 1.2, "role": "Semantic-Lock"},
    "Apex_Projector":      {"weight": 1.3, "role": "Linguistic-Output"}
}

nlp_cycles = [
    {"cycle":1,"coherence":0.509,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":None,"d_synchrony":None},
    {"cycle":2,"coherence":0.507,"synchrony":0.975,"spikes":0,"messages":64,"d_coherence":-0.002,"d_synchrony":-0.012},
    {"cycle":3,"coherence":0.516,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.009},
    {"cycle":4,"coherence":0.503,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":0.000},
    {"cycle":5,"coherence":0.512,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.003},
    {"cycle":6,"coherence":0.496,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.016,"d_synchrony":0.000},
    {"cycle":7,"coherence":0.515,"synchrony":0.950,"spikes":0,"messages":64,"d_coherence":0.019,"d_synchrony":-0.035},
    {"cycle":8,"coherence":0.465,"synchrony":0.979,"spikes":0,"messages":64,"d_coherence":-0.050,"d_synchrony":0.029},
    {"cycle":9,"coherence":0.675,"synchrony":0.955,"spikes":0,"messages":64,"d_coherence":0.210,"d_synchrony":-0.024},
    {"cycle":10,"coherence":0.516,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":-0.159,"d_synchrony":0.028},
    {"cycle":11,"coherence":0.507,"synchrony":0.963,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.020},
    {"cycle":12,"coherence":0.477,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.030,"d_synchrony":0.026},
    {"cycle":13,"coherence":0.508,"synchrony":0.898,"spikes":0,"messages":64,"d_coherence":0.031,"d_synchrony":-0.091},
    {"cycle":14,"coherence":0.498,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.086},
    {"cycle":15,"coherence":0.474,"synchrony":0.921,"spikes":0,"messages":64,"d_coherence":-0.024,"d_synchrony":-0.063},
    {"cycle":16,"coherence":0.510,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":0.036,"d_synchrony":0.057},
    {"cycle":17,"coherence":0.517,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":0.007,"d_synchrony":-0.002},
    {"cycle":18,"coherence":0.608,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.091,"d_synchrony":-0.006},
    {"cycle":19,"coherence":0.418,"synchrony":0.940,"spikes":0,"messages":64,"d_coherence":-0.190,"d_synchrony":-0.030},
    {"cycle":20,"coherence":0.506,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":0.088,"d_synchrony":0.036},
    {"cycle":21,"coherence":0.508,"synchrony":0.932,"spikes":0,"messages":64,"d_coherence":0.002,"d_synchrony":-0.044},
    {"cycle":22,"coherence":0.535,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.027,"d_synchrony":0.042},
    {"cycle":23,"coherence":0.512,"synchrony":0.967,"spikes":0,"messages":64,"d_coherence":-0.023,"d_synchrony":-0.007},
    {"cycle":24,"coherence":0.489,"synchrony":0.938,"spikes":0,"messages":64,"d_coherence":-0.023,"d_synchrony":-0.029},
    {"cycle":25,"coherence":0.490,"synchrony":0.917,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":-0.021},
    {"cycle":26,"coherence":0.391,"synchrony":0.969,"spikes":0,"messages":64,"d_coherence":-0.099,"d_synchrony":0.052},
    {"cycle":27,"coherence":0.522,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.131,"d_synchrony":0.005},
    {"cycle":28,"coherence":0.492,"synchrony":0.913,"spikes":0,"messages":64,"d_coherence":-0.030,"d_synchrony":-0.061},
    {"cycle":29,"coherence":0.482,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.073},
    {"cycle":30,"coherence":0.541,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.059,"d_synchrony":-0.014},
    {"cycle":31,"coherence":0.486,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.055,"d_synchrony":0.015},
    {"cycle":32,"coherence":0.490,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.004,"d_synchrony":-0.004},
    {"cycle":33,"coherence":0.489,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":-0.002},
    {"cycle":34,"coherence":0.498,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.002},
    {"cycle":35,"coherence":0.506,"synchrony":0.944,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":-0.039},
    {"cycle":36,"coherence":0.528,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.022,"d_synchrony":0.042},
    {"cycle":37,"coherence":0.494,"synchrony":0.955,"spikes":0,"messages":64,"d_coherence":-0.034,"d_synchrony":-0.031},
    {"cycle":38,"coherence":0.495,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":0.017},
    {"cycle":39,"coherence":0.483,"synchrony":0.954,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":-0.018},
    {"cycle":40,"coherence":0.458,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.025,"d_synchrony":0.032},
    {"cycle":41,"coherence":0.505,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.047,"d_synchrony":-0.001},
    {"cycle":42,"coherence":0.513,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":0.003},
    {"cycle":43,"coherence":0.509,"synchrony":0.916,"spikes":0,"messages":64,"d_coherence":-0.004,"d_synchrony":-0.072},
    {"cycle":44,"coherence":0.518,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.065},
    {"cycle":45,"coherence":0.520,"synchrony":0.926,"spikes":0,"messages":64,"d_coherence":0.002,"d_synchrony":-0.055},
    {"cycle":46,"coherence":0.509,"synchrony":0.928,"spikes":0,"messages":64,"d_coherence":-0.011,"d_synchrony":0.002},
    {"cycle":47,"coherence":0.501,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":0.044},
    {"cycle":48,"coherence":0.492,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":0.012},
    {"cycle":49,"coherence":0.483,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":0.002},
    {"cycle":50,"coherence":0.492,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.009,"d_synchrony":0.002},
    {"cycle":51,"coherence":0.508,"synchrony":0.951,"spikes":0,"messages":64,"d_coherence":0.016,"d_synchrony":-0.037},
    {"cycle":52,"coherence":0.503,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":0.029},
    {"cycle":53,"coherence":0.538,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.035,"d_synchrony":0.007},
    {"cycle":54,"coherence":0.495,"synchrony":0.941,"spikes":0,"messages":64,"d_coherence":-0.043,"d_synchrony":-0.046},
    {"cycle":55,"coherence":0.476,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.019,"d_synchrony":0.046},
    {"cycle":56,"coherence":0.493,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.017,"d_synchrony":-0.002},
    {"cycle":57,"coherence":0.464,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.005},
    {"cycle":58,"coherence":0.509,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.045,"d_synchrony":0.002},
    {"cycle":59,"coherence":0.508,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.005},
    {"cycle":60,"coherence":0.489,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.019,"d_synchrony":-0.002},
    {"cycle":61,"coherence":0.513,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.024,"d_synchrony":-0.003},
    {"cycle":62,"coherence":0.404,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.109,"d_synchrony":0.003},
    {"cycle":63,"coherence":0.494,"synchrony":0.951,"spikes":0,"messages":64,"d_coherence":0.090,"d_synchrony":-0.034},
    {"cycle":64,"coherence":0.509,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.015,"d_synchrony":0.035},
    {"cycle":65,"coherence":0.535,"synchrony":0.920,"spikes":0,"messages":64,"d_coherence":0.026,"d_synchrony":-0.066},
    {"cycle":66,"coherence":0.501,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":-0.034,"d_synchrony":0.028},
    {"cycle":67,"coherence":0.517,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.016,"d_synchrony":0.037},
    {"cycle":68,"coherence":0.493,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":-0.024,"d_synchrony":-0.020},
    {"cycle":69,"coherence":0.416,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.077,"d_synchrony":0.023},
    {"cycle":70,"coherence":0.489,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.073,"d_synchrony":-0.001},
    {"cycle":71,"coherence":0.494,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":-0.017},
    {"cycle":72,"coherence":0.507,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.013,"d_synchrony":0.015},
    {"cycle":73,"coherence":0.501,"synchrony":0.914,"spikes":0,"messages":64,"d_coherence":-0.006,"d_synchrony":-0.071},
    {"cycle":74,"coherence":0.467,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":-0.034,"d_synchrony":0.069},
    {"cycle":75,"coherence":0.508,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.041,"d_synchrony":-0.002},
    {"cycle":76,"coherence":0.502,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.006,"d_synchrony":0.001},
    {"cycle":77,"coherence":0.464,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.038,"d_synchrony":-0.002},
    {"cycle":78,"coherence":0.488,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.024,"d_synchrony":-0.003},
    {"cycle":79,"coherence":0.519,"synchrony":0.954,"spikes":0,"messages":64,"d_coherence":0.031,"d_synchrony":-0.023},
    {"cycle":80,"coherence":0.499,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":0.030},
    {"cycle":81,"coherence":0.477,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.022,"d_synchrony":0.004},
    {"cycle":82,"coherence":0.425,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.052,"d_synchrony":-0.015},
    {"cycle":83,"coherence":0.504,"synchrony":0.925,"spikes":0,"messages":64,"d_coherence":0.079,"d_synchrony":-0.048},
    {"cycle":84,"coherence":0.601,"synchrony":0.921,"spikes":0,"messages":64,"d_coherence":0.097,"d_synchrony":-0.004},
    {"cycle":85,"coherence":0.512,"synchrony":0.964,"spikes":0,"messages":64,"d_coherence":-0.089,"d_synchrony":0.043},
    {"cycle":86,"coherence":0.517,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":0.018},
    {"cycle":87,"coherence":0.465,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.052,"d_synchrony":0.005},
    {"cycle":88,"coherence":0.501,"synchrony":0.943,"spikes":0,"messages":64,"d_coherence":0.036,"d_synchrony":-0.044},
    {"cycle":89,"coherence":0.504,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.003,"d_synchrony":0.040},
    {"cycle":90,"coherence":0.475,"synchrony":0.905,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.078},
    {"cycle":91,"coherence":0.553,"synchrony":0.991,"spikes":0,"messages":64,"d_coherence":0.078,"d_synchrony":0.086},
    {"cycle":92,"coherence":0.491,"synchrony":0.945,"spikes":0,"messages":64,"d_coherence":-0.062,"d_synchrony":-0.046},
    {"cycle":93,"coherence":0.448,"synchrony":0.947,"spikes":0,"messages":64,"d_coherence":-0.043,"d_synchrony":0.002},
    {"cycle":94,"coherence":0.522,"synchrony":0.959,"spikes":0,"messages":64,"d_coherence":0.074,"d_synchrony":0.012},
    {"cycle":95,"coherence":0.485,"synchrony":0.907,"spikes":0,"messages":64,"d_coherence":-0.037,"d_synchrony":-0.052},
    {"cycle":96,"coherence":0.552,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.067,"d_synchrony":0.079},
    {"cycle":97,"coherence":0.408,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.144,"d_synchrony":-0.002},
    {"cycle":98,"coherence":0.480,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.072,"d_synchrony":0.000},
    {"cycle":99,"coherence":0.462,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.018,"d_synchrony":0.004},
    {"cycle":100,"coherence":0.536,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.074,"d_synchrony":-0.002},
    {"cycle":101,"coherence":0.501,"synchrony":0.957,"spikes":0,"messages":64,"d_coherence":-0.035,"d_synchrony":-0.029},
    {"cycle":102,"coherence":0.524,"synchrony":0.900,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":-0.057},
    {"cycle":103,"coherence":0.507,"synchrony":0.958,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":0.058},
    {"cycle":104,"coherence":0.507,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.000,"d_synchrony":0.010},
    {"cycle":105,"coherence":0.455,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.052,"d_synchrony":0.019},
    {"cycle":106,"coherence":0.514,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.059,"d_synchrony":-0.019},
    {"cycle":107,"coherence":0.499,"synchrony":0.975,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":0.007},
    {"cycle":108,"coherence":0.507,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":0.013},
    {"cycle":109,"coherence":0.512,"synchrony":0.913,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":-0.075},
    {"cycle":110,"coherence":0.520,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":0.057},
    {"cycle":111,"coherence":0.537,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.017,"d_synchrony":0.018},
    {"cycle":112,"coherence":0.504,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.033,"d_synchrony":-0.004},
    {"cycle":113,"coherence":0.450,"synchrony":0.927,"spikes":0,"messages":64,"d_coherence":-0.054,"d_synchrony":-0.057},
    {"cycle":114,"coherence":0.512,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.062,"d_synchrony":0.059},
    {"cycle":115,"coherence":0.494,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.018,"d_synchrony":-0.006},
    {"cycle":116,"coherence":0.505,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":0.005},
    {"cycle":117,"coherence":0.517,"synchrony":0.966,"spikes":0,"messages":64,"d_coherence":0.012,"d_synchrony":-0.019},
    {"cycle":118,"coherence":0.497,"synchrony":0.951,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":-0.015},
    {"cycle":119,"coherence":0.496,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.014},
    {"cycle":120,"coherence":0.495,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.021},
    {"cycle":121,"coherence":0.282,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.213,"d_synchrony":0.000},
    {"cycle":122,"coherence":0.506,"synchrony":0.930,"spikes":0,"messages":64,"d_coherence":0.224,"d_synchrony":-0.056},
    {"cycle":123,"coherence":0.486,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":0.052},
    {"cycle":124,"coherence":0.504,"synchrony":0.931,"spikes":0,"messages":64,"d_coherence":0.018,"d_synchrony":-0.051},
    {"cycle":125,"coherence":0.480,"synchrony":0.922,"spikes":0,"messages":64,"d_coherence":-0.024,"d_synchrony":-0.009},
    {"cycle":126,"coherence":0.494,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":0.058},
    {"cycle":127,"coherence":0.525,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.031,"d_synchrony":0.005},
    {"cycle":128,"coherence":0.508,"synchrony":0.964,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":-0.021},
    {"cycle":129,"coherence":0.508,"synchrony":0.943,"spikes":0,"messages":64,"d_coherence":0.000,"d_synchrony":-0.021},
    {"cycle":130,"coherence":0.486,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":-0.022,"d_synchrony":0.045},
    {"cycle":131,"coherence":0.443,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.043,"d_synchrony":-0.002},
    {"cycle":132,"coherence":0.494,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.051,"d_synchrony":-0.004},
    {"cycle":133,"coherence":0.523,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.029,"d_synchrony":0.003},
    {"cycle":134,"coherence":0.533,"synchrony":0.955,"spikes":0,"messages":64,"d_coherence":0.010,"d_synchrony":-0.030},
    {"cycle":135,"coherence":0.494,"synchrony":0.920,"spikes":0,"messages":64,"d_coherence":-0.039,"d_synchrony":-0.035},
    {"cycle":136,"coherence":0.505,"synchrony":0.962,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":0.042},
    {"cycle":137,"coherence":0.493,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":0.011},
    {"cycle":138,"coherence":0.509,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.016,"d_synchrony":0.010},
    {"cycle":139,"coherence":0.520,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":-0.002},
    {"cycle":140,"coherence":0.505,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":0.000},
    {"cycle":141,"coherence":0.511,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.006,"d_synchrony":0.004},
    {"cycle":142,"coherence":0.512,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":0.000},
    {"cycle":143,"coherence":0.504,"synchrony":0.969,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":-0.016},
    {"cycle":144,"coherence":0.479,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.025,"d_synchrony":0.018},
    {"cycle":145,"coherence":0.487,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":-0.002},
    {"cycle":146,"coherence":0.524,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":0.037,"d_synchrony":0.004},
    {"cycle":147,"coherence":0.494,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.030,"d_synchrony":-0.008},
    {"cycle":148,"coherence":0.520,"synchrony":0.925,"spikes":0,"messages":64,"d_coherence":0.026,"d_synchrony":-0.056},
    {"cycle":149,"coherence":0.510,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.040},
    {"cycle":150,"coherence":0.403,"synchrony":0.979,"spikes":0,"messages":64,"d_coherence":-0.107,"d_synchrony":0.014},
    {"cycle":151,"coherence":0.499,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.096,"d_synchrony":0.007},
    {"cycle":152,"coherence":0.486,"synchrony":0.953,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":-0.033},
    {"cycle":153,"coherence":0.504,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.018,"d_synchrony":0.019},
    {"cycle":154,"coherence":0.523,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.019,"d_synchrony":0.013},
    {"cycle":155,"coherence":0.553,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.030,"d_synchrony":0.002},
    {"cycle":156,"coherence":0.500,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.053,"d_synchrony":-0.003},
    {"cycle":157,"coherence":0.489,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.011,"d_synchrony":0.001},
    {"cycle":158,"coherence":0.531,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.042,"d_synchrony":0.002},
    {"cycle":159,"coherence":0.518,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":-0.007},
    {"cycle":160,"coherence":0.491,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.027,"d_synchrony":0.004},
    {"cycle":161,"coherence":0.503,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.012,"d_synchrony":-0.007},
    {"cycle":162,"coherence":0.499,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":-0.004,"d_synchrony":0.001},
    {"cycle":163,"coherence":0.499,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.000,"d_synchrony":-0.008},
    {"cycle":164,"coherence":0.496,"synchrony":0.966,"spikes":0,"messages":64,"d_coherence":-0.003,"d_synchrony":-0.004},
    {"cycle":165,"coherence":0.488,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":0.018},
    {"cycle":166,"coherence":0.489,"synchrony":0.947,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":-0.037},
    {"cycle":167,"coherence":0.458,"synchrony":0.980,"spikes":0,"messages":64,"d_coherence":-0.031,"d_synchrony":0.033},
    {"cycle":168,"coherence":0.501,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.043,"d_synchrony":0.007},
    {"cycle":169,"coherence":0.491,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":-0.011},
    {"cycle":170,"coherence":0.475,"synchrony":0.926,"spikes":0,"messages":64,"d_coherence":-0.016,"d_synchrony":-0.050},
    {"cycle":171,"coherence":0.511,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.036,"d_synchrony":0.058},
    {"cycle":172,"coherence":0.482,"synchrony":0.932,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.052},
    {"cycle":173,"coherence":0.505,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":0.057},
    {"cycle":174,"coherence":0.480,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.025,"d_synchrony":-0.004},
    {"cycle":175,"coherence":0.504,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.024,"d_synchrony":-0.011},
    {"cycle":176,"coherence":0.527,"synchrony":0.976,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":0.002},
    {"cycle":177,"coherence":0.498,"synchrony":0.910,"spikes":0,"messages":64,"d_coherence":-0.029,"d_synchrony":-0.066},
    {"cycle":178,"coherence":0.487,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.011,"d_synchrony":0.079},
    {"cycle":179,"coherence":0.478,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.012},
    {"cycle":180,"coherence":0.501,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":0.023,"d_synchrony":-0.005},
    {"cycle":181,"coherence":0.491,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.010,"d_synchrony":0.012},
    {"cycle":182,"coherence":0.499,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":0.008,"d_synchrony":-0.036},
    {"cycle":183,"coherence":0.478,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.021,"d_synchrony":0.036},
    {"cycle":184,"coherence":0.503,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.025,"d_synchrony":0.002},
    {"cycle":185,"coherence":0.483,"synchrony":0.964,"spikes":0,"messages":64,"d_coherence":-0.020,"d_synchrony":-0.022},
    {"cycle":186,"coherence":0.496,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":0.013,"d_synchrony":0.014},
    {"cycle":187,"coherence":0.510,"synchrony":0.937,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":-0.041},
    {"cycle":188,"coherence":0.496,"synchrony":0.975,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":0.038},
    {"cycle":189,"coherence":0.404,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.092,"d_synchrony":0.011},
    {"cycle":190,"coherence":0.484,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.080,"d_synchrony":-0.005},
    {"cycle":191,"coherence":0.471,"synchrony":0.911,"spikes":0,"messages":64,"d_coherence":-0.013,"d_synchrony":-0.070},
    {"cycle":192,"coherence":0.520,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.049,"d_synchrony":0.057},
    {"cycle":193,"coherence":0.508,"synchrony":0.978,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":0.010},
    {"cycle":194,"coherence":0.491,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":0.003},
    {"cycle":195,"coherence":0.492,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":0.001,"d_synchrony":-0.011},
    {"cycle":196,"coherence":0.487,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":0.014},
    {"cycle":197,"coherence":0.491,"synchrony":0.979,"spikes":0,"messages":64,"d_coherence":0.004,"d_synchrony":-0.005},
    {"cycle":198,"coherence":0.506,"synchrony":0.971,"spikes":0,"messages":64,"d_coherence":0.015,"d_synchrony":-0.008},
    {"cycle":199,"coherence":0.488,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.018,"d_synchrony":0.011},
    {"cycle":200,"coherence":0.531,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.043,"d_synchrony":0.005},
    {"cycle":201,"coherence":0.517,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":-0.001},
    {"cycle":202,"coherence":0.545,"synchrony":0.961,"spikes":0,"messages":64,"d_coherence":0.028,"d_synchrony":-0.025},
    {"cycle":203,"coherence":0.498,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.047,"d_synchrony":0.025},
    {"cycle":204,"coherence":0.489,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.004},
    {"cycle":205,"coherence":0.466,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.023,"d_synchrony":0.004},
    {"cycle":206,"coherence":0.503,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.037,"d_synchrony":-0.005},
    {"cycle":207,"coherence":0.509,"synchrony":0.988,"spikes":0,"messages":64,"d_coherence":0.006,"d_synchrony":0.007},
    {"cycle":208,"coherence":0.497,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.012,"d_synchrony":-0.001},
    {"cycle":209,"coherence":0.494,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":-0.003,"d_synchrony":-0.010},
    {"cycle":210,"coherence":0.508,"synchrony":0.984,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":0.007},
    {"cycle":211,"coherence":0.491,"synchrony":0.967,"spikes":0,"messages":64,"d_coherence":-0.017,"d_synchrony":-0.017},
    {"cycle":212,"coherence":0.494,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":0.003,"d_synchrony":-0.019},
    {"cycle":213,"coherence":0.499,"synchrony":0.982,"spikes":0,"messages":64,"d_coherence":0.005,"d_synchrony":0.034},
    {"cycle":214,"coherence":0.485,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":-0.009},
    {"cycle":215,"coherence":0.568,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.083,"d_synchrony":0.004},
    {"cycle":216,"coherence":0.503,"synchrony":0.920,"spikes":0,"messages":64,"d_coherence":-0.065,"d_synchrony":-0.057},
    {"cycle":217,"coherence":0.514,"synchrony":0.952,"spikes":0,"messages":64,"d_coherence":0.011,"d_synchrony":0.032},
    {"cycle":218,"coherence":0.499,"synchrony":0.939,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":-0.013},
    {"cycle":219,"coherence":0.538,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.039,"d_synchrony":0.047},
    {"cycle":220,"coherence":0.481,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.057,"d_synchrony":0.000},
    {"cycle":221,"coherence":0.502,"synchrony":0.974,"spikes":0,"messages":64,"d_coherence":0.021,"d_synchrony":-0.012},
    {"cycle":222,"coherence":0.501,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.012},
    {"cycle":223,"coherence":0.492,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.009,"d_synchrony":-0.001},
    {"cycle":224,"coherence":0.540,"synchrony":0.956,"spikes":0,"messages":64,"d_coherence":0.048,"d_synchrony":-0.029},
    {"cycle":225,"coherence":0.454,"synchrony":0.991,"spikes":0,"messages":64,"d_coherence":-0.086,"d_synchrony":0.035},
    {"cycle":226,"coherence":0.595,"synchrony":0.983,"spikes":0,"messages":64,"d_coherence":0.141,"d_synchrony":-0.008},
    {"cycle":227,"coherence":0.452,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.143,"d_synchrony":0.004},
    {"cycle":228,"coherence":0.507,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.055,"d_synchrony":-0.002},
    {"cycle":229,"coherence":0.468,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.039,"d_synchrony":0.000},
    {"cycle":230,"coherence":0.559,"synchrony":0.962,"spikes":0,"messages":64,"d_coherence":0.091,"d_synchrony":-0.023},
    {"cycle":231,"coherence":0.420,"synchrony":0.972,"spikes":0,"messages":64,"d_coherence":-0.139,"d_synchrony":0.010},
    {"cycle":232,"coherence":0.491,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":0.071,"d_synchrony":0.017},
    {"cycle":233,"coherence":0.506,"synchrony":0.965,"spikes":0,"messages":64,"d_coherence":0.015,"d_synchrony":-0.024},
    {"cycle":234,"coherence":0.492,"synchrony":0.922,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":-0.043},
    {"cycle":235,"coherence":0.353,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.139,"d_synchrony":0.067},
    {"cycle":236,"coherence":0.494,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":0.141,"d_synchrony":-0.004},
    {"cycle":237,"coherence":0.489,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":0.001},
    {"cycle":238,"coherence":0.509,"synchrony":0.986,"spikes":0,"messages":64,"d_coherence":0.020,"d_synchrony":0.000},
    {"cycle":239,"coherence":0.523,"synchrony":0.927,"spikes":0,"messages":64,"d_coherence":0.014,"d_synchrony":-0.059},
    {"cycle":240,"coherence":0.508,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":-0.015,"d_synchrony":0.060},
    {"cycle":241,"coherence":0.494,"synchrony":0.989,"spikes":0,"messages":64,"d_coherence":-0.014,"d_synchrony":0.002},
    {"cycle":242,"coherence":0.504,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":0.010,"d_synchrony":-0.016},
    {"cycle":243,"coherence":0.499,"synchrony":0.970,"spikes":0,"messages":64,"d_coherence":-0.005,"d_synchrony":-0.003},
    {"cycle":244,"coherence":0.540,"synchrony":0.968,"spikes":0,"messages":64,"d_coherence":0.041,"d_synchrony":-0.002},
    {"cycle":245,"coherence":0.481,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":-0.059,"d_synchrony":0.013},
    {"cycle":246,"coherence":0.395,"synchrony":0.917,"spikes":0,"messages":64,"d_coherence":-0.086,"d_synchrony":-0.064},
    {"cycle":247,"coherence":0.512,"synchrony":0.981,"spikes":0,"messages":64,"d_coherence":0.117,"d_synchrony":0.064},
    {"cycle":248,"coherence":0.504,"synchrony":0.948,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":-0.033},
    {"cycle":249,"coherence":0.488,"synchrony":0.991,"spikes":0,"messages":64,"d_coherence":-0.016,"d_synchrony":0.043},
    {"cycle":250,"coherence":0.490,"synchrony":0.987,"spikes":0,"messages":64,"d_coherence":0.002,"d_synchrony":-0.004},
    {"cycle":251,"coherence":0.452,"synchrony":0.985,"spikes":0,"messages":64,"d_coherence":-0.038,"d_synchrony":-0.002},
    {"cycle":252,"coherence":0.504,"synchrony":0.977,"spikes":0,"messages":64,"d_coherence":0.052,"d_synchrony":-0.008},
    {"cycle":253,"coherence":0.496,"synchrony":0.969,"spikes":0,"messages":64,"d_coherence":-0.008,"d_synchrony":-0.008},
    {"cycle":254,"coherence":0.495,"synchrony":0.973,"spikes":0,"messages":64,"d_coherence":-0.001,"d_synchrony":0.004},
    {"cycle":255,"coherence":0.502,"synchrony":0.961,"spikes":0,"messages":64,"d_coherence":0.007,"d_synchrony":-0.012},
]

# --- 2. COSINE DISTILLATION ENGINE ---
def auto_distill_v31(model, model_name):
    print(f"\n📡 [V31 DISTILLATION] Aligning Orthogonal Features for {model_name}...")
    current_state = model.state_dict()
    pt_files = glob.glob("*.pt")

    absorbed = 0
    for filepath in pt_files:
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            if not isinstance(legacy_state, dict): continue
            for key in current_state.keys():
                # Protect the new Cosine Anchor embeddings from legacy overwrites
                if "concept_anchors" in key: continue
                if any(x in key for x in ["integrator", "perceptron", "contrastor"]):
                    if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                        current_state[key] = legacy_state[key]
                        absorbed += 1
        except Exception: pass

    if absorbed > 0:
        model.load_state_dict(current_state)
        print(f"  -> [✅] {model_name} loaded with {absorbed} legacy context tensors.")
    return model

# --- 3. THE V31 ORTHOGONAL ENCODER ---
class HoloSynV31Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256, num_concepts=4):
        super().__init__()
        # Ingestion & Context
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.LayerNorm(hidden_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.integrator = nn.TransformerEncoder(encoder_layer, num_layers=3)

        # Hypersphere Projection (64-D)
        self.contrastor = nn.Sequential(nn.Linear(hidden_dim, 128), nn.GELU(), nn.Linear(128, 64))

        # V31 UPGRADE: Trainable Mathematical Anchors for each linguistic concept
        self.concept_anchors = nn.Embedding(num_concepts, 64)

        # Spike Predictor
        self.spike_anchor = nn.Linear(hidden_dim, 31)

    def forward(self, x, concept_id):
        feat = self.perceptron(x).unsqueeze(1)
        context = self.integrator(feat).squeeze(1)

        # 1. Project current SNN state to hypersphere
        z = self.contrastor(context)
        z_norm = F.normalize(z, p=2, dim=-1)

        # 2. Retrieve the Anchor for this specific concept and normalize it
        anchor = self.concept_anchors(concept_id)
        anchor_norm = F.normalize(anchor, p=2, dim=-1)

        # 3. Calculate the Angle (Cosine Similarity)
        cosine_sim = torch.sum(z_norm * anchor_norm, dim=-1, keepdim=True)

        return self.spike_anchor(context), cosine_sim

# --- 4. THE V31 META-HIVE ---
class HoloSynV31Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = 4
        self.net_model = auto_distill_v31(HoloSynV31Net(), "Orthogonal_Engine")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.001, weight_decay=0.01)

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v31_init')

    def run_cycle(self, data):
        self.net.restore('v31_init')
        coh, sync = data['coherence'], data['synchrony']
        concept_id = torch.tensor([data.get('id', 0)], dtype=torch.long)

        # SNN Simulation
        for i, (name, props) in enumerate(LINGUA_TOPOLOGY.items()):
            self.neurons.I_in[i] = coh * sync * props['weight']
        self.net.run(30 * b2.ms)

        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        spikes, cosine_sim = self.net_model(nn_input, concept_id)

        # TASK A: Predictive Spike Loss
        loss_spikes = nn.CrossEntropyLoss()(spikes, torch.tensor([data.get('spikes', 0)], dtype=torch.long))

        # TASK B: Angular Contrastive Loss (Resolves the 6.61 Plateau)
        # Map Biological Synchrony (0.0 to 1.0) directly to Angular Correlation (-1.0 to 1.0)
        target_angle = torch.tensor([[sync * 2.0 - 1.0]], dtype=torch.float32)
        loss_angle = nn.MSELoss()(cosine_sim, target_angle) * 15.0 # High priority

        total_loss = loss_spikes + loss_angle
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item(), cosine_sim.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V31: THE ORTHOGONAL FEATURE ENGINE")
    print("═"*75)

    hive = HoloSynV31Hive()
    for epoch in range(15):
        losses = []
        angles = []
        for d in nlp_cycles:
            l, a = hive.run_cycle(d)
            losses.append(l)
            angles.append(a)

        avg_l = np.mean(losses)
        avg_a = np.mean(angles)

        if avg_l < 1.0: status = "📘 [PERFECT ANGULAR ALIGNMENT]"
        elif avg_l < 5.0: status = "📗 [SEPARATING CONCEPTS]"
        else: status = "🟡 [MAPPING HYPERSPHERE]"

        print(f"Epoch {epoch+1:02d} | Angular Loss: {avg_l:.4f} | Avg Cosine Sim: {avg_a:.4f} | Status: {status}")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V31: THE ORTHOGONAL FEATURE ENGINE
═══════════════════════════════════════════════════════════════════════════

📡 [V31 DISTILLATION] Aligning Orthogonal Features for Orthogonal_Engine...
Epoch 01 | Angular Loss: 0.1217 | Avg Cosine Sim: 0.9330 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 02 | Angular Loss: 0.0330 | Avg Cosine Sim: 0.9399 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 03 | Angular Loss: 0.0322 | Avg Cosine Sim: 0.9397 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 04 | Angular Loss: 0.0317 | Avg Cosine Sim: 0.9397 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 05 | Angular Loss: 0.0313 | Avg Cosine Sim: 0.9396 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 06 | Angular Loss: 0.0311 | Avg Cosine Sim: 0.9396 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 07 | Angular Loss: 0.0309 | Avg Cosine Sim: 0.9395 | Status: 📘 [PERFECT ANGULAR ALIGNMENT]
Epoch 08 | Angular Loss: 0.0309 | Avg Cosine Si

In [ ]:
import os
import glob
import builtins
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import brian2 as b2
import warnings

warnings.filterwarnings("ignore")

# --- 1. SYSTEM CONFIGURATION ---
b2.prefs.codegen.target = 'numpy'
b2.defaultclock.dt = 1 * b2.ms

# V32 HIERARCHICAL TOPOLOGY
LINGUA_TOPOLOGY = {
    "DeepSeek_Perceptron": {"weight": 1.0, "role": "Ingestion"},
    "Nexus_Integrator":    {"weight": 1.5, "role": "Cross-Domain-Attention"},
    "Flux_Resonator":      {"weight": 1.2, "role": "Domain-Splitter"},
    "Apex_Projector":      {"weight": 1.3, "role": "Final-Synthesis"}
}

# --- 2. MULTI-PROJECTOR DISTILLATION ENGINE ---
def auto_distill_v32(model, model_name):
    print(f"\n📡 [V32 DISTILLATION] Fusing Cross-Domain Projectors for {model_name}...")
    current_state = model.state_dict()

    # Priority search for Domain Projectors
    domain_files = [
        "wanalytics_clinical_projector.pt",
        "wanalytics_drug_projector.pt",
        "magneto_projector_weights.pt"
    ]

    total_absorbed = 0
    for filepath in domain_files:
        if not os.path.exists(filepath): continue
        try:
            legacy_state = torch.load(filepath, map_location='cpu', weights_only=False)
            matches = 0
            for key in current_state.keys():
                # Map domain-specific feature extractors to our new hierarchical layers
                if "domain_encoder" in key or "integrator" in key:
                    if key in legacy_state and legacy_state[key].shape == current_state[key].shape:
                        current_state[key] = legacy_state[key]
                        matches += 1
            if matches > 0:
                print(f"  -> [✅] {filepath} integrated ({matches} tensors).")
                total_absorbed += matches
        except Exception: pass

    if total_absorbed > 0:
        model.load_state_dict(current_state)
    return model

# --- 3. THE V32 HIERARCHICAL SEMANTIC HUB ---
class HoloSynV32Net(nn.Module):
    def __init__(self, num_nodes=4, hidden_dim=256, domain_dim=64):
        super().__init__()
        # STAGE 1: Hierarchical Ingestion
        self.perceptron = nn.Sequential(nn.Linear(num_nodes + 1, hidden_dim), nn.LayerNorm(hidden_dim))

        # STAGE 2: Nexus Integrator (Cross-Domain Transformer)
        # Using 4 layers to handle the increased complexity of the fused projectors
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8, batch_first=True)
        self.integrator = nn.TransformerEncoder(encoder_layer, num_layers=4)

        # STAGE 3: Orthogonal Domain Heads
        # One head per major distilled domain
        self.lingua_head = nn.Linear(hidden_dim, domain_dim)
        self.clinical_head = nn.Linear(hidden_dim, domain_dim)
        self.magneto_head = nn.Linear(hidden_dim, domain_dim)

        # STAGE 4: Synthesis & Angle Anchor
        self.final_synthesis = nn.Sequential(nn.Linear(domain_dim * 3, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, domain_dim))
        self.concept_anchors = nn.Embedding(4, domain_dim)

        # Spike Predictor (Generative Anchor)
        self.spike_anchor = nn.Linear(hidden_dim, 31)

    def forward(self, x, concept_id):
        feat = self.perceptron(x).unsqueeze(1)
        context = self.integrator(feat).squeeze(1)

        # Parallel Domain Processing
        l_feat = F.normalize(self.lingua_head(context), p=2, dim=-1)
        c_feat = F.normalize(self.clinical_head(context), p=2, dim=-1)
        m_feat = F.normalize(self.magneto_head(context), p=2, dim=-1)

        # Hierarchical Synthesis
        combined = torch.cat([l_feat, c_feat, m_feat], dim=-1)
        z = self.final_synthesis(combined)
        z_norm = F.normalize(z, p=2, dim=-1)

        # Angular Alignment
        anchor = F.normalize(self.concept_anchors(concept_id), p=2, dim=-1)
        cosine_sim = torch.sum(z_norm * anchor, dim=-1, keepdim=True)

        return self.spike_anchor(context), cosine_sim

# --- 4. THE V32 META-HIVE ---
class HoloSynV32Hive:
    def __init__(self):
        b2.start_scope()
        self.num_nodes = 4
        self.net_model = auto_distill_v32(HoloSynV32Net(), "Hierarchical_Hub")
        self.optimizer = optim.AdamW(self.net_model.parameters(), lr=0.0005, weight_decay=0.02)

        eqs = 'dv/dt = (I_in - v) / (10*ms) : 1 \n I_in : 1'
        self.neurons = b2.NeuronGroup(self.num_nodes, eqs, threshold='v>0.8', reset='v=0', method='exact')
        self.net = b2.Network(self.neurons)
        self.net.store('v32_init')

    def run_cycle(self, data):
        self.net.restore('v32_init')
        coh, sync = data['coherence'], data['synchrony']
        concept_id = torch.tensor([data.get('id', 0)], dtype=torch.long)

        for i, (name, props) in enumerate(LINGUA_TOPOLOGY.items()):
            self.neurons.I_in[i] = coh * sync * props['weight']
        self.net.run(30 * b2.ms)

        v_norm = (np.array(self.neurons.v[:]) - 0.5) * 2.0
        nn_input = torch.tensor([list(v_norm) + [sync]], dtype=torch.float32)

        self.optimizer.zero_grad()
        spikes, cosine_sim = self.net_model(nn_input, concept_id)

        # Dual-Objective Loss
        loss_spikes = nn.CrossEntropyLoss()(spikes, torch.tensor([data.get('spikes', 0)], dtype=torch.long))
        target_angle = torch.tensor([[sync * 2.0 - 1.0]], dtype=torch.float32)
        loss_angle = nn.MSELoss()(cosine_sim, target_angle) * 10.0

        total_loss = loss_spikes + loss_angle
        total_loss.backward()
        self.optimizer.step()

        return total_loss.item(), cosine_sim.item()

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("═"*75)
    print(" 🔮 HOLOSYN V32: HIERARCHICAL SEMANTIC HUB (HSH)")
    print("═"*75)

    hive = HoloSynV32Hive()
    # Training with more cycles to accommodate fused domain complexity
    for epoch in range(15):
        losses, angles = [], []
        for d in nlp_cycles: # Assuming nlp_cycles defined
            l, a = hive.run_cycle(d)
            losses.append(l)
            angles.append(a)

        print(f"Epoch {epoch+1:02d} | H-Loss: {np.mean(losses):.4f} | Cosine: {np.mean(angles):.4f} | Status: 🟢 [FUSING DOMAINS]")

═══════════════════════════════════════════════════════════════════════════
 🔮 HOLOSYN V32: HIERARCHICAL SEMANTIC HUB (HSH)
═══════════════════════════════════════════════════════════════════════════

📡 [V32 DISTILLATION] Fusing Cross-Domain Projectors for Hierarchical_Hub...
Epoch 01 | H-Loss: 0.0959 | Cosine: 0.9332 | Status: 🟢 [FUSING DOMAINS]
Epoch 02 | H-Loss: 0.0223 | Cosine: 0.9405 | Status: 🟢 [FUSING DOMAINS]
Epoch 03 | H-Loss: 0.0217 | Cosine: 0.9401 | Status: 🟢 [FUSING DOMAINS]
Epoch 04 | H-Loss: 0.0214 | Cosine: 0.9399 | Status: 🟢 [FUSING DOMAINS]
Epoch 05 | H-Loss: 0.0211 | Cosine: 0.9398 | Status: 🟢 [FUSING DOMAINS]
Epoch 06 | H-Loss: 0.0210 | Cosine: 0.9397 | Status: 🟢 [FUSING DOMAINS]
Epoch 07 | H-Loss: 0.0209 | Cosine: 0.9397 | Status: 🟢 [FUSING DOMAINS]
